In [0]:
# ============================================================================
# PRICING FEATURES - ADOPTION & ENGAGEMENT DEPTH ANALYSIS
# ============================================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pyspark.sql import functions as F
from pyspark.sql.window import Window
import warnings
warnings.filterwarnings('ignore')

# Clean color palette: Orange gradient
COLORS = {
    'light': '#FFE0B2',
    'medium_light': '#FFB74D', 
    'medium': '#FF9800',
    'medium_dark': '#F57C00',
    'dark': '#E65100',
    'grey_light': '#EEEEEE',
    'grey': '#BDBDBD',
    'grey_dark': '#757575',
    'border': '#424242'
}

print("="*100)
print("PRICING FEATURES - ADOPTION & ENGAGEMENT DEPTH ANALYSIS")
print("="*100)

# ============================================================================
# STEP 1: LOAD DATA & CREATE GMV TIERS
# ============================================================================

print("\n[1/5] Loading data...")

df_spark = spark.table("production.supply_analytics.pricing_feature_combined")

# Create supplier GMV tiers
supplier_gmv = df_spark.filter(F.col("gmv_l12mo") > 0) \
    .groupBy("supplier_id") \
    .agg(F.sum("gmv_l12mo").alias("supplier_gmv_l12mo"))

window_spec = Window.orderBy("supplier_gmv_l12mo")
supplier_gmv = supplier_gmv.withColumn("gmv_quartile", F.ntile(4).over(window_spec)) \
    .withColumn("gmv_tier",
        F.when(F.col("gmv_quartile") == 1, F.lit("Bottom 25%"))
         .when(F.col("gmv_quartile") == 2, F.lit("25-50%"))
         .when(F.col("gmv_quartile") == 3, F.lit("50-75%"))
         .when(F.col("gmv_quartile") == 4, F.lit("Top 25%"))
    )

df_spark = df_spark.join(
    supplier_gmv.select("supplier_id", F.col("gmv_tier")),
    on="supplier_id", how="left"
)

df = df_spark.toPandas()

# Convert decimals
decimal_cols = ['gmv_l12mo', 'nr_l12mo', 'bookings_l12mo', 'avg_booking_value_l12mo']
for col in decimal_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

# Convert numeric columns
numeric_cols = ['num_individual_categories', 'num_individual_tiers', 'num_group_configs', 
                'num_group_tiers', 'max_group_tiers', 'num_addons', 'num_addon_tiers']
for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)

# Filter active tours
df = df[df['gmv_l12mo'] > 0].copy()
df['supplier_segment'] = df['supplier_segment'].fillna('Unknown')
df['activity_type'] = df['activity_type'].fillna('Unknown')

# Fix is_managed
if 'is_managed' in df.columns:
    df['is_managed'] = df['is_managed'].apply(lambda x: 1 if x in [1, 'Yes', True, '1'] else 0)
else:
    df['is_managed'] = 0

# CORRECT FEATURE DEFINITIONS (using the properly separated fields)
PRICING_FEATURES = {
    'has_group_pricing': 'Group Pricing',
    'has_addons': 'Add-ons',
    'has_native_scales': 'Native Pricing Scales',
    'has_api_scales': 'API Pricing Scales',
    'has_static_api_pricing': 'Static API Pricing',
    'has_live_dynamic_pricing': 'Live Dynamic Pricing'
}

# ENGAGEMENT DEPTH METRICS (counts, not binary)
DEPTH_METRICS = {
    'num_individual_categories': 'Individual Categories',
    'num_individual_tiers': 'Individual Price Tiers',
    'num_group_configs': 'Group Configurations',
    'num_group_tiers': 'Group Price Tiers',
    'num_addons': 'Add-ons Count',
    'num_addon_tiers': 'Add-on Price Tiers'
}

GMV_TIERS = ['Bottom 25%', '25-50%', '50-75%', 'Top 25%']

print(f"✓ Loaded: {len(df):,} tours, {df['supplier_id'].nunique():,} suppliers")
print(f"✓ GMV: €{df['gmv_l12mo'].sum()/1e6:.1f}M")

# ============================================================================
# SEGMENTATION EXPLANATIONS
# ============================================================================

print("\n" + "="*100)
print("SEGMENTATION & METRICS GUIDE")
print("="*100)

print("\n[ACTIVATION METRICS - Binary Flags]")
print("  - Group Pricing: Tour has group pricing configured")
print("  - Add-ons: Tour has add-ons configured")
print("  - Native Pricing Scales: Tour uses pricing scales (NOT via API)")
print("  - API Pricing Scales: Tour uses pricing scales via API")
print("  - Static API Pricing: Tour gets prices from API but NOT real-time")
print("  - Live Dynamic Pricing: Tour uses real-time pricing and availability")

print("\n[ENGAGEMENT DEPTH METRICS - Counts]")
print("  - Individual Categories: Count of ticket types (Adult, Child, Infant, etc.)")
print("  - Individual Price Tiers: Total pricing tiers across all individual categories")
print("  - Group Configurations: Count of group pricing setups")
print("  - Group Price Tiers: Total pricing tiers across all group configs")
print("  - Add-ons Count: Number of add-ons configured")
print("  - Add-on Price Tiers: Total pricing tiers across all add-ons")

print("\n[SEGMENTATION DIMENSIONS]")
print("  - GMV Tier: Supplier size by revenue quartile")
print("  - Supplier Segment: Business classification (Scale Seeker, Independent Creator, etc.)")
print("  - Activity Type: Experience category (privateTour, transfer, dayTrip, etc.)")
print("  - Managed Status: Has dedicated account manager or not")

print("="*100 + "\n")

# ============================================================================
# STEP 2: ACTIVATION - OVERALL FEATURE ADOPTION
# ============================================================================

print("\n[2/5] Analyzing feature activation rates...")

fig, axes = plt.subplots(1, 2, figsize=(20, 7))
fig.suptitle('PRICING FEATURES - ACTIVATION RATES', fontsize=18, fontweight='bold', y=0.98)

# Chart 1: Advanced Features Activation
ax1 = axes[0]
activation_rates = {}
for col, name in PRICING_FEATURES.items():
    if col in df.columns:
        activation_rates[name] = 100 * df[col].sum() / len(df)

sorted_features = sorted(activation_rates.items(), key=lambda x: x[1], reverse=True)
names = [x[0] for x in sorted_features]
rates = [x[1] for x in sorted_features]

colors = [COLORS['dark'] if r > 25 else COLORS['medium_dark'] if r > 15 else 
          COLORS['medium'] if r > 8 else COLORS['medium_light'] for r in rates]

bars = ax1.barh(range(len(names)), rates, color=colors, alpha=0.9, 
                edgecolor=COLORS['border'], linewidth=1.5)
ax1.set_yticks(range(len(names)))
ax1.set_yticklabels(names, fontsize=11)
ax1.set_xlabel('Activation Rate (%)', fontweight='bold', fontsize=12)
ax1.set_title('Feature Activation (% of Tours)', fontweight='bold', fontsize=14, pad=15)
ax1.grid(axis='x', alpha=0.2, color=COLORS['grey_dark'])
ax1.set_xlim(0, max(rates) * 1.25)

for i, (name, rate) in enumerate(sorted_features):
    col_name = [k for k, v in PRICING_FEATURES.items() if v == name][0]
    tours = df[col_name].sum()
    ax1.text(rate + max(rates)*0.02, i, f'{rate:.1f}%\n({int(tours):,})', 
            va='center', fontsize=10, fontweight='bold')

# Chart 2: Managed vs Not Managed
ax2 = axes[1]
managed_stats = []
for managed_val in [0, 1]:
    managed_df = df[df['is_managed'] == managed_val]
    label = 'Managed' if managed_val == 1 else 'Not Managed'
    
    for feat_col, feat_name in PRICING_FEATURES.items():
        if feat_col in managed_df.columns:
            rate = 100 * managed_df[feat_col].sum() / len(managed_df) if len(managed_df) > 0 else 0
            managed_stats.append({
                'Status': label,
                'Feature': feat_name,
                'Rate': rate
            })

managed_df_plot = pd.DataFrame(managed_stats)
managed_pivot = managed_df_plot.pivot(index='Feature', columns='Status', values='Rate')

x = np.arange(len(managed_pivot))
width = 0.35

bars1 = ax2.bar(x - width/2, managed_pivot['Not Managed'], width, 
               label='Not Managed', color=COLORS['grey'], alpha=0.9,
               edgecolor=COLORS['border'], linewidth=1.5)
bars2 = ax2.bar(x + width/2, managed_pivot['Managed'], width,
               label='Managed', color=COLORS['medium_dark'], alpha=0.9,
               edgecolor=COLORS['border'], linewidth=1.5)

ax2.set_xticks(x)
ax2.set_xticklabels(managed_pivot.index, rotation=25, ha='right', fontsize=10)
ax2.set_ylabel('Activation Rate (%)', fontweight='bold', fontsize=12)
ax2.set_title('Activation by Managed Status', fontweight='bold', fontsize=14, pad=15)
ax2.legend(fontsize=11, framealpha=0.95)
ax2.grid(axis='y', alpha=0.2, color=COLORS['grey_dark'])

plt.tight_layout()
plt.show()

print("✓ Activation analysis complete")

# ============================================================================
# STEP 3: ENGAGEMENT DEPTH - AVERAGE COUNTS
# ============================================================================

print("\n[3/5] Analyzing engagement depth...")

fig, axes = plt.subplots(2, 3, figsize=(22, 12))
fig.suptitle('ENGAGEMENT DEPTH - AVERAGE COUNTS PER TOUR', fontsize=18, fontweight='bold', y=0.98)

depth_metrics_list = list(DEPTH_METRICS.items())

for idx, (col, name) in enumerate(depth_metrics_list):
    if col not in df.columns:
        continue
    
    row = idx // 3
    col_idx = idx % 3
    ax = axes[row, col_idx]
    
    # Calculate average by GMV tier
    tier_avgs = []
    for tier in GMV_TIERS:
        tier_df = df[df['gmv_tier'] == tier]
        avg_val = tier_df[col].mean() if len(tier_df) > 0 else 0
        tier_avgs.append(avg_val)
    
    # Color by intensity
    colors_bar = [COLORS['light'] if v < 1 else COLORS['medium_light'] if v < 2 else
                  COLORS['medium'] if v < 3 else COLORS['medium_dark'] for v in tier_avgs]
    
    bars = ax.bar(range(len(GMV_TIERS)), tier_avgs, color=colors_bar, alpha=0.9,
                  edgecolor=COLORS['border'], linewidth=1.5)
    ax.set_xticks(range(len(GMV_TIERS)))
    ax.set_xticklabels(['Bottom 25%', '25-50%', '50-75%', 'Top 25%'], 
                       fontsize=9, rotation=20, ha='right')
    ax.set_ylabel('Average Count', fontweight='bold', fontsize=11)
    ax.set_title(name, fontweight='bold', fontsize=12, pad=10)
    ax.grid(axis='y', alpha=0.2, color=COLORS['grey_dark'])
    
    # Add value labels
    for i, val in enumerate(tier_avgs):
        ax.text(i, val + max(tier_avgs)*0.02, f'{val:.1f}', 
               ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.show()

print("✓ Engagement depth analysis complete")

# ============================================================================
# STEP 4: ACTIVATION HEATMAPS BY SEGMENT
# ============================================================================

print("\n[4/5] Creating segmentation heatmaps...")

fig, axes = plt.subplots(2, 1, figsize=(20, 14))
fig.suptitle('FEATURE ACTIVATION BY SEGMENT', fontsize=18, fontweight='bold', y=0.995)

# Chart 1: By Supplier Segment
ax1 = axes[0]
top_segments = df['supplier_segment'].value_counts().head(8).index.tolist()
heatmap_seg = []
seg_labels = []

for seg in top_segments:
    seg_df = df[df['supplier_segment'] == seg]
    row = []
    for col in PRICING_FEATURES.keys():
        if col in seg_df.columns:
            rate = 100 * seg_df[col].sum() / len(seg_df) if len(seg_df) > 0 else 0
            row.append(rate)
    heatmap_seg.append(row)
    seg_labels.append(f"{seg}\n(n={len(seg_df):,})")

heatmap_seg_df = pd.DataFrame(heatmap_seg, columns=list(PRICING_FEATURES.values()), index=seg_labels)

sns.heatmap(heatmap_seg_df, annot=True, fmt='.1f', cmap='YlOrRd', ax=ax1,
            cbar_kws={'label': 'Activation %'}, linewidths=2.5, linecolor='white',
            vmin=0, vmax=70, annot_kws={'fontsize': 10, 'fontweight': 'bold'})
ax1.set_xlabel('Pricing Features', fontweight='bold', fontsize=13)
ax1.set_ylabel('Supplier Segment', fontweight='bold', fontsize=13)
ax1.set_title('Activation Rate by Supplier Segment (%)', fontweight='bold', fontsize=15, pad=15)
ax1.set_yticklabels(seg_labels, rotation=0, fontsize=10)
ax1.set_xticklabels(list(PRICING_FEATURES.values()), rotation=30, ha='right', fontsize=11)

# Chart 2: By Activity Type
ax2 = axes[1]
top_activities = df['activity_type'].value_counts().head(10).index.tolist()
heatmap_act = []
act_labels = []

for act in top_activities:
    act_df = df[df['activity_type'] == act]
    row = []
    for col in PRICING_FEATURES.keys():
        if col in act_df.columns:
            rate = 100 * act_df[col].sum() / len(act_df) if len(act_df) > 0 else 0
            row.append(rate)
    heatmap_act.append(row)
    act_labels.append(f"{act}\n(n={len(act_df):,})")

heatmap_act_df = pd.DataFrame(heatmap_act, columns=list(PRICING_FEATURES.values()), index=act_labels)

sns.heatmap(heatmap_act_df, annot=True, fmt='.1f', cmap='YlOrRd', ax=ax2,
            cbar_kws={'label': 'Activation %'}, linewidths=2.5, linecolor='white',
            vmin=0, vmax=80, annot_kws={'fontsize': 10, 'fontweight': 'bold'})
ax2.set_xlabel('Pricing Features', fontweight='bold', fontsize=13)
ax2.set_ylabel('Activity Type', fontweight='bold', fontsize=13)
ax2.set_title('Activation Rate by Activity Type (%)', fontweight='bold', fontsize=15, pad=15)
ax2.set_yticklabels(act_labels, rotation=0, fontsize=9)
ax2.set_xticklabels(list(PRICING_FEATURES.values()), rotation=30, ha='right', fontsize=11)

plt.tight_layout()
plt.show()

print("✓ Segmentation heatmaps complete")

# ============================================================================
# STEP 5: GMV TIER COMPARISON
# ============================================================================

print("\n[5/5] Creating GMV tier comparison...")

fig, axes = plt.subplots(1, 2, figsize=(20, 7))
fig.suptitle('FEATURE ADOPTION BY GMV TIER', fontsize=18, fontweight='bold', y=0.98)

# Chart 1: Heatmap
ax1 = axes[0]
heatmap_tier = []
for tier in GMV_TIERS:
    tier_df = df[df['gmv_tier'] == tier]
    row = []
    for col in PRICING_FEATURES.keys():
        if col in tier_df.columns:
            rate = 100 * tier_df[col].sum() / len(tier_df) if len(tier_df) > 0 else 0
            row.append(rate)
    heatmap_tier.append(row)

heatmap_tier_df = pd.DataFrame(heatmap_tier, columns=list(PRICING_FEATURES.values()), index=GMV_TIERS)

sns.heatmap(heatmap_tier_df, annot=True, fmt='.1f', cmap='YlOrRd', ax=ax1,
            cbar_kws={'label': 'Activation %'}, linewidths=2.5, linecolor='white',
            vmin=0, vmax=50, annot_kws={'fontsize': 11, 'fontweight': 'bold'})
ax1.set_xlabel('Pricing Features', fontweight='bold', fontsize=13)
ax1.set_ylabel('GMV Tier', fontweight='bold', fontsize=13)
ax1.set_title('Activation Rate by GMV Tier (%)', fontweight='bold', fontsize=14, pad=15)
ax1.set_yticklabels(GMV_TIERS, rotation=0, fontsize=11)
ax1.set_xticklabels(list(PRICING_FEATURES.values()), rotation=30, ha='right', fontsize=10)

# Chart 2: Top 4 features comparison
ax2 = axes[1]
x = np.arange(len(GMV_TIERS))
width = 0.18

top_4_features = sorted_features[:4]
colors_comp = [COLORS['dark'], COLORS['medium_dark'], COLORS['medium'], COLORS['medium_light']]

for i, (feat_name, _) in enumerate(top_4_features):
    feat_col = [k for k, v in PRICING_FEATURES.items() if v == feat_name][0]
    rates_by_tier = []
    for tier in GMV_TIERS:
        tier_df = df[df['gmv_tier'] == tier]
        rate = 100 * tier_df[feat_col].sum() / len(tier_df) if len(tier_df) > 0 else 0
        rates_by_tier.append(rate)
    
    offset = width * (i - 1.5)
    ax2.bar(x + offset, rates_by_tier, width, label=feat_name,
            color=colors_comp[i], alpha=0.9, edgecolor=COLORS['border'], linewidth=1.2)

ax2.set_xticks(x)
ax2.set_xticklabels(GMV_TIERS, fontsize=11)
ax2.set_ylabel('Activation Rate (%)', fontweight='bold', fontsize=12)
ax2.set_xlabel('GMV Tier', fontweight='bold', fontsize=12)
ax2.set_title('Top 4 Features Comparison', fontweight='bold', fontsize=14, pad=15)
ax2.legend(loc='upper left', fontsize=10, framealpha=0.95)
ax2.grid(axis='y', alpha=0.2, color=COLORS['grey_dark'])

plt.tight_layout()
plt.show()

print("✓ GMV tier comparison complete")

# ============================================================================
# SUMMARY STATISTICS
# ============================================================================

print("\n" + "="*100)
print("SUMMARY STATISTICS")
print("="*100)

print("\n[OVERALL ACTIVATION]")
for feat_name, rate in sorted_features:
    col_name = [k for k, v in PRICING_FEATURES.items() if v == feat_name][0]
    tours = int(df[col_name].sum())
    print(f"  {feat_name}: {rate:.1f}% ({tours:,} tours)")

print("\n[ENGAGEMENT DEPTH - Overall Averages]")
for col, name in DEPTH_METRICS.items():
    if col in df.columns:
        avg_val = df[col].mean()
        median_val = df[col].median()
        print(f"  {name}: Avg={avg_val:.2f}, Median={median_val:.1f}")

print("\n[BY GMV TIER - Top Feature]")
top_feature_name = sorted_features[0][0]
top_feature_col = [k for k, v in PRICING_FEATURES.items() if v == top_feature_name][0]
for tier in GMV_TIERS:
    tier_df = df[df['gmv_tier'] == tier]
    rate = 100 * tier_df[top_feature_col].sum() / len(tier_df)
    print(f"  {tier}: {rate:.1f}% ({len(tier_df):,} tours)")

print("\n" + "="*100)
print("✓ ANALYSIS COMPLETE")
print("="*100)


In [0]:
# ============================================================================
# GET ALL UNIQUE INDIVIDUAL PRICING CATEGORIES - PURE SQL
# ============================================================================

print("="*100)
print("ALL UNIQUE INDIVIDUAL PRICING CATEGORIES")
print("="*100)

unique_categories_sql = """
  SELECT 
    category.ticketCategory AS ticket_category,
    COUNT(*) AS occurrence_count
  FROM production.supply_analytics.pricing_feature_audit_base
  LATERAL VIEW EXPLODE(individual_categories) AS category
  WHERE individual_categories IS NOT NULL
  GROUP BY category.ticketCategory
  ORDER BY occurrence_count DESC
"""

categories_df = spark.sql(unique_categories_sql).toPandas()

print(f"\n✓ Found {len(categories_df)} unique ticket categories:\n")
print(categories_df.to_string(index=False))

print("\n" + "="*100)
print("FULL BREAKDOWN")
print("="*100)

total_occurrences = categories_df['occurrence_count'].sum()

for idx, row in categories_df.iterrows():
    cat = row['ticket_category']
    count = row['occurrence_count']
    pct = 100 * count / total_occurrences
    print(f"  {cat}: {count:,} ({pct:.1f}%)")

print(f"\n  TOTAL: {total_occurrences:,} category occurrences")
print("="*100)


In [0]:
# Databricks notebook source

# MARKDOWN
"""
# Pricing Feature Adoption Audit

## Objective
This notebook analyzes pricing feature adoption across GetYourGuide's supplier base to identify which segments are utilizing advanced pricing capabilities and which segments represent opportunities for feature education and adoption.

## Business Questions
1. What percentage of tours use advanced pricing features?
2. Which supplier segments (by volume, maturity, category) have highest/lowest adoption?
3. Are there specific feature combinations that drive better outcomes?
4. What opportunities exist to increase feature penetration?

## Features Analyzed (10 Total)
- Individual Pricing
- Group Pricing  
- Pricing without API (Native)
- Pricing with API
- Scales without API (Native Scales)
- Scales with API
- Live Pricing and Availability
- Non-live Price and Availability
- Add-ons
- Options
"""

# COMMAND ----------

# MARKDOWN
"""
## Section 1: Data Loading and Type Conversion

**What this does:** 
- Loads the pricing feature data from the Spark table
- Converts all columns to proper numeric types to avoid calculation errors
- Handles missing values by filling with 0
- Converts dates to datetime format

**Why it matters:**
Data type issues cause errors in aggregations and binning. This ensures clean numeric data for all calculations.
"""

# COMMAND ----------

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import rcParams
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Load data
print("Loading data from Spark...")
spark_df = spark.table("production.supply_analytics.pricing_feature_combined")
df = spark_df.toPandas()

print(f"Loaded {len(df):,} tours from {df['supplier_id'].nunique():,} suppliers")

# Fix data types - critical for avoiding errors
numeric_cols = ['gmv_l12mo', 'bookings_l12mo', 'has_group_pricing', 'has_individual_pricing',
                'has_addons', 'has_options', 'has_native_scales', 'has_api_scales', 
                'has_live_dynamic_pricing', 'has_non_live_pricing', 'uses_api_pricing',
                'has_scale_pricing', 'is_managed', 'num_individual_categories']

for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)

if 'date_of_first_online' in df.columns:
    df['date_of_first_online'] = pd.to_datetime(df['date_of_first_online'], errors='coerce')

print("✓ Data types converted")
display(df.head())

# COMMAND ----------

# MARKDOWN
"""
## Section 2: Segmentation Creation

**What this does:**
- Creates Volume Tiers based on GMV quartiles (Emerging, Growing, Established, Top Performers)
- Creates Market Maturity segments based on time on platform (New 0-6mo, Developing 6-18mo, Mature 18mo+)
- Identifies API connection status and managed supplier status

**Why it matters:**
Different supplier segments may have different adoption patterns. High-volume suppliers may adopt features faster. Newer suppliers may need more education. This allows us to identify where to focus enablement efforts.

**Analytical approach:**
Using quartiles for volume ensures equal-sized cohorts for comparison. Time-based maturity segments align with typical supplier lifecycle stages.
"""

# COMMAND ----------

# Volume tier - using percentile-based segmentation
gmv_25 = df['gmv_l12mo'].quantile(0.25)
gmv_50 = df['gmv_l12mo'].quantile(0.50)
gmv_75 = df['gmv_l12mo'].quantile(0.75)

df['volume_tier'] = 'Top Performers'
df.loc[df['gmv_l12mo'] <= gmv_75, 'volume_tier'] = 'Established'
df.loc[df['gmv_l12mo'] <= gmv_50, 'volume_tier'] = 'Growing'
df.loc[df['gmv_l12mo'] <= gmv_25, 'volume_tier'] = 'Emerging'

# Market maturity - time-based segmentation
df['days_on_platform'] = (datetime.now() - df['date_of_first_online']).dt.days
df['market_maturity'] = 'Mature (18+ months)'
df.loc[df['days_on_platform'] <= 540, 'market_maturity'] = 'Developing (6-18 months)'
df.loc[df['days_on_platform'] <= 180, 'market_maturity'] = 'New (0-6 months)'

# Connection and management status
df['connection_status'] = df['uses_api_pricing'].apply(lambda x: 'Connected' if x == 1 else 'Not Connected')
df['managed_status'] = df['is_managed'].apply(lambda x: 'Managed' if x == 1 else 'Not Managed')

# Verify segment distribution
print("Volume Tier Distribution:")
display(df['volume_tier'].value_counts())
print("\nMarket Maturity Distribution:")
display(df['market_maturity'].value_counts())

# COMMAND ----------

# MARKDOWN
"""
## Section 3: Feature Count Calculation and Adoption Level Assignment

**What this does:**
- Counts how many of the 10 pricing features each tour uses
- Creates 5 adoption levels: Non-adopters (0), Low (1-2), Medium (3-5), High (6-7), Power (8+)

**Why it matters:**
This is the core metric for measuring feature adoption maturity. Tours using more features likely generate more revenue per customer through upselling and dynamic optimization. This metric helps identify tours at risk (non-adopters) and best-in-class (power users).

**Analytical approach:**
Additive feature counting treats all features as equally valuable. The 5-tier classification creates actionable segments for targeted interventions.
"""

# COMMAND ----------

# Build feature list dynamically based on available columns
feature_cols = []
if 'has_individual_pricing' in df.columns:
    feature_cols.append('has_individual_pricing')
if 'has_group_pricing' in df.columns:
    feature_cols.append('has_group_pricing')
if 'uses_api_pricing' in df.columns:
    feature_cols.append('uses_api_pricing')
if 'has_native_scales' in df.columns:
    feature_cols.append('has_native_scales')
if 'has_api_scales' in df.columns:
    feature_cols.append('has_api_scales')
if 'has_live_dynamic_pricing' in df.columns:
    feature_cols.append('has_live_dynamic_pricing')
if 'has_non_live_pricing' in df.columns:
    feature_cols.append('has_non_live_pricing')
if 'has_addons' in df.columns:
    feature_cols.append('has_addons')
if 'has_options' in df.columns:
    feature_cols.append('has_options')

# Calculate native pricing as inverse of API pricing
if 'uses_api_pricing' in df.columns:
    df['has_native_pricing'] = (1 - df['uses_api_pricing']).clip(0, 1)
    feature_cols.append('has_native_pricing')

# Sum features per tour
df['advanced_features_count'] = df[feature_cols].sum(axis=1)

print(f"Counting {len(feature_cols)} features: {feature_cols}")
print(f"Max features used by any tour: {df['advanced_features_count'].max()}")
print(f"Mean features per tour: {df['advanced_features_count'].mean():.2f}")
print(f"Median features per tour: {df['advanced_features_count'].median():.0f}")

# Create adoption levels
df['adoption_level'] = 'Power (8+)'
df.loc[df['advanced_features_count'] <= 7, 'adoption_level'] = 'High (6-7)'
df.loc[df['advanced_features_count'] <= 5, 'adoption_level'] = 'Medium (3-5)'
df.loc[df['advanced_features_count'] <= 2, 'adoption_level'] = 'Low (1-2)'
df.loc[df['advanced_features_count'] == 0, 'adoption_level'] = 'Non-adopters'

df['uses_multiple_categories'] = (df['num_individual_categories'] >= 3).astype(int)

# Display distribution
print("\nAdoption Level Distribution:")
display(df['adoption_level'].value_counts())

# COMMAND ----------

# MARKDOWN
"""
## Section 4: Visualization Setup

**What this does:**
- Sets consistent color palettes and styling for all charts
- Defines segment ordering for consistent visualization
- Configures matplotlib parameters

**Why it matters:**
Consistent visual styling makes the analysis professional and easier to interpret. Ordered categories (e.g., Emerging → Top Performers) show progression naturally.
"""

# COMMAND ----------

# Styling
rcParams['font.family'] = 'sans-serif'
rcParams['font.sans-serif'] = ['Arial', 'DejaVu Sans']
rcParams['font.size'] = 10

COLORS = {'primary': '#FF6B35', 'secondary': '#F77F51', 'tertiary': '#FFA07A', 
          'quaternary': '#C0C0C0', 'quinary': '#808080', 'dark': '#404040'}
PALETTE_5 = ['#FF6B35', '#FF8C5A', '#FFAD7F', '#CCCCCC', '#808080']

# Define consistent orderings
volume_order = ['Emerging', 'Growing', 'Established', 'Top Performers']
maturity_order = ['New (0-6 months)', 'Developing (6-18 months)', 'Mature (18+ months)']
adoption_order = ['Non-adopters', 'Low (1-2)', 'Medium (3-5)', 'High (6-7)', 'Power (8+)']
segment_order = ['Enterprise', 'Mid-Market', 'SMB', 'Micro']

print("✓ Styling configured")

# COMMAND ----------

# MARKDOWN
"""
## Chart 1: Overall Feature Adoption Overview

**What this analyzes:**
- Distribution of tours across the 5 adoption levels
- Individual feature adoption rates (which features are most/least used)
- Ticket category complexity distribution
- Overall distribution of feature count

**Business insights to extract:**
- What percentage of our catalog is at risk (non-adopters)?
- Which features have low adoption and need education/enablement?
- Are suppliers using category complexity effectively?

**Interpretation guide:**
- High percentage of non-adopters = opportunity for revenue growth through feature education
- Wide gap between top features and bottom features = some features may need better UX or clearer value prop
"""

# COMMAND ----------

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Pricing Feature Adoption Overview', fontsize=16, fontweight='bold', y=0.995)

# Adoption level distribution
ax = axes[0, 0]
adoption_pcts = (df['adoption_level'].value_counts().reindex(adoption_order) / len(df) * 100).values
bars = ax.barh(adoption_order, adoption_pcts, color=PALETTE_5, edgecolor='white', linewidth=2)
ax.set_xlabel('Percentage of Tours', fontweight='bold')
ax.set_title(f'Feature Adoption Levels ({len(feature_cols)} Features)', fontweight='bold', pad=10)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
for i, val in enumerate(adoption_pcts):
    ax.text(val + 1, i, f'{val:.1f}%', va='center', fontweight='bold', color=COLORS['dark'])

# Individual feature adoption
ax = axes[0, 1]
features = {}
if 'has_group_pricing' in df.columns:
    features['Group Pricing'] = df['has_group_pricing'].sum()
if 'has_individual_pricing' in df.columns:
    features['Individual Pricing'] = df['has_individual_pricing'].sum()
if 'has_addons' in df.columns:
    features['Add-ons'] = df['has_addons'].sum()
if 'has_options' in df.columns:
    features['Options'] = df['has_options'].sum()
if 'uses_api_pricing' in df.columns:
    features['API Pricing'] = df['uses_api_pricing'].sum()
if 'has_native_scales' in df.columns:
    features['Native Scales'] = df['has_native_scales'].sum()
if 'has_api_scales' in df.columns:
    features['API Scales'] = df['has_api_scales'].sum()
if 'has_live_dynamic_pricing' in df.columns:
    features['Live Pricing'] = df['has_live_dynamic_pricing'].sum()

feature_pcts = [(v / len(df)) * 100 for v in features.values()]
colors_bar = ['#FF6B35', '#FF7A47', '#FF8C5A', '#FF9E6C', '#FFAD7F', '#FFBC91', '#FFCBA4', '#D9D9D9']
bars = ax.barh(list(features.keys()), feature_pcts, color=colors_bar[:len(features)], 
               edgecolor='white', linewidth=2)
ax.set_xlabel('Adoption Rate (%)', fontweight='bold')
ax.set_title('Individual Feature Adoption', fontweight='bold', pad=10)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
for i, val in enumerate(feature_pcts):
    ax.text(val + 1, i, f'{val:.1f}%', va='center', fontweight='bold', color=COLORS['dark'])

# Category complexity
ax = axes[1, 0]
cat_dist = {'Adult Only': (df['num_individual_categories'] == 1).sum(),
            'Adult + Child': (df['num_individual_categories'] == 2).sum(),
            '3 Categories': (df['num_individual_categories'] == 3).sum(),
            '4+ Categories': (df['num_individual_categories'] >= 4).sum()}
cat_pcts = [(v / len(df)) * 100 for v in cat_dist.values()]
bars = ax.bar(range(len(cat_pcts)), cat_pcts, color=PALETTE_5[:4], edgecolor='white', linewidth=2)
ax.set_xticks(range(len(cat_dist)))
ax.set_xticklabels(cat_dist.keys(), rotation=0)
ax.set_ylabel('Percentage of Tours', fontweight='bold')
ax.set_title('Ticket Category Complexity', fontweight='bold', pad=10)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
for i, val in enumerate(cat_pcts):
    ax.text(i, val + 1, f'{val:.1f}%', ha='center', fontweight='bold', color=COLORS['dark'])

# Feature count distribution
ax = axes[1, 1]
feature_dist = df['advanced_features_count'].value_counts().sort_index()
ax.bar(feature_dist.index, feature_dist.values, color='#FF6B35', edgecolor='white', linewidth=2)
ax.set_xlabel('Number of Features Used', fontweight='bold')
ax.set_ylabel('Number of Tours', fontweight='bold')
ax.set_title('Distribution of Feature Count', fontweight='bold', pad=10)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
display(fig)
plt.close()

# COMMAND ----------

# MARKDOWN
"""
## Chart 2: Adoption by Volume Tier

**What this analyzes:**
- How feature adoption differs between low-volume and high-volume suppliers
- Average feature count by volume tier

**Business hypothesis to test:**
Higher volume suppliers should have higher adoption rates because:
1. They have more resources to invest in pricing optimization
2. They see clearer ROI from advanced features
3. They may receive more support from GetYourGuide account managers

**Key metrics:**
- Percentage of Power Users in Top Performers vs Emerging
- Adoption gap between tiers (indicates opportunity or resource constraints)
"""

# COMMAND ----------

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Feature Adoption by Volume Tier', fontsize=16, fontweight='bold', y=1.02)

# Adoption distribution by volume tier
ax = axes[0]
volume_adoption = pd.crosstab(df['volume_tier'], df['adoption_level'], normalize='index') * 100
x = np.arange(len(volume_order))
width = 0.18
for i, adoption in enumerate(adoption_order):
    values = [volume_adoption.loc[vol, adoption] if vol in volume_adoption.index and adoption in volume_adoption.columns else 0 for vol in volume_order]
    ax.bar(x + i * width, values, width, label=adoption, color=PALETTE_5[i], edgecolor='white', linewidth=1.5)
ax.set_xlabel('Volume Tier', fontweight='bold')
ax.set_ylabel('Percentage (%)', fontweight='bold')
ax.set_title('Adoption Distribution Within Volume Tier', fontweight='bold', pad=10)
ax.set_xticks(x + width * 2)
ax.set_xticklabels(volume_order, rotation=0, fontsize=9)
ax.legend(loc='upper left', fontsize=7)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.grid(axis='y', alpha=0.3, linestyle='--', linewidth=0.5)

# Average feature count by volume tier
ax = axes[1]
avg_features_vol = df.groupby('volume_tier')['advanced_features_count'].mean().reindex(volume_order)
bars = ax.bar(volume_order, avg_features_vol, color=['#FF6B35', '#FF8C5A', '#FFAD7F', '#D9D9D9'], 
              edgecolor='white', linewidth=2)
ax.set_xlabel('Volume Tier', fontweight='bold')
ax.set_ylabel('Average Features Used', fontweight='bold')
ax.set_title('Average Feature Count by Volume Tier', fontweight='bold', pad=10)
ax.set_xticklabels(volume_order, rotation=0, fontsize=9)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.grid(axis='y', alpha=0.3, linestyle='--', linewidth=0.5)
for i, val in enumerate(avg_features_vol):
    ax.text(i, val + 0.1, f'{val:.2f}', ha='center', fontweight='bold', color=COLORS['dark'])

plt.tight_layout()
display(fig)
plt.close()

# COMMAND ----------

# MARKDOWN
"""
## Chart 3: Adoption by Market Maturity

**What this analyzes:**
- How feature adoption evolves as suppliers mature on the platform
- Average feature usage by time on platform

**Business hypothesis to test:**
Mature suppliers should have higher adoption because:
1. They have had more time to learn the platform
2. They have established processes and can invest in optimization
3. They may have received more training over time

**Counterpoint to watch for:**
If NEW suppliers have higher adoption, it may indicate:
- Improved onboarding programs
- Better product education for new suppliers
- Selection bias (only sophisticated suppliers joining recently)
"""

# COMMAND ----------

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Feature Adoption by Market Maturity', fontsize=16, fontweight='bold', y=1.02)

# Adoption distribution by maturity
ax = axes[0]
mat_adoption = pd.crosstab(df['market_maturity'], df['adoption_level'], normalize='index') * 100
x = np.arange(len(maturity_order))
for i, adoption in enumerate(adoption_order):
    values = [mat_adoption.loc[mat, adoption] if mat in mat_adoption.index and adoption in mat_adoption.columns else 0 for mat in maturity_order]
    ax.bar(x + i * 0.18, values, 0.18, label=adoption, color=PALETTE_5[i], edgecolor='white', linewidth=1.5)
ax.set_xlabel('Market Maturity', fontweight='bold')
ax.set_ylabel('Percentage (%)', fontweight='bold')
ax.set_title('Adoption Distribution by Maturity', fontweight='bold', pad=10)
ax.set_xticks(x + 0.36)
ax.set_xticklabels(['New\n(0-6m)', 'Developing\n(6-18m)', 'Mature\n(18m+)'], rotation=0)
ax.legend(loc='upper left', fontsize=7)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.grid(axis='y', alpha=0.3, linestyle='--', linewidth=0.5)

# Average feature count by maturity
ax = axes[1]
avg_features_mat = df.groupby('market_maturity')['advanced_features_count'].mean().reindex(maturity_order)
bars = ax.bar(range(len(maturity_order)), avg_features_mat, color=['#FF6B35', '#FF8C5A', '#FFAD7F'], 
              edgecolor='white', linewidth=2)
ax.set_xlabel('Market Maturity', fontweight='bold')
ax.set_ylabel('Average Features Used', fontweight='bold')
ax.set_title('Average Feature Count by Maturity', fontweight='bold', pad=10)
ax.set_xticks(range(len(maturity_order)))
ax.set_xticklabels(['New\n(0-6m)', 'Developing\n(6-18m)', 'Mature\n(18m+)'], rotation=0)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.grid(axis='y', alpha=0.3, linestyle='--', linewidth=0.5)
for i, val in enumerate(avg_features_mat):
    ax.text(i, val + 0.1, f'{val:.2f}', ha='center', fontweight='bold', color=COLORS['dark'])

plt.tight_layout()
display(fig)
plt.close()

# COMMAND ----------

# MARKDOWN
"""
## Chart 4: Adoption by Activity Category

**What this analyzes:**
- Which activity types (tours, attractions, food tours, etc.) have highest feature adoption
- Average feature usage by activity category

**Business insights:**
- Some activity types may be more complex and require more pricing features (e.g., multi-day tours)
- Some categories may be under-utilizing features relative to their potential
- Categories sorted by average feature count to identify leaders and laggards

**Action items:**
- Low-adoption categories may need category-specific enablement materials
- High-adoption categories can be case studies for others
"""

# COMMAND ----------

fig, axes = plt.subplots(2, 1, figsize=(14, 10))
fig.suptitle('Feature Adoption by Activity Category', fontsize=16, fontweight='bold', y=0.995)

# Adoption distribution by activity
ax = axes[0]
act_adoption = pd.crosstab(df['activity_type'], df['adoption_level'], normalize='index') * 100
cat_order = df.groupby('activity_type')['advanced_features_count'].mean().sort_values(ascending=True).index.tolist()
act_adoption_sorted = act_adoption.reindex(cat_order)
x = np.arange(len(cat_order))
for i, adoption in enumerate(adoption_order):
    values = [act_adoption_sorted.loc[cat, adoption] if adoption in act_adoption_sorted.columns else 0 for cat in cat_order]
    ax.barh(x + i * 0.18, values, 0.18, label=adoption, color=PALETTE_5[i], edgecolor='white', linewidth=1.5)
ax.set_ylabel('Activity Category', fontweight='bold')
ax.set_xlabel('Percentage (%)', fontweight='bold')
ax.set_title('Adoption Within Each Activity Category', fontweight='bold', pad=10)
ax.set_yticks(x + 0.36)
ax.set_yticklabels(cat_order, fontsize=8)
ax.legend(loc='lower right', fontsize=7, ncol=2)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.grid(axis='x', alpha=0.3, linestyle='--', linewidth=0.5)

# Average feature count by activity
ax = axes[1]
avg_features_act = df.groupby('activity_type')['advanced_features_count'].mean().reindex(cat_order)
bars = ax.barh(cat_order, avg_features_act, color='#FF6B35', edgecolor='white', linewidth=2)
ax.set_ylabel('Activity Category', fontweight='bold')
ax.set_xlabel('Average Features Used', fontweight='bold')
ax.set_title('Average Feature Count by Activity', fontweight='bold', pad=10)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.grid(axis='x', alpha=0.3, linestyle='--', linewidth=0.5)

plt.tight_layout()
display(fig)
plt.close()

# COMMAND ----------

# MARKDOWN
"""
## Chart 5: Adoption by Supplier Segment

**What this analyzes:**
- Feature adoption across Enterprise, Mid-Market, SMB, and Micro suppliers
- Average feature count by segment

**Business context:**
- Enterprise suppliers have dedicated account managers and technical resources
- SMB and Micro suppliers may need self-service education and simpler onboarding
- Segment-specific adoption gaps indicate where to invest in support resources
"""

# COMMAND ----------

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Feature Adoption by Supplier Segment', fontsize=16, fontweight='bold', y=1.02)

# Adoption distribution by segment
ax = axes[0]
seg_adoption = pd.crosstab(df['supplier_segment'], df['adoption_level'], normalize='index') * 100
x = np.arange(len(segment_order))
for i, adoption in enumerate(adoption_order):
    values = [seg_adoption.loc[seg, adoption] if seg in seg_adoption.index and adoption in seg_adoption.columns else 0 for seg in segment_order]
    ax.bar(x + i * 0.18, values, 0.18, label=adoption, color=PALETTE_5[i], edgecolor='white', linewidth=1.5)
ax.set_xlabel('Supplier Segment', fontweight='bold')
ax.set_ylabel('Percentage (%)', fontweight='bold')
ax.set_title('Adoption by Segment', fontweight='bold', pad=10)
ax.set_xticks(x + 0.36)
ax.set_xticklabels(segment_order, rotation=0)
ax.legend(loc='upper right', fontsize=7)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.grid(axis='y', alpha=0.3, linestyle='--', linewidth=0.5)

# Average feature count by segment
ax = axes[1]
avg_features_seg = df.groupby('supplier_segment')['advanced_features_count'].mean().reindex(segment_order)
bars = ax.bar(segment_order, avg_features_seg, color=['#FF6B35', '#FF8C5A', '#FFAD7F', '#D9D9D9'], 
              edgecolor='white', linewidth=2)
ax.set_xlabel('Supplier Segment', fontweight='bold')
ax.set_ylabel('Average Features Used', fontweight='bold')
ax.set_title('Average Feature Count by Segment', fontweight='bold', pad=10)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.grid(axis='y', alpha=0.3, linestyle='--', linewidth=0.5)
for i, val in enumerate(avg_features_seg):
    ax.text(i, val + 0.1, f'{val:.2f}', ha='center', fontweight='bold', color=COLORS['dark'])

plt.tight_layout()
display(fig)
plt.close()

# COMMAND ----------

# MARKDOWN
"""
## Chart 6: Adoption by Connection and Management Status

**What this analyzes:**
- API-connected vs non-connected suppliers (API integration enables advanced features)
- Managed vs non-managed suppliers (managed suppliers receive direct account support)

**Hypothesis:**
- API-connected suppliers should have higher adoption (API unlocks scaling features)
- Managed suppliers should have higher adoption (direct support drives feature usage)

**Business value:**
- Quantifies the impact of API connectivity on feature adoption
- Validates the ROI of supplier management programs
"""

# COMMAND ----------

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Adoption by Connection and Management Status', fontsize=16, fontweight='bold', y=0.995)

# Adoption by connection status
ax = axes[0, 0]
conn_adoption = pd.crosstab(df['connection_status'], df['adoption_level'], normalize='index') * 100
conn_order = ['Not Connected', 'Connected']
x = np.arange(len(conn_order))
for i, adoption in enumerate(adoption_order):
    values = [conn_adoption.loc[conn, adoption] if conn in conn_adoption.index and adoption in conn_adoption.columns else 0 for conn in conn_order]
    ax.bar(x + i * 0.18, values, 0.18, label=adoption, color=PALETTE_5[i], edgecolor='white', linewidth=1.5)
ax.set_xlabel('Connection Status', fontweight='bold')
ax.set_ylabel('Percentage (%)', fontweight='bold')
ax.set_title('Adoption by Connection', fontweight='bold', pad=10)
ax.set_xticks(x + 0.36)
ax.set_xticklabels(conn_order, rotation=0)
ax.legend(loc='upper left', fontsize=6)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.grid(axis='y', alpha=0.3, linestyle='--', linewidth=0.5)

# Average features by connection
ax = axes[0, 1]
avg_feat_conn = df.groupby('connection_status')['advanced_features_count'].mean().reindex(conn_order)
bars = ax.bar(conn_order, avg_feat_conn, color=['#FF6B35', '#FFAD7F'], edgecolor='white', linewidth=2)
ax.set_xlabel('Connection Status', fontweight='bold')
ax.set_ylabel('Average Features Used', fontweight='bold')
ax.set_title('Features by Connection', fontweight='bold', pad=10)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.grid(axis='y', alpha=0.3, linestyle='--', linewidth=0.5)
for i, val in enumerate(avg_feat_conn):
    ax.text(i, val + 0.1, f'{val:.2f}', ha='center', fontweight='bold', color=COLORS['dark'])

# Adoption by managed status
ax = axes[1, 0]
mgd_adoption = pd.crosstab(df['managed_status'], df['adoption_level'], normalize='index') * 100
mgd_order = ['Not Managed', 'Managed']
x = np.arange(len(mgd_order))
for i, adoption in enumerate(adoption_order):
    values = [mgd_adoption.loc[mgd, adoption] if mgd in mgd_adoption.index and adoption in mgd_adoption.columns else 0 for mgd in mgd_order]
    ax.bar(x + i * 0.18, values, 0.18, label=adoption, color=PALETTE_5[i], edgecolor='white', linewidth=1.5)
ax.set_xlabel('Management Status', fontweight='bold')
ax.set_ylabel('Percentage (%)', fontweight='bold')
ax.set_title('Adoption by Managed Status', fontweight='bold', pad=10)
ax.set_xticks(x + 0.36)
ax.set_xticklabels(mgd_order, rotation=0)
ax.legend(loc='upper left', fontsize=6)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.grid(axis='y', alpha=0.3, linestyle='--', linewidth=0.5)

# Average features by managed status
ax = axes[1, 1]
avg_feat_mgd = df.groupby('managed_status')['advanced_features_count'].mean().reindex(mgd_order)
bars = ax.bar(mgd_order, avg_feat_mgd, color=['#FF6B35', '#FFAD7F'], edgecolor='white', linewidth=2)
ax.set_xlabel('Management Status', fontweight='bold')
ax.set_ylabel('Average Features Used', fontweight='bold')
ax.set_title('Features by Managed Status', fontweight='bold', pad=10)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.grid(axis='y', alpha=0.3, linestyle='--', linewidth=0.5)
for i, val in enumerate(avg_feat_mgd):
    ax.text(i, val + 0.1, f'{val:.2f}', ha='center', fontweight='bold', color=COLORS['dark'])

plt.tight_layout()
display(fig)
plt.close()

# COMMAND ----------

# MARKDOWN
"""
## Chart 7: Cross-Segment Analysis - Volume × Maturity

**What this analyzes:**
- Power user adoption rates across combinations of volume tier and maturity
- Average feature usage across combinations

**Why cross-segment matters:**
Single-dimension analysis can hide important patterns. For example:
- Are NEW high-volume suppliers adopting faster than MATURE low-volume suppliers?
- Do high-volume suppliers start strong or grow into features over time?

**Heatmap interpretation:**
- Hot spots (orange/red) = high adoption segments to use as case studies
- Cold spots (light) = opportunity segments for targeted enablement
"""

# COMMAND ----------

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Cross-Segment: Volume Tier × Market Maturity', fontsize=16, fontweight='bold', y=1.02)

# Power user adoption heatmap
ax = axes[0]
power_cross = pd.crosstab(df['volume_tier'], df['market_maturity'], 
                          values=(df['adoption_level'] == 'Power (8+)').astype(int), aggfunc='mean') * 100
power_cross = power_cross.reindex(index=volume_order, columns=maturity_order)
im = ax.imshow(power_cross.values, cmap='Oranges', aspect='auto', vmin=0, vmax=power_cross.values.max())
ax.set_xticks(range(len(maturity_order)))
ax.set_xticklabels(['New\n(0-6m)', 'Developing\n(6-18m)', 'Mature\n(18m+)'], rotation=0)
ax.set_yticks(range(len(volume_order)))
ax.set_yticklabels(volume_order)
ax.set_xlabel('Market Maturity', fontweight='bold')
ax.set_ylabel('Volume Tier', fontweight='bold')
ax.set_title('Power User Adoption Rate (%)', fontweight='bold', pad=10)
for i in range(len(volume_order)):
    for j in range(len(maturity_order)):
        ax.text(j, i, f'{power_cross.values[i, j]:.1f}%', ha='center', va='center', 
               color='white' if power_cross.values[i, j] > power_cross.values.max()/2 else 'black', fontweight='bold')
cbar = plt.colorbar(im, ax=ax)
cbar.set_label('Adoption Rate (%)', rotation=270, labelpad=20, fontweight='bold')

# Average features heatmap
ax = axes[1]
avg_feat_cross = pd.crosstab(df['volume_tier'], df['market_maturity'], 
                             values=df['advanced_features_count'], aggfunc='mean')
avg_feat_cross = avg_feat_cross.reindex(index=volume_order, columns=maturity_order)
im = ax.imshow(avg_feat_cross.values, cmap='Oranges', aspect='auto', vmin=0, vmax=avg_feat_cross.values.max())
ax.set_xticks(range(len(maturity_order)))
ax.set_xticklabels(['New\n(0-6m)', 'Developing\n(6-18m)', 'Mature\n(18m+)'], rotation=0)
ax.set_yticks(range(len(volume_order)))
ax.set_yticklabels(volume_order)
ax.set_xlabel('Market Maturity', fontweight='bold')
ax.set_ylabel('Volume Tier', fontweight='bold')
ax.set_title('Average Features Used', fontweight='bold', pad=10)
for i in range(len(volume_order)):
    for j in range(len(maturity_order)):
        ax.text(j, i, f'{avg_feat_cross.values[i, j]:.2f}', ha='center', va='center',
               color='white' if avg_feat_cross.values[i, j] > avg_feat_cross.values.max()/2 else 'black', fontweight='bold')
cbar = plt.colorbar(im, ax=ax)
cbar.set_label('Avg Features', rotation=270, labelpad=20, fontweight='bold')

plt.tight_layout()
display(fig)
plt.close()

# COMMAND ----------

# MARKDOWN
"""
## Summary Statistics and Key Takeaways

**What this provides:**
- Overall adoption rate (percentage using at least one feature)
- Power user rate (percentage using 8+ features)
- Average features per tour
- Distribution summary

**Use these metrics to:**
- Track adoption improvement over time (run this monthly)
- Set targets for product and enablement teams
- Communicate impact to leadership
"""

# COMMAND ----------

# Calculate summary metrics
adopters = (df['adoption_level'] != 'Non-adopters').sum()
adoption_rate = (adopters / len(df)) * 100
power_users = (df['adoption_level'] == 'Power (8+)').sum()
power_rate = (power_users / len(df)) * 100
avg_feat = df['advanced_features_count'].mean()
median_feat = df['advanced_features_count'].median()

print("="*60)
print("PRICING FEATURE ADOPTION SUMMARY")
print("="*60)
print(f"\nTotal tours analyzed: {len(df):,}")
print(f"Total suppliers: {df['supplier_id'].nunique():,}")
print(f"Features tracked: {len(feature_cols)}")
print(f"\nOVERALL ADOPTION METRICS:")
print(f"  • Overall adoption rate: {adoption_rate:.1f}%")
print(f"  • Non-adopters: {100-adoption_rate:.1f}%")
print(f"  • Power users (8+ features): {power_rate:.1f}%")
print(f"  • Average features per tour: {avg_feat:.2f}")
print(f"  • Median features per tour: {median_feat:.0f}")
print(f"\nFEATURES ANALYZED:")
for i, feat in enumerate(feature_cols, 1):
    print(f"  {i}. {feat}")
print("="*60)

# Show adoption breakdown
print("\nADOPTION LEVEL BREAKDOWN:")
adoption_breakdown = df['adoption_level'].value_counts().reindex(adoption_order)
for level in adoption_order:
    count = adoption_breakdown.get(level, 0)
    pct = (count / len(df)) * 100
    print(f"  {level:20s}: {count:6,} tours ({pct:5.1f}%)")

print("\n" + "="*60)


In [0]:
# Databricks notebook source

# MARKDOWN
"""
# Pricing Feature Adoption Audit

## Objective
This notebook analyzes pricing feature adoption across GetYourGuide's supplier base to identify which segments are utilizing advanced pricing capabilities and which segments represent opportunities for feature education and adoption.

## Business Questions
1. What percentage of tours use advanced pricing features?
2. Which supplier segments (by volume, maturity, category) have highest/lowest adoption?
3. Are there specific feature combinations that drive better outcomes?
4. What opportunities exist to increase feature penetration?

## Features Analyzed (10 Total)
- Individual Pricing
- Group Pricing  
- Pricing without API (Native)
- Pricing with API
- Scales without API (Native Scales)
- Scales with API
- Live Pricing and Availability
- Non-live Price and Availability
- Add-ons
- Options
"""

# COMMAND ----------

# MARKDOWN
"""
## Section 1: Data Loading and Type Conversion

**What this does:** 
- Loads the pricing feature data from the Spark table
- Converts all columns to proper numeric types to avoid calculation errors
- Handles missing values by filling with 0
- Converts dates to datetime format

**Why it matters:**
Data type issues cause errors in aggregations and binning. This ensures clean numeric data for all calculations.
"""

# COMMAND ----------

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import rcParams
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Load data
print("Loading data from Spark...")
spark_df = spark.table("production.supply_analytics.pricing_feature_combined")
df = spark_df.toPandas()

print(f"Loaded {len(df):,} tours from {df['supplier_id'].nunique():,} suppliers")

# Fix data types - critical for avoiding errors
numeric_cols = ['gmv_l12mo', 'bookings_l12mo', 'has_group_pricing', 'has_individual_pricing',
                'has_addons', 'has_options', 'has_native_scales', 'has_api_scales', 
                'has_live_dynamic_pricing', 'has_non_live_pricing', 'uses_api_pricing',
                'has_scale_pricing', 'is_managed', 'num_individual_categories']

for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)

if 'date_of_first_online' in df.columns:
    df['date_of_first_online'] = pd.to_datetime(df['date_of_first_online'], errors='coerce')

# Ensure supplier_segment and activity_type are strings
if 'supplier_segment' in df.columns:
    df['supplier_segment'] = df['supplier_segment'].fillna('Unknown').astype(str)
if 'activity_type' in df.columns:
    df['activity_type'] = df['activity_type'].fillna('Unknown').astype(str)

print("✓ Data types converted")
print("\nChecking key fields:")
print(f"  - supplier_segment field exists: {'supplier_segment' in df.columns}")
print(f"  - activity_type field exists: {'activity_type' in df.columns}")

if 'supplier_segment' in df.columns:
    print(f"\nSupplier segments in data: {df['supplier_segment'].unique()}")
if 'activity_type' in df.columns:
    print(f"\nActivity types in data (first 10): {df['activity_type'].unique()[:10]}")

display(df.head())

# COMMAND ----------

# MARKDOWN
"""
## Section 2: Segmentation Creation

**What this does:**
- Creates GMV tiers based on percentiles with clear labels: Top 5%, Top 20%, Top 50%, Bottom 50%
- Creates Market Maturity segments based on time on platform (New 0-6mo, Developing 6-18mo, Mature 18mo+)
- Uses the actual supplier_segment field from the data (Enterprise, Mid-Market, SMB, Micro)
- Uses the actual activity_type field from the data
- Identifies API connection status and managed supplier status

**Why it matters:**
Different supplier segments have different adoption patterns. Percentile-based GMV tiers provide clear understanding of revenue concentration. Using actual segment and activity type fields ensures analysis reflects real business classifications.

**Analytical approach:**
- Top 5%: Tours generating top 5% of GMV (high-value catalog requiring personalized support)
- Top 20%: Tours in 5-20% percentile (strong performers)
- Top 50%: Tours in 20-50% percentile (mid-tier performers)
- Bottom 50%: Tours in bottom 50% (long tail, may need scaled enablement)
"""

# COMMAND ----------

# GMV tier - using percentile-based clear labels
gmv_95 = df['gmv_l12mo'].quantile(0.95)
gmv_80 = df['gmv_l12mo'].quantile(0.80)
gmv_50 = df['gmv_l12mo'].quantile(0.50)

df['gmv_tier'] = 'Bottom 50%'
df.loc[df['gmv_l12mo'] >= gmv_50, 'gmv_tier'] = 'Top 50%'
df.loc[df['gmv_l12mo'] >= gmv_80, 'gmv_tier'] = 'Top 20%'
df.loc[df['gmv_l12mo'] >= gmv_95, 'gmv_tier'] = 'Top 5%'

print(f"GMV Tier Thresholds:")
print(f"  - Top 5%: >= ${gmv_95:,.0f}")
print(f"  - Top 20%: >= ${gmv_80:,.0f}")
print(f"  - Top 50%: >= ${gmv_50:,.0f}")
print(f"  - Bottom 50%: < ${gmv_50:,.0f}")

# Market maturity - time-based segmentation
df['days_on_platform'] = (datetime.now() - df['date_of_first_online']).dt.days
df['market_maturity'] = 'Mature (18+ months)'
df.loc[df['days_on_platform'] <= 540, 'market_maturity'] = 'Developing (6-18 months)'
df.loc[df['days_on_platform'] <= 180, 'market_maturity'] = 'New (0-6 months)'

# Connection and management status
df['connection_status'] = df['uses_api_pricing'].apply(lambda x: 'Connected' if x == 1 else 'Not Connected')
df['managed_status'] = df['is_managed'].apply(lambda x: 'Managed' if x == 1 else 'Not Managed')

# Verify segment distribution
print("\n✓ Segments created\n")
print("GMV Tier Distribution:")
display(df['gmv_tier'].value_counts().sort_index())
print("\nMarket Maturity Distribution:")
display(df['market_maturity'].value_counts())
print("\nSupplier Segment Distribution (from data):")
if 'supplier_segment' in df.columns:
    display(df['supplier_segment'].value_counts())
print("\nActivity Type Distribution (from data - top 15):")
if 'activity_type' in df.columns:
    display(df['activity_type'].value_counts().head(15))

# COMMAND ----------

# MARKDOWN
"""
## Section 3: Feature Count Calculation and Adoption Level Assignment

**What this does:**
- Counts how many of the 10 pricing features each tour uses
- Creates 5 adoption levels: Non-adopters (0), Low (1-2), Medium (3-5), High (6-7), Power (8+)

**Why it matters:**
This is the core metric for measuring feature adoption maturity. Tours using more features likely generate more revenue per customer through upselling and dynamic optimization. This metric helps identify tours at risk (non-adopters) and best-in-class (power users).

**Analytical approach:**
Additive feature counting treats all features as equally valuable. The 5-tier classification creates actionable segments for targeted interventions.
"""

# COMMAND ----------

# Build feature list dynamically based on available columns
feature_cols = []
if 'has_individual_pricing' in df.columns:
    feature_cols.append('has_individual_pricing')
if 'has_group_pricing' in df.columns:
    feature_cols.append('has_group_pricing')
if 'uses_api_pricing' in df.columns:
    feature_cols.append('uses_api_pricing')
if 'has_native_scales' in df.columns:
    feature_cols.append('has_native_scales')
if 'has_api_scales' in df.columns:
    feature_cols.append('has_api_scales')
if 'has_live_dynamic_pricing' in df.columns:
    feature_cols.append('has_live_dynamic_pricing')
if 'has_non_live_pricing' in df.columns:
    feature_cols.append('has_non_live_pricing')
if 'has_addons' in df.columns:
    feature_cols.append('has_addons')
if 'has_options' in df.columns:
    feature_cols.append('has_options')

# Calculate native pricing as inverse of API pricing
if 'uses_api_pricing' in df.columns:
    df['has_native_pricing'] = (1 - df['uses_api_pricing']).clip(0, 1)
    feature_cols.append('has_native_pricing')

# Sum features per tour
df['advanced_features_count'] = df[feature_cols].sum(axis=1)

print(f"Counting {len(feature_cols)} features: {feature_cols}")
print(f"Max features used by any tour: {df['advanced_features_count'].max()}")
print(f"Mean features per tour: {df['advanced_features_count'].mean():.2f}")
print(f"Median features per tour: {df['advanced_features_count'].median():.0f}")

# Create adoption levels
df['adoption_level'] = 'Power (8+)'
df.loc[df['advanced_features_count'] <= 7, 'adoption_level'] = 'High (6-7)'
df.loc[df['advanced_features_count'] <= 5, 'adoption_level'] = 'Medium (3-5)'
df.loc[df['advanced_features_count'] <= 2, 'adoption_level'] = 'Low (1-2)'
df.loc[df['advanced_features_count'] == 0, 'adoption_level'] = 'Non-adopters'

df['uses_multiple_categories'] = (df['num_individual_categories'] >= 3).astype(int)

# Display distribution
print("\nAdoption Level Distribution:")
display(df['adoption_level'].value_counts())

# COMMAND ----------

# MARKDOWN
"""
## Section 4: Visualization Setup

**What this does:**
- Sets consistent color palettes and styling for all charts
- Defines segment ordering for consistent visualization
- Configures matplotlib parameters

**Why it matters:**
Consistent visual styling makes the analysis professional and easier to interpret. Ordered categories show progression naturally.
"""

# COMMAND ----------

# Styling
rcParams['font.family'] = 'sans-serif'
rcParams['font.sans-serif'] = ['Arial', 'DejaVu Sans']
rcParams['font.size'] = 10

COLORS = {'primary': '#FF6B35', 'secondary': '#F77F51', 'tertiary': '#FFA07A', 
          'quaternary': '#C0C0C0', 'quinary': '#808080', 'dark': '#404040'}
PALETTE_5 = ['#FF6B35', '#FF8C5A', '#FFAD7F', '#CCCCCC', '#808080']

# Define consistent orderings
gmv_tier_order = ['Top 5%', 'Top 20%', 'Top 50%', 'Bottom 50%']
maturity_order = ['New (0-6 months)', 'Developing (6-18 months)', 'Mature (18+ months)']
adoption_order = ['Non-adopters', 'Low (1-2)', 'Medium (3-5)', 'High (6-7)', 'Power (8+)']

# Get actual segment order from data
if 'supplier_segment' in df.columns:
    segment_actual = df['supplier_segment'].value_counts().index.tolist()
    print(f"Supplier segments from data: {segment_actual}")
else:
    segment_actual = []

print("✓ Styling configured")

# COMMAND ----------

# MARKDOWN
"""
## Chart 1: Overall Feature Adoption Overview

**What this analyzes:**
- Distribution of tours across the 5 adoption levels
- Individual feature adoption rates (which features are most/least used)
- Ticket category complexity distribution
- Overall distribution of feature count

**Business insights to extract:**
- What percentage of our catalog is at risk (non-adopters)?
- Which features have low adoption and need education/enablement?
- Are suppliers using category complexity effectively?

**Interpretation guide:**
- High percentage of non-adopters = opportunity for revenue growth through feature education
- Wide gap between top features and bottom features = some features may need better UX or clearer value prop
"""

# COMMAND ----------

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Pricing Feature Adoption Overview', fontsize=16, fontweight='bold', y=0.995)

# Adoption level distribution
ax = axes[0, 0]
adoption_pcts = (df['adoption_level'].value_counts().reindex(adoption_order) / len(df) * 100)
adoption_pcts = adoption_pcts.fillna(0).values
bars = ax.barh(adoption_order, adoption_pcts, color=PALETTE_5, edgecolor='white', linewidth=2)
ax.set_xlabel('Percentage of Tours', fontweight='bold')
ax.set_title(f'Feature Adoption Levels ({len(feature_cols)} Features)', fontweight='bold', pad=10)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
for i, val in enumerate(adoption_pcts):
    if not np.isnan(val) and np.isfinite(val):
        ax.text(val + 1, i, f'{val:.1f}%', va='center', fontweight='bold', color=COLORS['dark'])

# Individual feature adoption
ax = axes[0, 1]
features = {}
if 'has_group_pricing' in df.columns:
    features['Group Pricing'] = df['has_group_pricing'].sum()
if 'has_individual_pricing' in df.columns:
    features['Individual Pricing'] = df['has_individual_pricing'].sum()
if 'has_addons' in df.columns:
    features['Add-ons'] = df['has_addons'].sum()
if 'has_options' in df.columns:
    features['Options'] = df['has_options'].sum()
if 'uses_api_pricing' in df.columns:
    features['API Pricing'] = df['uses_api_pricing'].sum()
if 'has_native_scales' in df.columns:
    features['Native Scales'] = df['has_native_scales'].sum()
if 'has_api_scales' in df.columns:
    features['API Scales'] = df['has_api_scales'].sum()
if 'has_live_dynamic_pricing' in df.columns:
    features['Live Pricing'] = df['has_live_dynamic_pricing'].sum()

feature_pcts = [(v / len(df)) * 100 for v in features.values()]
colors_bar = ['#FF6B35', '#FF7A47', '#FF8C5A', '#FF9E6C', '#FFAD7F', '#FFBC91', '#FFCBA4', '#D9D9D9']
bars = ax.barh(list(features.keys()), feature_pcts, color=colors_bar[:len(features)], 
               edgecolor='white', linewidth=2)
ax.set_xlabel('Adoption Rate (%)', fontweight='bold')
ax.set_title('Individual Feature Adoption', fontweight='bold', pad=10)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
for i, val in enumerate(feature_pcts):
    if not np.isnan(val) and np.isfinite(val):
        ax.text(val + 1, i, f'{val:.1f}%', va='center', fontweight='bold', color=COLORS['dark'])

# Category complexity
ax = axes[1, 0]
cat_dist = {'Adult Only': (df['num_individual_categories'] == 1).sum(),
            'Adult + Child': (df['num_individual_categories'] == 2).sum(),
            '3 Categories': (df['num_individual_categories'] == 3).sum(),
            '4+ Categories': (df['num_individual_categories'] >= 4).sum()}
cat_pcts = [(v / len(df)) * 100 for v in cat_dist.values()]
bars = ax.bar(range(len(cat_pcts)), cat_pcts, color=PALETTE_5[:4], edgecolor='white', linewidth=2)
ax.set_xticks(range(len(cat_dist)))
ax.set_xticklabels(cat_dist.keys(), rotation=0)
ax.set_ylabel('Percentage of Tours', fontweight='bold')
ax.set_title('Ticket Category Complexity', fontweight='bold', pad=10)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
for i, val in enumerate(cat_pcts):
    if not np.isnan(val) and np.isfinite(val):
        ax.text(i, val + 1, f'{val:.1f}%', ha='center', fontweight='bold', color=COLORS['dark'])

# Feature count distribution
ax = axes[1, 1]
feature_dist = df['advanced_features_count'].value_counts().sort_index()
ax.bar(feature_dist.index, feature_dist.values, color='#FF6B35', edgecolor='white', linewidth=2)
ax.set_xlabel('Number of Features Used', fontweight='bold')
ax.set_ylabel('Number of Tours', fontweight='bold')
ax.set_title('Distribution of Feature Count', fontweight='bold', pad=10)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
display(fig)
plt.close()

# COMMAND ----------

# MARKDOWN
"""
## Chart 2: Adoption by GMV Tier

**What this analyzes:**
- How feature adoption differs between Top 5%, Top 20%, Top 50%, and Bottom 50% GMV tiers
- Average feature count by GMV tier

**Business hypothesis to test:**
Higher GMV tours should have higher adoption rates because:
1. They have more resources to invest in pricing optimization
2. They see clearer ROI from advanced features
3. They may receive more support from GetYourGuide account managers

**Key metrics:**
- Percentage of Power Users in Top 5% vs Bottom 50%
- Adoption gap between tiers (indicates opportunity or resource constraints)
"""

# COMMAND ----------

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Feature Adoption by GMV Tier', fontsize=16, fontweight='bold', y=1.02)

# Adoption distribution by GMV tier
ax = axes[0]
gmv_adoption = pd.crosstab(df['gmv_tier'], df['adoption_level'], normalize='index') * 100
x = np.arange(len(gmv_tier_order))
width = 0.18
for i, adoption in enumerate(adoption_order):
    values = [gmv_adoption.loc[tier, adoption] if tier in gmv_adoption.index and adoption in gmv_adoption.columns else 0 for tier in gmv_tier_order]
    values = [v if np.isfinite(v) else 0 for v in values]
    ax.bar(x + i * width, values, width, label=adoption, color=PALETTE_5[i], edgecolor='white', linewidth=1.5)
ax.set_xlabel('GMV Tier', fontweight='bold')
ax.set_ylabel('Percentage (%)', fontweight='bold')
ax.set_title('Adoption Distribution Within GMV Tier', fontweight='bold', pad=10)
ax.set_xticks(x + width * 2)
ax.set_xticklabels(gmv_tier_order, rotation=0, fontsize=9)
ax.legend(loc='upper left', fontsize=7)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.grid(axis='y', alpha=0.3, linestyle='--', linewidth=0.5)

# Average feature count by GMV tier
ax = axes[1]
avg_features_gmv = df.groupby('gmv_tier')['advanced_features_count'].mean().reindex(gmv_tier_order)
avg_features_gmv = avg_features_gmv.fillna(0)
bars = ax.bar(gmv_tier_order, avg_features_gmv, color=['#FF6B35', '#FF8C5A', '#FFAD7F', '#D9D9D9'], 
              edgecolor='white', linewidth=2)
ax.set_xlabel('GMV Tier', fontweight='bold')
ax.set_ylabel('Average Features Used', fontweight='bold')
ax.set_title('Average Feature Count by GMV Tier', fontweight='bold', pad=10)
ax.set_xticklabels(gmv_tier_order, rotation=0, fontsize=9)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.grid(axis='y', alpha=0.3, linestyle='--', linewidth=0.5)
for i, val in enumerate(avg_features_gmv):
    if not np.isnan(val) and np.isfinite(val):
        ax.text(i, val + 0.1, f'{val:.2f}', ha='center', fontweight='bold', color=COLORS['dark'])

plt.tight_layout()
display(fig)
plt.close()

# COMMAND ----------

# MARKDOWN
"""
## Chart 3: Adoption by Market Maturity

**What this analyzes:**
- How feature adoption evolves as suppliers mature on the platform
- Average feature usage by time on platform

**Business hypothesis to test:**
Mature suppliers should have higher adoption because:
1. They have had more time to learn the platform
2. They have established processes and can invest in optimization
3. They may have received more training over time

**Counterpoint to watch for:**
If NEW suppliers have higher adoption, it may indicate:
- Improved onboarding programs
- Better product education for new suppliers
- Selection bias (only sophisticated suppliers joining recently)
"""

# COMMAND ----------

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Feature Adoption by Market Maturity', fontsize=16, fontweight='bold', y=1.02)

# Adoption distribution by maturity
ax = axes[0]
mat_adoption = pd.crosstab(df['market_maturity'], df['adoption_level'], normalize='index') * 100
x = np.arange(len(maturity_order))
for i, adoption in enumerate(adoption_order):
    values = [mat_adoption.loc[mat, adoption] if mat in mat_adoption.index and adoption in mat_adoption.columns else 0 for mat in maturity_order]
    values = [v if np.isfinite(v) else 0 for v in values]
    ax.bar(x + i * 0.18, values, 0.18, label=adoption, color=PALETTE_5[i], edgecolor='white', linewidth=1.5)
ax.set_xlabel('Market Maturity', fontweight='bold')
ax.set_ylabel('Percentage (%)', fontweight='bold')
ax.set_title('Adoption Distribution by Maturity', fontweight='bold', pad=10)
ax.set_xticks(x + 0.36)
ax.set_xticklabels(['New\n(0-6m)', 'Developing\n(6-18m)', 'Mature\n(18m+)'], rotation=0)
ax.legend(loc='upper left', fontsize=7)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.grid(axis='y', alpha=0.3, linestyle='--', linewidth=0.5)

# Average feature count by maturity
ax = axes[1]
avg_features_mat = df.groupby('market_maturity')['advanced_features_count'].mean().reindex(maturity_order)
avg_features_mat = avg_features_mat.fillna(0)
bars = ax.bar(range(len(maturity_order)), avg_features_mat, color=['#FF6B35', '#FF8C5A', '#FFAD7F'], 
              edgecolor='white', linewidth=2)
ax.set_xlabel('Market Maturity', fontweight='bold')
ax.set_ylabel('Average Features Used', fontweight='bold')
ax.set_title('Average Feature Count by Maturity', fontweight='bold', pad=10)
ax.set_xticks(range(len(maturity_order)))
ax.set_xticklabels(['New\n(0-6m)', 'Developing\n(6-18m)', 'Mature\n(18m+)'], rotation=0)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.grid(axis='y', alpha=0.3, linestyle='--', linewidth=0.5)
for i, val in enumerate(avg_features_mat):
    if not np.isnan(val) and np.isfinite(val):
        ax.text(i, val + 0.1, f'{val:.2f}', ha='center', fontweight='bold', color=COLORS['dark'])

plt.tight_layout()
display(fig)
plt.close()

# COMMAND ----------

# MARKDOWN
"""
## Chart 4: Adoption by Activity Type (FROM DATA FIELD)

**What this analyzes:**
- Which activity types have highest feature adoption
- Average feature usage by activity category
- Uses the actual activity_type field from the data

**Business insights:**
- Some activity types may be more complex and require more pricing features (e.g., multi-day tours)
- Some categories may be under-utilizing features relative to their potential
- Categories sorted by average feature count to identify leaders and laggards

**Action items:**
- Low-adoption categories may need category-specific enablement materials
- High-adoption categories can be case studies for others
"""

# COMMAND ----------

if 'activity_type' in df.columns:
    fig, axes = plt.subplots(2, 1, figsize=(14, 10))
    fig.suptitle('Feature Adoption by Activity Type (from data)', fontsize=16, fontweight='bold', y=0.995)
    
    # Get top 15 activity types by count
    top_activities = df['activity_type'].value_counts().head(15).index.tolist()
    df_act = df[df['activity_type'].isin(top_activities)].copy()
    
    # Adoption distribution by activity
    ax = axes[0]
    act_adoption = pd.crosstab(df_act['activity_type'], df_act['adoption_level'], normalize='index') * 100
    cat_order = df_act.groupby('activity_type')['advanced_features_count'].mean().sort_values(ascending=True).index.tolist()
    act_adoption_sorted = act_adoption.reindex(cat_order)
    x = np.arange(len(cat_order))
    for i, adoption in enumerate(adoption_order):
        values = [act_adoption_sorted.loc[cat, adoption] if adoption in act_adoption_sorted.columns else 0 for cat in cat_order]
        values = [v if np.isfinite(v) else 0 for v in values]
        ax.barh(x + i * 0.18, values, 0.18, label=adoption, color=PALETTE_5[i], edgecolor='white', linewidth=1.5)
    ax.set_ylabel('Activity Type', fontweight='bold')
    ax.set_xlabel('Percentage (%)', fontweight='bold')
    ax.set_title('Adoption Within Each Activity Type (Top 15 by volume)', fontweight='bold', pad=10)
    ax.set_yticks(x + 0.36)
    ax.set_yticklabels(cat_order, fontsize=8)
    ax.legend(loc='lower right', fontsize=7, ncol=2)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.grid(axis='x', alpha=0.3, linestyle='--', linewidth=0.5)
    
    # Average feature count by activity
    ax = axes[1]
    avg_features_act = df_act.groupby('activity_type')['advanced_features_count'].mean().reindex(cat_order)
    avg_features_act = avg_features_act.fillna(0)
    bars = ax.barh(cat_order, avg_features_act, color='#FF6B35', edgecolor='white', linewidth=2)
    ax.set_ylabel('Activity Type', fontweight='bold')
    ax.set_xlabel('Average Features Used', fontweight='bold')
    ax.set_title('Average Feature Count by Activity Type', fontweight='bold', pad=10)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.grid(axis='x', alpha=0.3, linestyle='--', linewidth=0.5)
    
    plt.tight_layout()
    display(fig)
    plt.close()
else:
    print("activity_type field not found in data")

# COMMAND ----------

# MARKDOWN
"""
## Chart 5: Adoption by Supplier Segment (FROM DATA FIELD)

**What this analyzes:**
- Feature adoption using the actual supplier_segment field from the data
- Average feature count by segment

**Business context:**
- Uses the real supplier segmentation from your data
- Enterprise suppliers have dedicated account managers and technical resources
- SMB and Micro suppliers may need self-service education and simpler onboarding
- Segment-specific adoption gaps indicate where to invest in support resources
"""

# COMMAND ----------

if 'supplier_segment' in df.columns and len(segment_actual) > 0:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle('Feature Adoption by Supplier Segment (from data)', fontsize=16, fontweight='bold', y=1.02)
    
    # Adoption distribution by segment
    ax = axes[0]
    seg_adoption = pd.crosstab(df['supplier_segment'], df['adoption_level'], normalize='index') * 100
    x = np.arange(len(segment_actual))
    for i, adoption in enumerate(adoption_order):
        values = [seg_adoption.loc[seg, adoption] if seg in seg_adoption.index and adoption in seg_adoption.columns else 0 for seg in segment_actual]
        values = [v if np.isfinite(v) else 0 for v in values]
        ax.bar(x + i * 0.18, values, 0.18, label=adoption, color=PALETTE_5[i], edgecolor='white', linewidth=1.5)
    ax.set_xlabel('Supplier Segment', fontweight='bold')
    ax.set_ylabel('Percentage (%)', fontweight='bold')
    ax.set_title('Adoption by Segment', fontweight='bold', pad=10)
    ax.set_xticks(x + 0.36)
    ax.set_xticklabels(segment_actual, rotation=45, ha='right')
    ax.legend(loc='upper right', fontsize=7)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.grid(axis='y', alpha=0.3, linestyle='--', linewidth=0.5)
    
    # Average feature count by segment
    ax = axes[1]
    avg_features_seg = df.groupby('supplier_segment')['advanced_features_count'].mean().reindex(segment_actual)
    avg_features_seg = avg_features_seg.fillna(0)
    colors_seg = ['#FF6B35', '#FF8C5A', '#FFAD7F', '#D9D9D9', '#B0B0B0', '#909090']
    bars = ax.bar(segment_actual, avg_features_seg, color=colors_seg[:len(segment_actual)], 
                  edgecolor='white', linewidth=2)
    ax.set_xlabel('Supplier Segment', fontweight='bold')
    ax.set_ylabel('Average Features Used', fontweight='bold')
    ax.set_title('Average Feature Count by Segment', fontweight='bold', pad=10)
    ax.set_xticklabels(segment_actual, rotation=45, ha='right')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.grid(axis='y', alpha=0.3, linestyle='--', linewidth=0.5)
    for i, val in enumerate(avg_features_seg):
        if not np.isnan(val) and np.isfinite(val):
            ax.text(i, val + 0.1, f'{val:.2f}', ha='center', fontweight='bold', color=COLORS['dark'])
    
    plt.tight_layout()
    display(fig)
    plt.close()
else:
    print("supplier_segment field not found in data or no segments available")

# COMMAND ----------

# MARKDOWN
"""
## Chart 6: Adoption by Connection and Management Status

**What this analyzes:**
- API-connected vs non-connected suppliers (API integration enables advanced features)
- Managed vs non-managed suppliers (managed suppliers receive direct account support)

**Hypothesis:**
- API-connected suppliers should have higher adoption (API unlocks scaling features)
- Managed suppliers should have higher adoption (direct support drives feature usage)

**Business value:**
- Quantifies the impact of API connectivity on feature adoption
- Validates the ROI of supplier management programs
"""

# COMMAND ----------

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Adoption by Connection and Management Status', fontsize=16, fontweight='bold', y=0.995)

# Adoption by connection status
ax = axes[0, 0]
conn_adoption = pd.crosstab(df['connection_status'], df['adoption_level'], normalize='index') * 100
conn_order = ['Not Connected', 'Connected']
x = np.arange(len(conn_order))
for i, adoption in enumerate(adoption_order):
    values = [conn_adoption.loc[conn, adoption] if conn in conn_adoption.index and adoption in conn_adoption.columns else 0 for conn in conn_order]
    values = [v if np.isfinite(v) else 0 for v in values]
    ax.bar(x + i * 0.18, values, 0.18, label=adoption, color=PALETTE_5[i], edgecolor='white', linewidth=1.5)
ax.set_xlabel('Connection Status', fontweight='bold')
ax.set_ylabel('Percentage (%)', fontweight='bold')
ax.set_title('Adoption by Connection', fontweight='bold', pad=10)
ax.set_xticks(x + 0.36)
ax.set_xticklabels(conn_order, rotation=0)
ax.legend(loc='upper left', fontsize=6)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.grid(axis='y', alpha=0.3, linestyle='--', linewidth=0.5)

# Average features by connection
ax = axes[0, 1]
avg_feat_conn = df.groupby('connection_status')['advanced_features_count'].mean().reindex(conn_order)
avg_feat_conn = avg_feat_conn.fillna(0)
bars = ax.bar(conn_order, avg_feat_conn, color=['#FF6B35', '#FFAD7F'], edgecolor='white', linewidth=2)
ax.set_xlabel('Connection Status', fontweight='bold')
ax.set_ylabel('Average Features Used', fontweight='bold')
ax.set_title('Features by Connection', fontweight='bold', pad=10)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.grid(axis='y', alpha=0.3, linestyle='--', linewidth=0.5)
for i, val in enumerate(avg_feat_conn):
    if not np.isnan(val) and np.isfinite(val):
        ax.text(i, val + 0.1, f'{val:.2f}', ha='center', fontweight='bold', color=COLORS['dark'])

# Adoption by managed status
ax = axes[1, 0]
mgd_adoption = pd.crosstab(df['managed_status'], df['adoption_level'], normalize='index') * 100
mgd_order = ['Not Managed', 'Managed']
x = np.arange(len(mgd_order))
for i, adoption in enumerate(adoption_order):
    values = [mgd_adoption.loc[mgd, adoption] if mgd in mgd_adoption.index and adoption in mgd_adoption.columns else 0 for mgd in mgd_order]
    values = [v if np.isfinite(v) else 0 for v in values]
    ax.bar(x + i * 0.18, values, 0.18, label=adoption, color=PALETTE_5[i], edgecolor='white', linewidth=1.5)
ax.set_xlabel('Management Status', fontweight='bold')
ax.set_ylabel('Percentage (%)', fontweight='bold')
ax.set_title('Adoption by Managed Status', fontweight='bold', pad=10)
ax.set_xticks(x + 0.36)
ax.set_xticklabels(mgd_order, rotation=0)
ax.legend(loc='upper left', fontsize=6)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.grid(axis='y', alpha=0.3, linestyle='--', linewidth=0.5)

# Average features by managed status
ax = axes[1, 1]
avg_feat_mgd = df.groupby('managed_status')['advanced_features_count'].mean().reindex(mgd_order)
avg_feat_mgd = avg_feat_mgd.fillna(0)
bars = ax.bar(mgd_order, avg_feat_mgd, color=['#FF6B35', '#FFAD7F'], edgecolor='white', linewidth=2)
ax.set_xlabel('Management Status', fontweight='bold')
ax.set_ylabel('Average Features Used', fontweight='bold')
ax.set_title('Features by Managed Status', fontweight='bold', pad=10)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.grid(axis='y', alpha=0.3, linestyle='--', linewidth=0.5)
for i, val in enumerate(avg_feat_mgd):
    if not np.isnan(val) and np.isfinite(val):
        ax.text(i, val + 0.1, f'{val:.2f}', ha='center', fontweight='bold', color=COLORS['dark'])

plt.tight_layout()
display(fig)
plt.close()

# COMMAND ----------

# MARKDOWN
"""
## Chart 7: Cross-Segment Analysis - GMV Tier × Market Maturity

**What this analyzes:**
- Power user adoption rates across combinations of GMV tier and maturity
- Average feature usage across combinations

**Why cross-segment matters:**
Single-dimension analysis can hide important patterns. For example:
- Are NEW high-GMV tours adopting faster than MATURE low-GMV tours?
- Do high-GMV tours start strong or grow into features over time?

**Heatmap interpretation:**
- Hot spots (orange/red) = high adoption segments to use as case studies
- Cold spots (light) = opportunity segments for targeted enablement
"""

# COMMAND ----------

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Cross-Segment: GMV Tier × Market Maturity', fontsize=16, fontweight='bold', y=1.02)

# Power user adoption heatmap
ax = axes[0]
power_cross = pd.crosstab(df['gmv_tier'], df['market_maturity'], 
                          values=(df['adoption_level'] == 'Power (8+)').astype(int), aggfunc='mean') * 100
power_cross = power_cross.reindex(index=gmv_tier_order, columns=maturity_order).fillna(0)
im = ax.imshow(power_cross.values, cmap='Oranges', aspect='auto', vmin=0, vmax=max(power_cross.values.max(), 0.1))
ax.set_xticks(range(len(maturity_order)))
ax.set_xticklabels(['New\n(0-6m)', 'Developing\n(6-18m)', 'Mature\n(18m+)'], rotation=0)
ax.set_yticks(range(len(gmv_tier_order)))
ax.set_yticklabels(gmv_tier_order)
ax.set_xlabel('Market Maturity', fontweight='bold')
ax.set_ylabel('GMV Tier', fontweight='bold')
ax.set_title('Power User Adoption Rate (%)', fontweight='bold', pad=10)
for i in range(len(gmv_tier_order)):
    for j in range(len(maturity_order)):
        val = power_cross.values[i, j]
        if np.isfinite(val):
            ax.text(j, i, f'{val:.1f}%', ha='center', va='center', 
                   color='white' if val > power_cross.values.max()/2 else 'black', fontweight='bold')
cbar = plt.colorbar(im, ax=ax)
cbar.set_label('Adoption Rate (%)', rotation=270, labelpad=20, fontweight='bold')

# Average features heatmap
ax = axes[1]
avg_feat_cross = pd.crosstab(df['gmv_tier'], df['market_maturity'], 
                             values=df['advanced_features_count'], aggfunc='mean')
avg_feat_cross = avg_feat_cross.reindex(index=gmv_tier_order, columns=maturity_order).fillna(0)
im = ax.imshow(avg_feat_cross.values, cmap='Oranges', aspect='auto', vmin=0, vmax=max(avg_feat_cross.values.max(), 0.1))
ax.set_xticks(range(len(maturity_order)))
ax.set_xticklabels(['New\n(0-6m)', 'Developing\n(6-18m)', 'Mature\n(18m+)'], rotation=0)
ax.set_yticks(range(len(gmv_tier_order)))
ax.set_yticklabels(gmv_tier_order)
ax.set_xlabel('Market Maturity', fontweight='bold')
ax.set_ylabel('GMV Tier', fontweight='bold')
ax.set_title('Average Features Used', fontweight='bold', pad=10)
for i in range(len(gmv_tier_order)):
    for j in range(len(maturity_order)):
        val = avg_feat_cross.values[i, j]
        if np.isfinite(val):
            ax.text(j, i, f'{val:.2f}', ha='center', va='center',
                   color='white' if val > avg_feat_cross.values.max()/2 else 'black', fontweight='bold')
cbar = plt.colorbar(im, ax=ax)
cbar.set_label('Avg Features', rotation=270, labelpad=20, fontweight='bold')

plt.tight_layout()
display(fig)
plt.close()

# COMMAND ----------

# MARKDOWN
"""
## Summary Statistics and Key Takeaways

**What this provides:**
- Overall adoption rate (percentage using at least one feature)
- Power user rate (percentage using 8+ features)
- Average features per tour
- Distribution summary
- GMV tier thresholds used

**Use these metrics to:**
- Track adoption improvement over time (run this monthly)
- Set targets for product and enablement teams
- Communicate impact to leadership
"""

# COMMAND ----------

# Calculate summary metrics
adopters = (df['adoption_level'] != 'Non-adopters').sum()
adoption_rate = (adopters / len(df)) * 100
power_users = (df['adoption_level'] == 'Power (8+)').sum()
power_rate = (power_users / len(df)) * 100
avg_feat = df['advanced_features_count'].mean()
median_feat = df['advanced_features_count'].median()

print("="*60)
print("PRICING FEATURE ADOPTION SUMMARY")
print("="*60)
print(f"\nTotal tours analyzed: {len(df):,}")
print(f"Total suppliers: {df['supplier_id'].nunique():,}")
print(f"Features tracked: {len(feature_cols)}")
print(f"\nGMV TIER THRESHOLDS:")
print(f"  • Top 5%: >= ${gmv_95:,.0f}")
print(f"  • Top 20%: >= ${gmv_80:,.0f}")
print(f"  • Top 50%: >= ${gmv_50:,.0f}")
print(f"  • Bottom 50%: < ${gmv_50:,.0f}")
print(f"\nOVERALL ADOPTION METRICS:")
print(f"  • Overall adoption rate: {adoption_rate:.1f}%")
print(f"  • Non-adopters: {100-adoption_rate:.1f}%")
print(f"  • Power users (8+ features): {power_rate:.1f}%")
print(f"  • Average features per tour: {avg_feat:.2f}")
print(f"  • Median features per tour: {median_feat:.0f}")
print(f"\nFEATURES ANALYZED:")
for i, feat in enumerate(feature_cols, 1):
    print(f"  {i}. {feat}")
print("="*60)

# Show adoption breakdown
print("\nADOPTION LEVEL BREAKDOWN:")
adoption_breakdown = df['adoption_level'].value_counts().reindex(adoption_order)
for level in adoption_order:
    count = adoption_breakdown.get(level, 0)
    pct = (count / len(df)) * 100
    print(f"  {level:20s}: {count:6,} tours ({pct:5.1f}%)")

print("\n" + "="*60)


In [0]:
# Databricks notebook source

# MARKDOWN
"""
# Pricing Feature Adoption Audit

## Objective
This notebook analyzes pricing feature adoption across GetYourGuide's supplier base to identify which segments are utilizing advanced pricing capabilities and which segments represent opportunities for feature education and adoption.

## Business Questions
1. What percentage of tours use advanced pricing features?
2. Which supplier segments (by volume, maturity, category) have highest/lowest adoption?
3. Are there specific feature combinations that drive better outcomes?
4. What opportunities exist to increase feature penetration?

## Features Analyzed (10 Total)
- Individual Pricing
- Group Pricing  
- Pricing without API (Native)
- Pricing with API
- Scales without API (Native Scales)
- Scales with API
- Live Pricing and Availability
- Non-live Price and Availability
- Add-ons
- Options
"""

# COMMAND ----------

# MARKDOWN
"""
## Section 1: Data Loading and Type Conversion

**What this does:** 
- Loads the pricing feature data from the Spark table
- Converts all columns to proper numeric types to avoid calculation errors
- Handles missing values by filling with 0
- Converts dates to datetime format

**Why it matters:**
Data type issues cause errors in aggregations and binning. This ensures clean numeric data for all calculations.
"""

# COMMAND ----------

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import rcParams
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Load data
print("Loading data from Spark...")
spark_df = spark.table("production.supply_analytics.pricing_feature_combined")
df = spark_df.toPandas()

print(f"Loaded {len(df):,} tours from {df['supplier_id'].nunique():,} suppliers")

# Fix data types - critical for avoiding errors
numeric_cols = ['gmv_l12mo', 'bookings_l12mo', 'has_group_pricing', 'has_individual_pricing',
                'has_addons', 'has_options', 'has_native_scales', 'has_api_scales', 
                'has_live_dynamic_pricing', 'has_non_live_pricing', 'uses_api_pricing',
                'has_scale_pricing', 'is_managed', 'num_individual_categories']

for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)

if 'date_of_first_online' in df.columns:
    df['date_of_first_online'] = pd.to_datetime(df['date_of_first_online'], errors='coerce')

# Ensure supplier_segment and activity_type are strings
if 'supplier_segment' in df.columns:
    df['supplier_segment'] = df['supplier_segment'].fillna('Unknown').astype(str)
if 'activity_type' in df.columns:
    df['activity_type'] = df['activity_type'].fillna('Unknown').astype(str)

print("✓ Data types converted")
print("\nChecking key fields:")
print(f"  - supplier_segment field exists: {'supplier_segment' in df.columns}")
print(f"  - activity_type field exists: {'activity_type' in df.columns}")

if 'supplier_segment' in df.columns:
    print(f"\nSupplier segments in data: {df['supplier_segment'].unique()}")
if 'activity_type' in df.columns:
    print(f"\nActivity types in data (first 10): {df['activity_type'].unique()[:10]}")

display(df.head())

# COMMAND ----------

# MARKDOWN
"""
## Section 2: Segmentation Creation

**What this does:**
- Creates GMV tiers using quartile-based segmentation directly from the data
- Creates Market Maturity segments based on time on platform (New 0-6mo, Developing 6-18mo, Mature 18mo+)
- Uses the actual supplier_segment field from the data (Enterprise, Mid-Market, SMB, Micro)
- Uses the actual activity_type field from the data
- Identifies API connection status and managed supplier status

**Why it matters:**
Different supplier segments have different adoption patterns. Quantile-based GMV tiers ensure equal cohort sizes for statistical comparison. Using actual segment and activity type fields ensures analysis reflects real business classifications.

**Analytical approach:**
GMV tiers are created using quartiles from the actual data distribution. Labels show quartile positions (Q1, Q2, Q3, Q4) with actual GMV ranges.
"""

# COMMAND ----------

# GMV tier - using quartiles directly from data
gmv_quantiles = df['gmv_l12mo'].quantile([0, 0.25, 0.5, 0.75, 1.0])
print("GMV Distribution Quantiles:")
print(gmv_quantiles)

# Create quartile-based tiers WITHOUT labels first to handle duplicates
df['gmv_tier'] = pd.qcut(df['gmv_l12mo'], q=4, duplicates='drop')

# Now map to readable labels based on actual bins created
unique_bins = df['gmv_tier'].unique()
n_bins = len(unique_bins)

if n_bins == 4:
    label_map = {
        sorted(unique_bins)[0]: 'Q1 (Lowest 25%)',
        sorted(unique_bins)[1]: 'Q2',
        sorted(unique_bins)[2]: 'Q3',
        sorted(unique_bins)[3]: 'Q4 (Top 25%)'
    }
    gmv_tier_order = ['Q1 (Lowest 25%)', 'Q2', 'Q3', 'Q4 (Top 25%)']
elif n_bins == 3:
    label_map = {
        sorted(unique_bins)[0]: 'Q1 (Lowest)',
        sorted(unique_bins)[1]: 'Q2-Q3 (Middle)',
        sorted(unique_bins)[2]: 'Q4 (Top)'
    }
    gmv_tier_order = ['Q1 (Lowest)', 'Q2-Q3 (Middle)', 'Q4 (Top)']
elif n_bins == 2:
    label_map = {
        sorted(unique_bins)[0]: 'Lower 50%',
        sorted(unique_bins)[1]: 'Upper 50%'
    }
    gmv_tier_order = ['Lower 50%', 'Upper 50%']
else:
    label_map = {bin_val: f'Tier {i+1}' for i, bin_val in enumerate(sorted(unique_bins))}
    gmv_tier_order = [f'Tier {i+1}' for i in range(n_bins)]

df['gmv_tier'] = df['gmv_tier'].map(label_map)

print(f"\nGMV Tiers created from data ({n_bins} tiers):")
for tier in gmv_tier_order:
    count = (df['gmv_tier'] == tier).sum()
    pct = count / len(df) * 100
    print(f"  {tier}: {count:,} tours ({pct:.1f}%)")

# Market maturity - time-based segmentation
df['days_on_platform'] = (datetime.now() - df['date_of_first_online']).dt.days
df['market_maturity'] = 'Mature (18+ months)'
df.loc[df['days_on_platform'] <= 540, 'market_maturity'] = 'Developing (6-18 months)'
df.loc[df['days_on_platform'] <= 180, 'market_maturity'] = 'New (0-6 months)'

# Connection and management status
df['connection_status'] = df['uses_api_pricing'].apply(lambda x: 'Connected' if x == 1 else 'Not Connected')
df['managed_status'] = df['is_managed'].apply(lambda x: 'Managed' if x == 1 else 'Not Managed')

# Verify segment distribution
print("\n✓ Segments created\n")
print("Market Maturity Distribution:")
display(df['market_maturity'].value_counts())
print("\nSupplier Segment Distribution (from data):")
if 'supplier_segment' in df.columns:
    display(df['supplier_segment'].value_counts())
print("\nActivity Type Distribution (from data - top 15):")
if 'activity_type' in df.columns:
    display(df['activity_type'].value_counts().head(15))

# COMMAND ----------

# MARKDOWN
"""
## Section 3: Feature Count Calculation and Adoption Level Assignment

**What this does:**
- Counts how many of the 10 pricing features each tour uses
- Creates 5 adoption levels: Non-adopters (0), Low (1-2), Medium (3-5), High (6-7), Power (8+)

**Why it matters:**
This is the core metric for measuring feature adoption maturity. Tours using more features likely generate more revenue per customer through upselling and dynamic optimization. This metric helps identify tours at risk (non-adopters) and best-in-class (power users).

**Analytical approach:**
Additive feature counting treats all features as equally valuable. The 5-tier classification creates actionable segments for targeted interventions.
"""

# COMMAND ----------

# Build feature list dynamically based on available columns
feature_cols = []
if 'has_individual_pricing' in df.columns:
    feature_cols.append('has_individual_pricing')
if 'has_group_pricing' in df.columns:
    feature_cols.append('has_group_pricing')
if 'uses_api_pricing' in df.columns:
    feature_cols.append('uses_api_pricing')
if 'has_native_scales' in df.columns:
    feature_cols.append('has_native_scales')
if 'has_api_scales' in df.columns:
    feature_cols.append('has_api_scales')
if 'has_live_dynamic_pricing' in df.columns:
    feature_cols.append('has_live_dynamic_pricing')
if 'has_non_live_pricing' in df.columns:
    feature_cols.append('has_non_live_pricing')
if 'has_addons' in df.columns:
    feature_cols.append('has_addons')
if 'has_options' in df.columns:
    feature_cols.append('has_options')

# Calculate native pricing as inverse of API pricing
if 'uses_api_pricing' in df.columns:
    df['has_native_pricing'] = (1 - df['uses_api_pricing']).clip(0, 1)
    feature_cols.append('has_native_pricing')

# Sum features per tour
df['advanced_features_count'] = df[feature_cols].sum(axis=1)

print(f"Counting {len(feature_cols)} features: {feature_cols}")
print(f"Max features used by any tour: {df['advanced_features_count'].max()}")
print(f"Mean features per tour: {df['advanced_features_count'].mean():.2f}")
print(f"Median features per tour: {df['advanced_features_count'].median():.0f}")

# Create adoption levels
df['adoption_level'] = 'Power (8+)'
df.loc[df['advanced_features_count'] <= 7, 'adoption_level'] = 'High (6-7)'
df.loc[df['advanced_features_count'] <= 5, 'adoption_level'] = 'Medium (3-5)'
df.loc[df['advanced_features_count'] <= 2, 'adoption_level'] = 'Low (1-2)'
df.loc[df['advanced_features_count'] == 0, 'adoption_level'] = 'Non-adopters'

df['uses_multiple_categories'] = (df['num_individual_categories'] >= 3).astype(int)

# Display distribution
print("\nAdoption Level Distribution:")
display(df['adoption_level'].value_counts())

# COMMAND ----------

# MARKDOWN
"""
## Section 4: Visualization Setup

**What this does:**
- Sets consistent color palettes and styling for all charts
- Defines segment ordering for consistent visualization
- Configures matplotlib parameters

**Why it matters:**
Consistent visual styling makes the analysis professional and easier to interpret. Ordered categories show progression naturally.
"""

# COMMAND ----------

# Styling
rcParams['font.family'] = 'sans-serif'
rcParams['font.sans-serif'] = ['Arial', 'DejaVu Sans']
rcParams['font.size'] = 10

COLORS = {'primary': '#FF6B35', 'secondary': '#F77F51', 'tertiary': '#FFA07A', 
          'quaternary': '#C0C0C0', 'quinary': '#808080', 'dark': '#404040'}
PALETTE_5 = ['#FF6B35', '#FF8C5A', '#FFAD7F', '#CCCCCC', '#808080']

# Define consistent orderings
maturity_order = ['New (0-6 months)', 'Developing (6-18 months)', 'Mature (18+ months)']
adoption_order = ['Non-adopters', 'Low (1-2)', 'Medium (3-5)', 'High (6-7)', 'Power (8+)']

# Get actual segment order from data
if 'supplier_segment' in df.columns:
    segment_actual = df['supplier_segment'].value_counts().index.tolist()
    print(f"Supplier segments from data: {segment_actual}")
else:
    segment_actual = []

print("✓ Styling configured")

# COMMAND ----------

# MARKDOWN
"""
## Chart 1: Overall Feature Adoption Overview

**What this analyzes:**
- Distribution of tours across the 5 adoption levels
- Individual feature adoption rates (which features are most/least used)
- Ticket category complexity distribution
- Overall distribution of feature count

**Business insights to extract:**
- What percentage of our catalog is at risk (non-adopters)?
- Which features have low adoption and need education/enablement?
- Are suppliers using category complexity effectively?

**Interpretation guide:**
- High percentage of non-adopters = opportunity for revenue growth through feature education
- Wide gap between top features and bottom features = some features may need better UX or clearer value prop
"""

# COMMAND ----------

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Pricing Feature Adoption Overview', fontsize=16, fontweight='bold', y=0.995)

# Adoption level distribution
ax = axes[0, 0]
adoption_pcts = (df['adoption_level'].value_counts().reindex(adoption_order) / len(df) * 100)
adoption_pcts = adoption_pcts.fillna(0).values
bars = ax.barh(adoption_order, adoption_pcts, color=PALETTE_5, edgecolor='white', linewidth=2)
ax.set_xlabel('Percentage of Tours', fontweight='bold')
ax.set_title(f'Feature Adoption Levels ({len(feature_cols)} Features)', fontweight='bold', pad=10)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
for i, val in enumerate(adoption_pcts):
    if not np.isnan(val) and np.isfinite(val):
        ax.text(val + 1, i, f'{val:.1f}%', va='center', fontweight='bold', color=COLORS['dark'])

# Individual feature adoption
ax = axes[0, 1]
features = {}
if 'has_group_pricing' in df.columns:
    features['Group Pricing'] = df['has_group_pricing'].sum()
if 'has_individual_pricing' in df.columns:
    features['Individual Pricing'] = df['has_individual_pricing'].sum()
if 'has_addons' in df.columns:
    features['Add-ons'] = df['has_addons'].sum()
if 'has_options' in df.columns:
    features['Options'] = df['has_options'].sum()
if 'uses_api_pricing' in df.columns:
    features['API Pricing'] = df['uses_api_pricing'].sum()
if 'has_native_scales' in df.columns:
    features['Native Scales'] = df['has_native_scales'].sum()
if 'has_api_scales' in df.columns:
    features['API Scales'] = df['has_api_scales'].sum()
if 'has_live_dynamic_pricing' in df.columns:
    features['Live Pricing'] = df['has_live_dynamic_pricing'].sum()

feature_pcts = [(v / len(df)) * 100 for v in features.values()]
colors_bar = ['#FF6B35', '#FF7A47', '#FF8C5A', '#FF9E6C', '#FFAD7F', '#FFBC91', '#FFCBA4', '#D9D9D9']
bars = ax.barh(list(features.keys()), feature_pcts, color=colors_bar[:len(features)], 
               edgecolor='white', linewidth=2)
ax.set_xlabel('Adoption Rate (%)', fontweight='bold')
ax.set_title('Individual Feature Adoption', fontweight='bold', pad=10)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
for i, val in enumerate(feature_pcts):
    if not np.isnan(val) and np.isfinite(val):
        ax.text(val + 1, i, f'{val:.1f}%', va='center', fontweight='bold', color=COLORS['dark'])

# Category complexity
ax = axes[1, 0]
cat_dist = {'Adult Only': (df['num_individual_categories'] == 1).sum(),
            'Adult + Child': (df['num_individual_categories'] == 2).sum(),
            '3 Categories': (df['num_individual_categories'] == 3).sum(),
            '4+ Categories': (df['num_individual_categories'] >= 4).sum()}
cat_pcts = [(v / len(df)) * 100 for v in cat_dist.values()]
bars = ax.bar(range(len(cat_pcts)), cat_pcts, color=PALETTE_5[:4], edgecolor='white', linewidth=2)
ax.set_xticks(range(len(cat_dist)))
ax.set_xticklabels(cat_dist.keys(), rotation=0)
ax.set_ylabel('Percentage of Tours', fontweight='bold')
ax.set_title('Ticket Category Complexity', fontweight='bold', pad=10)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
for i, val in enumerate(cat_pcts):
    if not np.isnan(val) and np.isfinite(val):
        ax.text(i, val + 1, f'{val:.1f}%', ha='center', fontweight='bold', color=COLORS['dark'])

# Feature count distribution
ax = axes[1, 1]
feature_dist = df['advanced_features_count'].value_counts().sort_index()
ax.bar(feature_dist.index, feature_dist.values, color='#FF6B35', edgecolor='white', linewidth=2)
ax.set_xlabel('Number of Features Used', fontweight='bold')
ax.set_ylabel('Number of Tours', fontweight='bold')
ax.set_title('Distribution of Feature Count', fontweight='bold', pad=10)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
display(fig)
plt.close()

# COMMAND ----------

# MARKDOWN
"""
## Chart 2: Adoption by GMV Tier (Data-Driven Quartiles)

**What this analyzes:**
- How feature adoption differs across GMV quartiles derived from actual data distribution
- Average feature count by GMV tier
- Uses quartile-based segmentation: Q1 (bottom 25%), Q2, Q3, Q4 (top 25%)

**Business hypothesis to test:**
Higher GMV tours should have higher adoption rates because:
1. They have more resources to invest in pricing optimization
2. They see clearer ROI from advanced features
3. They may receive more support from GetYourGuide account managers

**Key metrics:**
- Percentage of Power Users in Q4 vs Q1
- Adoption gap between tiers (indicates opportunity or resource constraints)
"""

# COMMAND ----------

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Feature Adoption by GMV Tier (Data-Driven Quartiles)', fontsize=16, fontweight='bold', y=1.02)

# Adoption distribution by GMV tier
ax = axes[0]
gmv_adoption = pd.crosstab(df['gmv_tier'], df['adoption_level'], normalize='index') * 100
x = np.arange(len(gmv_tier_order))
width = 0.18
for i, adoption in enumerate(adoption_order):
    values = [gmv_adoption.loc[tier, adoption] if tier in gmv_adoption.index and adoption in gmv_adoption.columns else 0 for tier in gmv_tier_order]
    values = [v if np.isfinite(v) else 0 for v in values]
    ax.bar(x + i * width, values, width, label=adoption, color=PALETTE_5[i], edgecolor='white', linewidth=1.5)
ax.set_xlabel('GMV Tier', fontweight='bold')
ax.set_ylabel('Percentage (%)', fontweight='bold')
ax.set_title('Adoption Distribution Within GMV Tier', fontweight='bold', pad=10)
ax.set_xticks(x + width * 2)
ax.set_xticklabels(gmv_tier_order, rotation=45, ha='right', fontsize=9)
ax.legend(loc='upper left', fontsize=7)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.grid(axis='y', alpha=0.3, linestyle='--', linewidth=0.5)

# Average feature count by GMV tier
ax = axes[1]
avg_features_gmv = df.groupby('gmv_tier')['advanced_features_count'].mean().reindex(gmv_tier_order)
avg_features_gmv = avg_features_gmv.fillna(0)
bars = ax.bar(range(len(gmv_tier_order)), avg_features_gmv, color=['#FF6B35', '#FF8C5A', '#FFAD7F', '#D9D9D9'][:len(gmv_tier_order)], 
              edgecolor='white', linewidth=2)
ax.set_xlabel('GMV Tier', fontweight='bold')
ax.set_ylabel('Average Features Used', fontweight='bold')
ax.set_title('Average Feature Count by GMV Tier', fontweight='bold', pad=10)
ax.set_xticks(range(len(gmv_tier_order)))
ax.set_xticklabels(gmv_tier_order, rotation=45, ha='right', fontsize=9)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.grid(axis='y', alpha=0.3, linestyle='--', linewidth=0.5)
for i, val in enumerate(avg_features_gmv):
    if not np.isnan(val) and np.isfinite(val):
        ax.text(i, val + 0.1, f'{val:.2f}', ha='center', fontweight='bold', color=COLORS['dark'])

plt.tight_layout()
display(fig)
plt.close()

# COMMAND ----------

# MARKDOWN
"""
## Chart 3: Adoption by Market Maturity

**What this analyzes:**
- How feature adoption evolves as suppliers mature on the platform
- Average feature usage by time on platform

**Business hypothesis to test:**
Mature suppliers should have higher adoption because:
1. They have had more time to learn the platform
2. They have established processes and can invest in optimization
3. They may have received more training over time

**Counterpoint to watch for:**
If NEW suppliers have higher adoption, it may indicate:
- Improved onboarding programs
- Better product education for new suppliers
- Selection bias (only sophisticated suppliers joining recently)
"""

# COMMAND ----------

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Feature Adoption by Market Maturity', fontsize=16, fontweight='bold', y=1.02)

# Adoption distribution by maturity
ax = axes[0]
mat_adoption = pd.crosstab(df['market_maturity'], df['adoption_level'], normalize='index') * 100
x = np.arange(len(maturity_order))
for i, adoption in enumerate(adoption_order):
    values = [mat_adoption.loc[mat, adoption] if mat in mat_adoption.index and adoption in mat_adoption.columns else 0 for mat in maturity_order]
    values = [v if np.isfinite(v) else 0 for v in values]
    ax.bar(x + i * 0.18, values, 0.18, label=adoption, color=PALETTE_5[i], edgecolor='white', linewidth=1.5)
ax.set_xlabel('Market Maturity', fontweight='bold')
ax.set_ylabel('Percentage (%)', fontweight='bold')
ax.set_title('Adoption Distribution by Maturity', fontweight='bold', pad=10)
ax.set_xticks(x + 0.36)
ax.set_xticklabels(['New\n(0-6m)', 'Developing\n(6-18m)', 'Mature\n(18m+)'], rotation=0)
ax.legend(loc='upper left', fontsize=7)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.grid(axis='y', alpha=0.3, linestyle='--', linewidth=0.5)

# Average feature count by maturity
ax = axes[1]
avg_features_mat = df.groupby('market_maturity')['advanced_features_count'].mean().reindex(maturity_order)
avg_features_mat = avg_features_mat.fillna(0)
bars = ax.bar(range(len(maturity_order)), avg_features_mat, color=['#FF6B35', '#FF8C5A', '#FFAD7F'], 
              edgecolor='white', linewidth=2)
ax.set_xlabel('Market Maturity', fontweight='bold')
ax.set_ylabel('Average Features Used', fontweight='bold')
ax.set_title('Average Feature Count by Maturity', fontweight='bold', pad=10)
ax.set_xticks(range(len(maturity_order)))
ax.set_xticklabels(['New\n(0-6m)', 'Developing\n(6-18m)', 'Mature\n(18m+)'], rotation=0)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.grid(axis='y', alpha=0.3, linestyle='--', linewidth=0.5)
for i, val in enumerate(avg_features_mat):
    if not np.isnan(val) and np.isfinite(val):
        ax.text(i, val + 0.1, f'{val:.2f}', ha='center', fontweight='bold', color=COLORS['dark'])

plt.tight_layout()
display(fig)
plt.close()

# COMMAND ----------

# MARKDOWN
"""
## Chart 4: Adoption by Activity Type (FROM DATA FIELD)

**What this analyzes:**
- Which activity types have highest feature adoption
- Average feature usage by activity category
- Uses the actual activity_type field from the data

**Business insights:**
- Some activity types may be more complex and require more pricing features (e.g., multi-day tours)
- Some categories may be under-utilizing features relative to their potential
- Categories sorted by average feature count to identify leaders and laggards

**Action items:**
- Low-adoption categories may need category-specific enablement materials
- High-adoption categories can be case studies for others
"""

# COMMAND ----------

if 'activity_type' in df.columns:
    fig, axes = plt.subplots(2, 1, figsize=(14, 10))
    fig.suptitle('Feature Adoption by Activity Type (from data)', fontsize=16, fontweight='bold', y=0.995)
    
    # Get top 15 activity types by count
    top_activities = df['activity_type'].value_counts().head(15).index.tolist()
    df_act = df[df['activity_type'].isin(top_activities)].copy()
    
    # Adoption distribution by activity
    ax = axes[0]
    act_adoption = pd.crosstab(df_act['activity_type'], df_act['adoption_level'], normalize='index') * 100
    cat_order = df_act.groupby('activity_type')['advanced_features_count'].mean().sort_values(ascending=True).index.tolist()
    act_adoption_sorted = act_adoption.reindex(cat_order)
    x = np.arange(len(cat_order))
    for i, adoption in enumerate(adoption_order):
        values = [act_adoption_sorted.loc[cat, adoption] if adoption in act_adoption_sorted.columns else 0 for cat in cat_order]
        values = [v if np.isfinite(v) else 0 for v in values]
        ax.barh(x + i * 0.18, values, 0.18, label=adoption, color=PALETTE_5[i], edgecolor='white', linewidth=1.5)
    ax.set_ylabel('Activity Type', fontweight='bold')
    ax.set_xlabel('Percentage (%)', fontweight='bold')
    ax.set_title('Adoption Within Each Activity Type (Top 15 by volume)', fontweight='bold', pad=10)
    ax.set_yticks(x + 0.36)
    ax.set_yticklabels(cat_order, fontsize=8)
    ax.legend(loc='lower right', fontsize=7, ncol=2)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.grid(axis='x', alpha=0.3, linestyle='--', linewidth=0.5)
    
    # Average feature count by activity
    ax = axes[1]
    avg_features_act = df_act.groupby('activity_type')['advanced_features_count'].mean().reindex(cat_order)
    avg_features_act = avg_features_act.fillna(0)
    bars = ax.barh(cat_order, avg_features_act, color='#FF6B35', edgecolor='white', linewidth=2)
    ax.set_ylabel('Activity Type', fontweight='bold')
    ax.set_xlabel('Average Features Used', fontweight='bold')
    ax.set_title('Average Feature Count by Activity Type', fontweight='bold', pad=10)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.grid(axis='x', alpha=0.3, linestyle='--', linewidth=0.5)
    
    plt.tight_layout()
    display(fig)
    plt.close()
else:
    print("activity_type field not found in data")

# COMMAND ----------

# MARKDOWN
"""
## Chart 5: Adoption by Supplier Segment (FROM DATA FIELD)

**What this analyzes:**
- Feature adoption using the actual supplier_segment field from the data
- Average feature count by segment

**Business context:**
- Uses the real supplier segmentation from your data
- Enterprise suppliers have dedicated account managers and technical resources
- SMB and Micro suppliers may need self-service education and simpler onboarding
- Segment-specific adoption gaps indicate where to invest in support resources
"""

# COMMAND ----------

if 'supplier_segment' in df.columns and len(segment_actual) > 0:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle('Feature Adoption by Supplier Segment (from data)', fontsize=16, fontweight='bold', y=1.02)
    
    # Adoption distribution by segment
    ax = axes[0]
    seg_adoption = pd.crosstab(df['supplier_segment'], df['adoption_level'], normalize='index') * 100
    x = np.arange(len(segment_actual))
    for i, adoption in enumerate(adoption_order):
        values = [seg_adoption.loc[seg, adoption] if seg in seg_adoption.index and adoption in seg_adoption.columns else 0 for seg in segment_actual]
        values = [v if np.isfinite(v) else 0 for v in values]
        ax.bar(x + i * 0.18, values, 0.18, label=adoption, color=PALETTE_5[i], edgecolor='white', linewidth=1.5)
    ax.set_xlabel('Supplier Segment', fontweight='bold')
    ax.set_ylabel('Percentage (%)', fontweight='bold')
    ax.set_title('Adoption by Segment', fontweight='bold', pad=10)
    ax.set_xticks(x + 0.36)
    ax.set_xticklabels(segment_actual, rotation=45, ha='right')
    ax.legend(loc='upper right', fontsize=7)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.grid(axis='y', alpha=0.3, linestyle='--', linewidth=0.5)
    
    # Average feature count by segment
    ax = axes[1]
    avg_features_seg = df.groupby('supplier_segment')['advanced_features_count'].mean().reindex(segment_actual)
    avg_features_seg = avg_features_seg.fillna(0)
    colors_seg = ['#FF6B35', '#FF8C5A', '#FFAD7F', '#D9D9D9', '#B0B0B0', '#909090']
    bars = ax.bar(segment_actual, avg_features_seg, color=colors_seg[:len(segment_actual)], 
                  edgecolor='white', linewidth=2)
    ax.set_xlabel('Supplier Segment', fontweight='bold')
    ax.set_ylabel('Average Features Used', fontweight='bold')
    ax.set_title('Average Feature Count by Segment', fontweight='bold', pad=10)
    ax.set_xticklabels(segment_actual, rotation=45, ha='right')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.grid(axis='y', alpha=0.3, linestyle='--', linewidth=0.5)
    for i, val in enumerate(avg_features_seg):
        if not np.isnan(val) and np.isfinite(val):
            ax.text(i, val + 0.1, f'{val:.2f}', ha='center', fontweight='bold', color=COLORS['dark'])
    
    plt.tight_layout()
    display(fig)
    plt.close()
else:
    print("supplier_segment field not found in data or no segments available")

# COMMAND ----------

# MARKDOWN
"""
## Chart 6: Adoption by Connection and Management Status

**What this analyzes:**
- API-connected vs non-connected suppliers (API integration enables advanced features)
- Managed vs non-managed suppliers (managed suppliers receive direct account support)

**Hypothesis:**
- API-connected suppliers should have higher adoption (API unlocks scaling features)
- Managed suppliers should have higher adoption (direct support drives feature usage)

**Business value:**
- Quantifies the impact of API connectivity on feature adoption
- Validates the ROI of supplier management programs
"""

# COMMAND ----------

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Adoption by Connection and Management Status', fontsize=16, fontweight='bold', y=0.995)

# Adoption by connection status
ax = axes[0, 0]
conn_adoption = pd.crosstab(df['connection_status'], df['adoption_level'], normalize='index') * 100
conn_order = ['Not Connected', 'Connected']
x = np.arange(len(conn_order))
for i, adoption in enumerate(adoption_order):
    values = [conn_adoption.loc[conn, adoption] if conn in conn_adoption.index and adoption in conn_adoption.columns else 0 for conn in conn_order]
    values = [v if np.isfinite(v) else 0 for v in values]
    ax.bar(x + i * 0.18, values, 0.18, label=adoption, color=PALETTE_5[i], edgecolor='white', linewidth=1.5)
ax.set_xlabel('Connection Status', fontweight='bold')
ax.set_ylabel('Percentage (%)', fontweight='bold')
ax.set_title('Adoption by Connection', fontweight='bold', pad=10)
ax.set_xticks(x + 0.36)
ax.set_xticklabels(conn_order, rotation=0)
ax.legend(loc='upper left', fontsize=6)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.grid(axis='y', alpha=0.3, linestyle='--', linewidth=0.5)

# Average features by connection
ax = axes[0, 1]
avg_feat_conn = df.groupby('connection_status')['advanced_features_count'].mean().reindex(conn_order)
avg_feat_conn = avg_feat_conn.fillna(0)
bars = ax.bar(conn_order, avg_feat_conn, color=['#FF6B35', '#FFAD7F'], edgecolor='white', linewidth=2)
ax.set_xlabel('Connection Status', fontweight='bold')
ax.set_ylabel('Average Features Used', fontweight='bold')
ax.set_title('Features by Connection', fontweight='bold', pad=10)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.grid(axis='y', alpha=0.3, linestyle='--', linewidth=0.5)
for i, val in enumerate(avg_feat_conn):
    if not np.isnan(val) and np.isfinite(val):
        ax.text(i, val + 0.1, f'{val:.2f}', ha='center', fontweight='bold', color=COLORS['dark'])

# Adoption by managed status
ax = axes[1, 0]
mgd_adoption = pd.crosstab(df['managed_status'], df['adoption_level'], normalize='index') * 100
mgd_order = ['Not Managed', 'Managed']
x = np.arange(len(mgd_order))
for i, adoption in enumerate(adoption_order):
    values = [mgd_adoption.loc[mgd, adoption] if mgd in mgd_adoption.index and adoption in mgd_adoption.columns else 0 for mgd in mgd_order]
    values = [v if np.isfinite(v) else 0 for v in values]
    ax.bar(x + i * 0.18, values, 0.18, label=adoption, color=PALETTE_5[i], edgecolor='white', linewidth=1.5)
ax.set_xlabel('Management Status', fontweight='bold')
ax.set_ylabel('Percentage (%)', fontweight='bold')
ax.set_title('Adoption by Managed Status', fontweight='bold', pad=10)
ax.set_xticks(x + 0.36)
ax.set_xticklabels(mgd_order, rotation=0)
ax.legend(loc='upper left', fontsize=6)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.grid(axis='y', alpha=0.3, linestyle='--', linewidth=0.5)

# Average features by managed status
ax = axes[1, 1]
avg_feat_mgd = df.groupby('managed_status')['advanced_features_count'].mean().reindex(mgd_order)
avg_feat_mgd = avg_feat_mgd.fillna(0)
bars = ax.bar(mgd_order, avg_feat_mgd, color=['#FF6B35', '#FFAD7F'], edgecolor='white', linewidth=2)
ax.set_xlabel('Management Status', fontweight='bold')
ax.set_ylabel('Average Features Used', fontweight='bold')
ax.set_title('Features by Managed Status', fontweight='bold', pad=10)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.grid(axis='y', alpha=0.3, linestyle='--', linewidth=0.5)
for i, val in enumerate(avg_feat_mgd):
    if not np.isnan(val) and np.isfinite(val):
        ax.text(i, val + 0.1, f'{val:.2f}', ha='center', fontweight='bold', color=COLORS['dark'])

plt.tight_layout()
display(fig)
plt.close()

# COMMAND ----------

# MARKDOWN
"""
## Chart 7: Cross-Segment Analysis - GMV Tier × Market Maturity

**What this analyzes:**
- Power user adoption rates across combinations of GMV tier and maturity
- Average feature usage across combinations

**Why cross-segment matters:**
Single-dimension analysis can hide important patterns. For example:
- Are NEW high-GMV tours adopting faster than MATURE low-GMV tours?
- Do high-GMV tours start strong or grow into features over time?

**Heatmap interpretation:**
- Hot spots (orange/red) = high adoption segments to use as case studies
- Cold spots (light) = opportunity segments for targeted enablement
"""

# COMMAND ----------

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Cross-Segment: GMV Tier × Market Maturity', fontsize=16, fontweight='bold', y=1.02)

# Power user adoption heatmap
ax = axes[0]
power_cross = pd.crosstab(df['gmv_tier'], df['market_maturity'], 
                          values=(df['adoption_level'] == 'Power (8+)').astype(int), aggfunc='mean') * 100
power_cross = power_cross.reindex(index=gmv_tier_order, columns=maturity_order).fillna(0)
im = ax.imshow(power_cross.values, cmap='Oranges', aspect='auto', vmin=0, vmax=max(power_cross.values.max(), 0.1))
ax.set_xticks(range(len(maturity_order)))
ax.set_xticklabels(['New\n(0-6m)', 'Developing\n(6-18m)', 'Mature\n(18m+)'], rotation=0)
ax.set_yticks(range(len(gmv_tier_order)))
ax.set_yticklabels(gmv_tier_order, fontsize=8)
ax.set_xlabel('Market Maturity', fontweight='bold')
ax.set_ylabel('GMV Tier', fontweight='bold')
ax.set_title('Power User Adoption Rate (%)', fontweight='bold', pad=10)
for i in range(len(gmv_tier_order)):
    for j in range(len(maturity_order)):
        val = power_cross.values[i, j]
        if np.isfinite(val):
            ax.text(j, i, f'{val:.1f}%', ha='center', va='center', 
                   color='white' if val > power_cross.values.max()/2 else 'black', fontweight='bold', fontsize=8)
cbar = plt.colorbar(im, ax=ax)
cbar.set_label('Adoption Rate (%)', rotation=270, labelpad=20, fontweight='bold')

# Average features heatmap
ax = axes[1]
avg_feat_cross = pd.crosstab(df['gmv_tier'], df['market_maturity'], 
                             values=df['advanced_features_count'], aggfunc='mean')
avg_feat_cross = avg_feat_cross.reindex(index=gmv_tier_order, columns=maturity_order).fillna(0)
im = ax.imshow(avg_feat_cross.values, cmap='Oranges', aspect='auto', vmin=0, vmax=max(avg_feat_cross.values.max(), 0.1))
ax.set_xticks(range(len(maturity_order)))
ax.set_xticklabels(['New\n(0-6m)', 'Developing\n(6-18m)', 'Mature\n(18m+)'], rotation=0)
ax.set_yticks(range(len(gmv_tier_order)))
ax.set_yticklabels(gmv_tier_order, fontsize=8)
ax.set_xlabel('Market Maturity', fontweight='bold')
ax.set_ylabel('GMV Tier', fontweight='bold')
ax.set_title('Average Features Used', fontweight='bold', pad=10)
for i in range(len(gmv_tier_order)):
    for j in range(len(maturity_order)):
        val = avg_feat_cross.values[i, j]
        if np.isfinite(val):
            ax.text(j, i, f'{val:.2f}', ha='center', va='center',
                   color='white' if val > avg_feat_cross.values.max()/2 else 'black', fontweight='bold', fontsize=8)
cbar = plt.colorbar(im, ax=ax)
cbar.set_label('Avg Features', rotation=270, labelpad=20, fontweight='bold')

plt.tight_layout()
display(fig)
plt.close()

# COMMAND ----------

# MARKDOWN
"""
## Summary Statistics and Key Takeaways

**What this provides:**
- Overall adoption rate (percentage using at least one feature)
- Power user rate (percentage using 8+ features)
- Average features per tour
- Distribution summary
- GMV tier ranges derived from data

**Use these metrics to:**
- Track adoption improvement over time (run this monthly)
- Set targets for product and enablement teams
- Communicate impact to leadership
"""

# COMMAND ----------

# Calculate summary metrics
adopters = (df['adoption_level'] != 'Non-adopters').sum()
adoption_rate = (adopters / len(df)) * 100
power_users = (df['adoption_level'] == 'Power (8+)').sum()
power_rate = (power_users / len(df)) * 100
avg_feat = df['advanced_features_count'].mean()
median_feat = df['advanced_features_count'].median()

print("="*60)
print("PRICING FEATURE ADOPTION SUMMARY")
print("="*60)
print(f"\nTotal tours analyzed: {len(df):,}")
print(f"Total suppliers: {df['supplier_id'].nunique():,}")
print(f"Features tracked: {len(feature_cols)}")
print(f"\nGMV TIERS (Data-Driven {n_bins} tiers):")
print(f"  Quartile 0% (min): {gmv_quantiles[0]:,.0f}")
print(f"  Quartile 25%: {gmv_quantiles[0.25]:,.0f}")
print(f"  Quartile 50% (median): {gmv_quantiles[0.5]:,.0f}")
print(f"  Quartile 75%: {gmv_quantiles[0.75]:,.0f}")
print(f"  Quartile 100% (max): {gmv_quantiles[1.0]:,.0f}")
print(f"\nOVERALL ADOPTION METRICS:")
print(f"  • Overall adoption rate: {adoption_rate:.1f}%")
print(f"  • Non-adopters: {100-adoption_rate:.1f}%")
print(f"  • Power users (8+ features): {power_rate:.1f}%")
print(f"  • Average features per tour: {avg_feat:.2f}")
print(f"  • Median features per tour: {median_feat:.0f}")
print(f"\nFEATURES ANALYZED:")
for i, feat in enumerate(feature_cols, 1):
    print(f"  {i}. {feat}")
print("="*60)

# Show adoption breakdown
print("\nADOPTION LEVEL BREAKDOWN:")
adoption_breakdown = df['adoption_level'].value_counts().reindex(adoption_order)
for level in adoption_order:
    count = adoption_breakdown.get(level, 0)
    pct = (count / len(df)) * 100
    print(f"  {level:20s}: {count:6,} tours ({pct:5.1f}%)")

print("\n" + "="*60)


In [0]:
# Databricks notebook source

# MARKDOWN
"""
# Pricing Feature Adoption Audit

## Objective
This notebook analyzes pricing feature adoption across GetYourGuide's supplier base to identify which segments are utilizing advanced pricing capabilities and which segments represent opportunities for feature education and adoption.

## Business Questions
1. What percentage of tours use advanced pricing features?
2. Which supplier segments (by volume, maturity, category) have highest/lowest adoption?
3. Which specific features are most used within each segment?
4. What opportunities exist to increase feature penetration?

## Features Analyzed (10 Total)
- Individual Pricing
- Group Pricing  
- Pricing without API (Native)
- Pricing with API
- Scales without API (Native Scales)
- Scales with API
- Live Pricing and Availability
- Non-live Price and Availability
- Add-ons
- Options
"""

# COMMAND ----------

# MARKDOWN
"""
## Section 1: Data Loading and Type Conversion

**What this does:** 
- Loads the pricing feature data from the Spark table
- Converts all columns to proper numeric types to avoid calculation errors
- Handles missing values by filling with 0
- Converts dates to datetime format

**Why it matters:**
Data type issues cause errors in aggregations and binning. This ensures clean numeric data for all calculations.
"""

# COMMAND ----------

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import rcParams
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Load data
print("Loading data from Spark...")
spark_df = spark.table("production.supply_analytics.pricing_feature_combined")
df = spark_df.toPandas()

print(f"Loaded {len(df):,} tours from {df['supplier_id'].nunique():,} suppliers")

# Fix data types - critical for avoiding errors
numeric_cols = ['gmv_l12mo', 'bookings_l12mo', 'has_group_pricing', 'has_individual_pricing',
                'has_addons', 'has_options', 'has_native_scales', 'has_api_scales', 
                'has_live_dynamic_pricing', 'has_non_live_pricing', 'uses_api_pricing',
                'has_scale_pricing', 'is_managed', 'num_individual_categories']

for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)

if 'date_of_first_online' in df.columns:
    df['date_of_first_online'] = pd.to_datetime(df['date_of_first_online'], errors='coerce')

# Ensure supplier_segment and activity_type are strings
if 'supplier_segment' in df.columns:
    df['supplier_segment'] = df['supplier_segment'].fillna('Unknown').astype(str)
if 'activity_type' in df.columns:
    df['activity_type'] = df['activity_type'].fillna('Unknown').astype(str)

print("✓ Data types converted")
print("\nChecking key fields:")
print(f"  - supplier_segment field exists: {'supplier_segment' in df.columns}")
print(f"  - activity_type field exists: {'activity_type' in df.columns}")

if 'supplier_segment' in df.columns:
    print(f"\nSupplier segments in data: {df['supplier_segment'].unique()}")
if 'activity_type' in df.columns:
    print(f"\nActivity types in data (first 10): {df['activity_type'].unique()[:10]}")

display(df.head())

# COMMAND ----------

# MARKDOWN
"""
## Section 2: Segmentation Creation

**What this does:**
- Creates GMV tiers based on percentile rankings: Top 5%, Top 25%, Top 50%, Bottom 50%
- Creates Market Maturity segments based on time on platform (New 0-6mo, Developing 6-18mo, Mature 18mo+)
- Uses the actual supplier_segment field from the data (Enterprise, Mid-Market, SMB, Micro)
- Uses the actual activity_type field from the data
- Identifies API connection status and managed supplier status

**Why it matters:**
Percentile-based GMV tiers identify high-value suppliers who drive the most revenue. Top 5% and Top 25% represent strategic segments for premium features. Different supplier segments have different adoption patterns.

**Analytical approach:**
GMV tiers use industry-standard percentile cuts to identify top performers. This enables targeted analysis of high-value supplier behavior versus long tail.
"""

# COMMAND ----------

# GMV tier - percentile-based segmentation
gmv_percentiles = df['gmv_l12mo'].quantile([0, 0.50, 0.75, 0.95, 1.0])
print("GMV Distribution Percentiles:")
print(gmv_percentiles)

# Create percentile-based tiers
df['gmv_tier'] = 'Bottom 50%'
df.loc[df['gmv_l12mo'] >= gmv_percentiles[0.50], 'gmv_tier'] = 'Top 50%'
df.loc[df['gmv_l12mo'] >= gmv_percentiles[0.75], 'gmv_tier'] = 'Top 25%'
df.loc[df['gmv_l12mo'] >= gmv_percentiles[0.95], 'gmv_tier'] = 'Top 5%'

# Define tier order
gmv_tier_order = ['Top 5%', 'Top 25%', 'Top 50%', 'Bottom 50%']

print(f"\nGMV Tiers created:")
print(f"  Top 5%: GMV ≥ {gmv_percentiles[0.95]:,.0f}")
print(f"  Top 25%: GMV ≥ {gmv_percentiles[0.75]:,.0f}")
print(f"  Top 50%: GMV ≥ {gmv_percentiles[0.50]:,.0f}")
print(f"  Bottom 50%: GMV < {gmv_percentiles[0.50]:,.0f}")

for tier in gmv_tier_order:
    count = (df['gmv_tier'] == tier).sum()
    pct = count / len(df) * 100
    avg_gmv = df[df['gmv_tier'] == tier]['gmv_l12mo'].mean()
    print(f"    {tier}: {count:,} tours ({pct:.1f}%), Avg GMV: {avg_gmv:,.0f}")

# Market maturity - time-based segmentation
df['days_on_platform'] = (datetime.now() - df['date_of_first_online']).dt.days
df['market_maturity'] = 'Mature (18+ months)'
df.loc[df['days_on_platform'] <= 540, 'market_maturity'] = 'Developing (6-18 months)'
df.loc[df['days_on_platform'] <= 180, 'market_maturity'] = 'New (0-6 months)'

# Connection and management status
df['connection_status'] = df['uses_api_pricing'].apply(lambda x: 'Connected' if x == 1 else 'Not Connected')
df['managed_status'] = df['is_managed'].apply(lambda x: 'Managed' if x == 1 else 'Not Managed')

# Verify segment distribution
print("\n✓ Segments created\n")
print("Market Maturity Distribution:")
display(df['market_maturity'].value_counts())
print("\nSupplier Segment Distribution (from data):")
if 'supplier_segment' in df.columns:
    display(df['supplier_segment'].value_counts())
print("\nActivity Type Distribution (from data - top 15):")
if 'activity_type' in df.columns:
    display(df['activity_type'].value_counts().head(15))

# COMMAND ----------

# MARKDOWN
"""
## Section 3: Feature Count Calculation and Adoption Level Assignment

**What this does:**
- Counts how many of the 10 pricing features each tour uses
- Creates 5 adoption levels: Non-adopters (0), Low (1-2), Medium (3-5), High (6-7), Power (8+)

**Why it matters:**
This is the core metric for measuring feature adoption maturity. Tours using more features likely generate more revenue per customer through upselling and dynamic optimization. This metric helps identify tours at risk (non-adopters) and best-in-class (power users).

**Analytical approach:**
Additive feature counting treats all features as equally valuable. The 5-tier classification creates actionable segments for targeted interventions.
"""

# COMMAND ----------

# Build feature list dynamically based on available columns
feature_cols = []
if 'has_individual_pricing' in df.columns:
    feature_cols.append('has_individual_pricing')
if 'has_group_pricing' in df.columns:
    feature_cols.append('has_group_pricing')
if 'uses_api_pricing' in df.columns:
    feature_cols.append('uses_api_pricing')
if 'has_native_scales' in df.columns:
    feature_cols.append('has_native_scales')
if 'has_api_scales' in df.columns:
    feature_cols.append('has_api_scales')
if 'has_live_dynamic_pricing' in df.columns:
    feature_cols.append('has_live_dynamic_pricing')
if 'has_non_live_pricing' in df.columns:
    feature_cols.append('has_non_live_pricing')
if 'has_addons' in df.columns:
    feature_cols.append('has_addons')
if 'has_options' in df.columns:
    feature_cols.append('has_options')

# Calculate native pricing as inverse of API pricing
if 'uses_api_pricing' in df.columns:
    df['has_native_pricing'] = (1 - df['uses_api_pricing']).clip(0, 1)
    feature_cols.append('has_native_pricing')

# Sum features per tour
df['advanced_features_count'] = df[feature_cols].sum(axis=1)

print(f"Counting {len(feature_cols)} features: {feature_cols}")
print(f"Max features used by any tour: {df['advanced_features_count'].max()}")
print(f"Mean features per tour: {df['advanced_features_count'].mean():.2f}")
print(f"Median features per tour: {df['advanced_features_count'].median():.0f}")

# Create adoption levels
df['adoption_level'] = 'Power (8+)'
df.loc[df['advanced_features_count'] <= 7, 'adoption_level'] = 'High (6-7)'
df.loc[df['advanced_features_count'] <= 5, 'adoption_level'] = 'Medium (3-5)'
df.loc[df['advanced_features_count'] <= 2, 'adoption_level'] = 'Low (1-2)'
df.loc[df['advanced_features_count'] == 0, 'adoption_level'] = 'Non-adopters'

df['uses_multiple_categories'] = (df['num_individual_categories'] >= 3).astype(int)

# Display distribution
print("\nAdoption Level Distribution:")
display(df['adoption_level'].value_counts())

# COMMAND ----------

# MARKDOWN
"""
## Section 4: Visualization Setup

**What this does:**
- Sets consistent color palettes and styling for all charts
- Defines segment ordering for consistent visualization
- Configures matplotlib parameters

**Why it matters:**
Consistent visual styling makes the analysis professional and easier to interpret. Ordered categories show progression naturally.
"""

# COMMAND ----------

# Styling
rcParams['font.family'] = 'sans-serif'
rcParams['font.sans-serif'] = ['Arial', 'DejaVu Sans']
rcParams['font.size'] = 10

COLORS = {'primary': '#FF6B35', 'secondary': '#F77F51', 'tertiary': '#FFA07A', 
          'quaternary': '#C0C0C0', 'quinary': '#808080', 'dark': '#404040'}
PALETTE_5 = ['#FF6B35', '#FF8C5A', '#FFAD7F', '#CCCCCC', '#808080']

# Define consistent orderings
maturity_order = ['New (0-6 months)', 'Developing (6-18 months)', 'Mature (18+ months)']
adoption_order = ['Non-adopters', 'Low (1-2)', 'Medium (3-5)', 'High (6-7)', 'Power (8+)']

# Get actual segment order from data
if 'supplier_segment' in df.columns:
    segment_actual = df['supplier_segment'].value_counts().index.tolist()
    print(f"Supplier segments from data: {segment_actual}")
else:
    segment_actual = []

print("✓ Styling configured")

# COMMAND ----------

# MARKDOWN
"""
## Chart 1: Overall Feature Adoption Overview

**What this analyzes:**
- Distribution of tours across the 5 adoption levels
- Individual feature adoption rates (which features are most/least used)
- Ticket category complexity distribution
- Overall distribution of feature count

**Business insights to extract:**
- What percentage of our catalog is at risk (non-adopters)?
- Which features have low adoption and need education/enablement?
- Are suppliers using category complexity effectively?

**Interpretation guide:**
- High percentage of non-adopters = opportunity for revenue growth through feature education
- Wide gap between top features and bottom features = some features may need better UX or clearer value prop
"""

# COMMAND ----------

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Pricing Feature Adoption Overview', fontsize=16, fontweight='bold', y=0.995)

# Adoption level distribution
ax = axes[0, 0]
adoption_pcts = (df['adoption_level'].value_counts().reindex(adoption_order) / len(df) * 100)
adoption_pcts = adoption_pcts.fillna(0).values
bars = ax.barh(adoption_order, adoption_pcts, color=PALETTE_5, edgecolor='white', linewidth=2)
ax.set_xlabel('Percentage of Tours', fontweight='bold')
ax.set_title(f'Feature Adoption Levels ({len(feature_cols)} Features)', fontweight='bold', pad=10)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
for i, val in enumerate(adoption_pcts):
    if not np.isnan(val) and np.isfinite(val):
        ax.text(val + 1, i, f'{val:.1f}%', va='center', fontweight='bold', color=COLORS['dark'])

# Individual feature adoption
ax = axes[0, 1]
features = {}
if 'has_group_pricing' in df.columns:
    features['Group Pricing'] = df['has_group_pricing'].sum()
if 'has_individual_pricing' in df.columns:
    features['Individual Pricing'] = df['has_individual_pricing'].sum()
if 'has_addons' in df.columns:
    features['Add-ons'] = df['has_addons'].sum()
if 'has_options' in df.columns:
    features['Options'] = df['has_options'].sum()
if 'uses_api_pricing' in df.columns:
    features['API Pricing'] = df['uses_api_pricing'].sum()
if 'has_native_scales' in df.columns:
    features['Native Scales'] = df['has_native_scales'].sum()
if 'has_api_scales' in df.columns:
    features['API Scales'] = df['has_api_scales'].sum()
if 'has_live_dynamic_pricing' in df.columns:
    features['Live Pricing'] = df['has_live_dynamic_pricing'].sum()

feature_pcts = [(v / len(df)) * 100 for v in features.values()]
colors_bar = ['#FF6B35', '#FF7A47', '#FF8C5A', '#FF9E6C', '#FFAD7F', '#FFBC91', '#FFCBA4', '#D9D9D9']
bars = ax.barh(list(features.keys()), feature_pcts, color=colors_bar[:len(features)], 
               edgecolor='white', linewidth=2)
ax.set_xlabel('Adoption Rate (%)', fontweight='bold')
ax.set_title('Individual Feature Adoption', fontweight='bold', pad=10)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
for i, val in enumerate(feature_pcts):
    if not np.isnan(val) and np.isfinite(val):
        ax.text(val + 1, i, f'{val:.1f}%', va='center', fontweight='bold', color=COLORS['dark'])

# Category complexity
ax = axes[1, 0]
cat_dist = {'Adult Only': (df['num_individual_categories'] == 1).sum(),
            'Adult + Child': (df['num_individual_categories'] == 2).sum(),
            '3 Categories': (df['num_individual_categories'] == 3).sum(),
            '4+ Categories': (df['num_individual_categories'] >= 4).sum()}
cat_pcts = [(v / len(df)) * 100 for v in cat_dist.values()]
bars = ax.bar(range(len(cat_pcts)), cat_pcts, color=PALETTE_5[:4], edgecolor='white', linewidth=2)
ax.set_xticks(range(len(cat_dist)))
ax.set_xticklabels(cat_dist.keys(), rotation=0)
ax.set_ylabel('Percentage of Tours', fontweight='bold')
ax.set_title('Ticket Category Complexity', fontweight='bold', pad=10)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
for i, val in enumerate(cat_pcts):
    if not np.isnan(val) and np.isfinite(val):
        ax.text(i, val + 1, f'{val:.1f}%', ha='center', fontweight='bold', color=COLORS['dark'])

# Feature count distribution
ax = axes[1, 1]
feature_dist = df['advanced_features_count'].value_counts().sort_index()
ax.bar(feature_dist.index, feature_dist.values, color='#FF6B35', edgecolor='white', linewidth=2)
ax.set_xlabel('Number of Features Used', fontweight='bold')
ax.set_ylabel('Number of Tours', fontweight='bold')
ax.set_title('Distribution of Feature Count', fontweight='bold', pad=10)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
display(fig)
plt.close()

# COMMAND ----------

# MARKDOWN
"""
## Chart 2: Adoption by GMV Tier (Percentile-Based: Top 5%, Top 25%, Top 50%, Bottom 50%)

**What this analyzes:**
- How feature adoption differs across GMV percentiles
- Average feature count by GMV tier
- Uses percentile-based segmentation to identify high-value suppliers

**Business hypothesis to test:**
Top 5% and Top 25% GMV tours should have higher adoption rates because:
1. They have more resources to invest in pricing optimization
2. They see clearer ROI from advanced features
3. They may receive more support from GetYourGuide account managers

**Key metrics:**
- Percentage of Power Users in Top 5% vs Bottom 50%
- Adoption gap between tiers indicates opportunity or resource constraints
"""

# COMMAND ----------

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Feature Adoption by GMV Tier (Top 5%, Top 25%, Top 50%, Bottom 50%)', fontsize=16, fontweight='bold', y=1.02)

# Adoption distribution by GMV tier
ax = axes[0]
gmv_adoption = pd.crosstab(df['gmv_tier'], df['adoption_level'], normalize='index') * 100
x = np.arange(len(gmv_tier_order))
width = 0.18
for i, adoption in enumerate(adoption_order):
    values = [gmv_adoption.loc[tier, adoption] if tier in gmv_adoption.index and adoption in gmv_adoption.columns else 0 for tier in gmv_tier_order]
    values = [v if np.isfinite(v) else 0 for v in values]
    ax.bar(x + i * width, values, width, label=adoption, color=PALETTE_5[i], edgecolor='white', linewidth=1.5)
ax.set_xlabel('GMV Tier', fontweight='bold')
ax.set_ylabel('Percentage (%)', fontweight='bold')
ax.set_title('Adoption Distribution Within GMV Tier', fontweight='bold', pad=10)
ax.set_xticks(x + width * 2)
ax.set_xticklabels(gmv_tier_order, rotation=0, fontsize=9)
ax.legend(loc='upper right', fontsize=7)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.grid(axis='y', alpha=0.3, linestyle='--', linewidth=0.5)

# Average feature count by GMV tier
ax = axes[1]
avg_features_gmv = df.groupby('gmv_tier')['advanced_features_count'].mean().reindex(gmv_tier_order)
avg_features_gmv = avg_features_gmv.fillna(0)
bars = ax.bar(range(len(gmv_tier_order)), avg_features_gmv, color=['#D32F2F', '#FF6B35', '#FF8C5A', '#CCCCCC'], 
              edgecolor='white', linewidth=2)
ax.set_xlabel('GMV Tier', fontweight='bold')
ax.set_ylabel('Average Features Used', fontweight='bold')
ax.set_title('Average Feature Count by GMV Tier', fontweight='bold', pad=10)
ax.set_xticks(range(len(gmv_tier_order)))
ax.set_xticklabels(gmv_tier_order, rotation=0, fontsize=9)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.grid(axis='y', alpha=0.3, linestyle='--', linewidth=0.5)
for i, val in enumerate(avg_features_gmv):
    if not np.isnan(val) and np.isfinite(val):
        ax.text(i, val + 0.1, f'{val:.2f}', ha='center', fontweight='bold', color=COLORS['dark'])

plt.tight_layout()
display(fig)
plt.close()

# COMMAND ----------

# MARKDOWN
"""
## Chart 3: Most Common Features by GMV Tier

**What this analyzes:**
- Which specific features are most frequently used within each GMV tier
- Shows top 5 features for each tier
- Reveals whether high-GMV tours use different features than low-GMV tours

**Business insights:**
- Do Top 5% tours leverage advanced features like Live Pricing and API Scales?
- Are Bottom 50% tours stuck on basic features like Individual Pricing only?
- Feature usage patterns can inform targeted education campaigns

**Interpretation:**
Higher adoption rates for specific features in top tiers may indicate those features drive revenue growth. Low adoption of advanced features in bottom tiers represents untapped opportunity.
"""

# COMMAND ----------

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Most Common Features by GMV Tier', fontsize=16, fontweight='bold', y=0.995)

feature_display_names = {
    'has_individual_pricing': 'Individual Pricing',
    'has_group_pricing': 'Group Pricing',
    'uses_api_pricing': 'API Pricing',
    'has_native_pricing': 'Native Pricing',
    'has_native_scales': 'Native Scales',
    'has_api_scales': 'API Scales',
    'has_live_dynamic_pricing': 'Live Pricing',
    'has_non_live_pricing': 'Non-live Pricing',
    'has_addons': 'Add-ons',
    'has_options': 'Options'
}

for idx, tier in enumerate(gmv_tier_order):
    ax = axes[idx // 2, idx % 2]
    tier_df = df[df['gmv_tier'] == tier]
    
    feature_usage = {}
    for feat in feature_cols:
        if feat in tier_df.columns:
            adoption_rate = (tier_df[feat].sum() / len(tier_df)) * 100
            feature_usage[feature_display_names.get(feat, feat)] = adoption_rate
    
    # Sort and get top 5
    sorted_features = sorted(feature_usage.items(), key=lambda x: x[1], reverse=True)[:5]
    feature_names = [f[0] for f in sorted_features]
    feature_vals = [f[1] for f in sorted_features]
    
    colors_grad = ['#D32F2F', '#FF6B35', '#FF8C5A', '#FFAD7F', '#CCCCCC']
    bars = ax.barh(feature_names, feature_vals, color=colors_grad[:len(feature_names)], 
                   edgecolor='white', linewidth=2)
    ax.set_xlabel('Adoption Rate (%)', fontweight='bold')
    ax.set_title(f'{tier} (n={len(tier_df):,})', fontweight='bold', pad=10)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.set_xlim(0, 100)
    
    for i, val in enumerate(feature_vals):
        if not np.isnan(val) and np.isfinite(val):
            ax.text(val + 2, i, f'{val:.1f}%', va='center', fontweight='bold', color=COLORS['dark'])

plt.tight_layout()
display(fig)
plt.close()

# COMMAND ----------

# MARKDOWN
"""
## Chart 4: Adoption by Market Maturity

**What this analyzes:**
- How feature adoption evolves as suppliers mature on the platform
- Average feature usage by time on platform

**Business hypothesis to test:**
Mature suppliers should have higher adoption because:
1. They have had more time to learn the platform
2. They have established processes and can invest in optimization
3. They may have received more training over time

**Counterpoint to watch for:**
If NEW suppliers have higher adoption, it may indicate improved onboarding programs or selection bias.
"""

# COMMAND ----------

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Feature Adoption by Market Maturity', fontsize=16, fontweight='bold', y=1.02)

# Adoption distribution by maturity
ax = axes[0]
mat_adoption = pd.crosstab(df['market_maturity'], df['adoption_level'], normalize='index') * 100
x = np.arange(len(maturity_order))
for i, adoption in enumerate(adoption_order):
    values = [mat_adoption.loc[mat, adoption] if mat in mat_adoption.index and adoption in mat_adoption.columns else 0 for mat in maturity_order]
    values = [v if np.isfinite(v) else 0 for v in values]
    ax.bar(x + i * 0.18, values, 0.18, label=adoption, color=PALETTE_5[i], edgecolor='white', linewidth=1.5)
ax.set_xlabel('Market Maturity', fontweight='bold')
ax.set_ylabel('Percentage (%)', fontweight='bold')
ax.set_title('Adoption Distribution by Maturity', fontweight='bold', pad=10)
ax.set_xticks(x + 0.36)
ax.set_xticklabels(['New\n(0-6m)', 'Developing\n(6-18m)', 'Mature\n(18m+)'], rotation=0)
ax.legend(loc='upper left', fontsize=7)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.grid(axis='y', alpha=0.3, linestyle='--', linewidth=0.5)

# Average feature count by maturity
ax = axes[1]
avg_features_mat = df.groupby('market_maturity')['advanced_features_count'].mean().reindex(maturity_order)
avg_features_mat = avg_features_mat.fillna(0)
bars = ax.bar(range(len(maturity_order)), avg_features_mat, color=['#FF6B35', '#FF8C5A', '#FFAD7F'], 
              edgecolor='white', linewidth=2)
ax.set_xlabel('Market Maturity', fontweight='bold')
ax.set_ylabel('Average Features Used', fontweight='bold')
ax.set_title('Average Feature Count by Maturity', fontweight='bold', pad=10)
ax.set_xticks(range(len(maturity_order)))
ax.set_xticklabels(['New\n(0-6m)', 'Developing\n(6-18m)', 'Mature\n(18m+)'], rotation=0)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.grid(axis='y', alpha=0.3, linestyle='--', linewidth=0.5)
for i, val in enumerate(avg_features_mat):
    if not np.isnan(val) and np.isfinite(val):
        ax.text(i, val + 0.1, f'{val:.2f}', ha='center', fontweight='bold', color=COLORS['dark'])

plt.tight_layout()
display(fig)
plt.close()

# COMMAND ----------

# MARKDOWN
"""
## Chart 5: Most Common Features by Market Maturity

**What this analyzes:**
- Which specific features are most frequently used within each maturity stage
- Shows top 5 features for New, Developing, and Mature suppliers
- Reveals how feature adoption evolves as suppliers gain experience

**Business insights:**
- Do new suppliers adopt different features than mature suppliers?
- Are certain features easier to adopt early (e.g., Individual Pricing) versus requiring more experience (e.g., Live Pricing)?
- Maturity-specific patterns can inform onboarding sequencing
"""

# COMMAND ----------

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Most Common Features by Market Maturity', fontsize=16, fontweight='bold', y=1.02)

for idx, maturity in enumerate(maturity_order):
    ax = axes[idx]
    mat_df = df[df['market_maturity'] == maturity]
    
    feature_usage = {}
    for feat in feature_cols:
        if feat in mat_df.columns:
            adoption_rate = (mat_df[feat].sum() / len(mat_df)) * 100
            feature_usage[feature_display_names.get(feat, feat)] = adoption_rate
    
    # Sort and get top 5
    sorted_features = sorted(feature_usage.items(), key=lambda x: x[1], reverse=True)[:5]
    feature_names = [f[0] for f in sorted_features]
    feature_vals = [f[1] for f in sorted_features]
    
    colors_grad = ['#FF6B35', '#FF7A47', '#FF8C5A', '#FFAD7F', '#CCCCCC']
    bars = ax.barh(feature_names, feature_vals, color=colors_grad[:len(feature_names)], 
                   edgecolor='white', linewidth=2)
    ax.set_xlabel('Adoption Rate (%)', fontweight='bold')
    ax.set_title(f'{maturity}\n(n={len(mat_df):,})', fontweight='bold', pad=10)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.set_xlim(0, 100)
    
    for i, val in enumerate(feature_vals):
        if not np.isnan(val) and np.isfinite(val):
            ax.text(val + 2, i, f'{val:.1f}%', va='center', fontweight='bold', color=COLORS['dark'])

plt.tight_layout()
display(fig)
plt.close()

# COMMAND ----------

# MARKDOWN
"""
## Chart 6: Adoption by Supplier Segment (FROM DATA FIELD)

**What this analyzes:**
- Feature adoption using the actual supplier_segment field from the data
- Average feature count by segment

**Business context:**
- Uses the real supplier segmentation from your data
- Enterprise suppliers have dedicated account managers and technical resources
- SMB and Micro suppliers may need self-service education and simpler onboarding
- Segment-specific adoption gaps indicate where to invest in support resources
"""

# COMMAND ----------

if 'supplier_segment' in df.columns and len(segment_actual) > 0:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle('Feature Adoption by Supplier Segment (from data)', fontsize=16, fontweight='bold', y=1.02)
    
    # Adoption distribution by segment
    ax = axes[0]
    seg_adoption = pd.crosstab(df['supplier_segment'], df['adoption_level'], normalize='index') * 100
    x = np.arange(len(segment_actual))
    for i, adoption in enumerate(adoption_order):
        values = [seg_adoption.loc[seg, adoption] if seg in seg_adoption.index and adoption in seg_adoption.columns else 0 for seg in segment_actual]
        values = [v if np.isfinite(v) else 0 for v in values]
        ax.bar(x + i * 0.18, values, 0.18, label=adoption, color=PALETTE_5[i], edgecolor='white', linewidth=1.5)
    ax.set_xlabel('Supplier Segment', fontweight='bold')
    ax.set_ylabel('Percentage (%)', fontweight='bold')
    ax.set_title('Adoption by Segment', fontweight='bold', pad=10)
    ax.set_xticks(x + 0.36)
    ax.set_xticklabels(segment_actual, rotation=45, ha='right')
    ax.legend(loc='upper right', fontsize=7)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.grid(axis='y', alpha=0.3, linestyle='--', linewidth=0.5)
    
    # Average feature count by segment
    ax = axes[1]
    avg_features_seg = df.groupby('supplier_segment')['advanced_features_count'].mean().reindex(segment_actual)
    avg_features_seg = avg_features_seg.fillna(0)
    colors_seg = ['#FF6B35', '#FF8C5A', '#FFAD7F', '#D9D9D9', '#B0B0B0', '#909090']
    bars = ax.bar(segment_actual, avg_features_seg, color=colors_seg[:len(segment_actual)], 
                  edgecolor='white', linewidth=2)
    ax.set_xlabel('Supplier Segment', fontweight='bold')
    ax.set_ylabel('Average Features Used', fontweight='bold')
    ax.set_title('Average Feature Count by Segment', fontweight='bold', pad=10)
    ax.set_xticklabels(segment_actual, rotation=45, ha='right')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.grid(axis='y', alpha=0.3, linestyle='--', linewidth=0.5)
    for i, val in enumerate(avg_features_seg):
        if not np.isnan(val) and np.isfinite(val):
            ax.text(i, val + 0.1, f'{val:.2f}', ha='center', fontweight='bold', color=COLORS['dark'])
    
    plt.tight_layout()
    display(fig)
    plt.close()
else:
    print("supplier_segment field not found in data or no segments available")

# COMMAND ----------

# MARKDOWN
"""
## Chart 7: Most Common Features by Supplier Segment

**What this analyzes:**
- Which specific features are most frequently used within each supplier segment
- Shows top 5 features for each segment from the data
- Reveals whether Enterprise suppliers use different features than SMB/Micro suppliers

**Business insights:**
- Do Enterprise suppliers leverage advanced API features more than smaller suppliers?
- Are smaller suppliers limited to basic features due to technical or resource constraints?
- Segment-specific feature patterns can guide account management priorities
"""

# COMMAND ----------

if 'supplier_segment' in df.columns and len(segment_actual) > 0:
    n_segments = len(segment_actual)
    n_rows = (n_segments + 1) // 2
    
    fig, axes = plt.subplots(n_rows, 2, figsize=(14, n_rows * 4))
    fig.suptitle('Most Common Features by Supplier Segment', fontsize=16, fontweight='bold', y=0.998)
    
    if n_rows == 1:
        axes = axes.reshape(1, -1)
    
    for idx, segment in enumerate(segment_actual):
        row = idx // 2
        col = idx % 2
        ax = axes[row, col]
        
        seg_df = df[df['supplier_segment'] == segment]
        
        feature_usage = {}
        for feat in feature_cols:
            if feat in seg_df.columns:
                adoption_rate = (seg_df[feat].sum() / len(seg_df)) * 100
                feature_usage[feature_display_names.get(feat, feat)] = adoption_rate
        
        # Sort and get top 5
        sorted_features = sorted(feature_usage.items(), key=lambda x: x[1], reverse=True)[:5]
        feature_names = [f[0] for f in sorted_features]
        feature_vals = [f[1] for f in sorted_features]
        
        colors_grad = ['#FF6B35', '#FF7A47', '#FF8C5A', '#FFAD7F', '#CCCCCC']
        bars = ax.barh(feature_names, feature_vals, color=colors_grad[:len(feature_names)], 
                       edgecolor='white', linewidth=2)
        ax.set_xlabel('Adoption Rate (%)', fontweight='bold')
        ax.set_title(f'{segment} (n={len(seg_df):,})', fontweight='bold', pad=10)
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        ax.set_xlim(0, 100)
        
        for i, val in enumerate(feature_vals):
            if not np.isnan(val) and np.isfinite(val):
                ax.text(val + 2, i, f'{val:.1f}%', va='center', fontweight='bold', color=COLORS['dark'])
    
    # Hide unused subplots
    for idx in range(n_segments, n_rows * 2):
        row = idx // 2
        col = idx % 2
        axes[row, col].axis('off')
    
    plt.tight_layout()
    display(fig)
    plt.close()
else:
    print("supplier_segment field not found in data or no segments available")

# COMMAND ----------

# MARKDOWN
"""
## Chart 8: Adoption by Activity Type (FROM DATA FIELD)

**What this analyzes:**
- Which activity types have highest feature adoption
- Average feature usage by activity category
- Uses the actual activity_type field from the data

**Business insights:**
- Some activity types may be more complex and require more pricing features (e.g., multi-day tours)
- Some categories may be under-utilizing features relative to their potential
- Categories sorted by average feature count to identify leaders and laggards

**Action items:**
- Low-adoption categories may need category-specific enablement materials
- High-adoption categories can be case studies for others
"""

# COMMAND ----------

if 'activity_type' in df.columns:
    fig, axes = plt.subplots(2, 1, figsize=(14, 10))
    fig.suptitle('Feature Adoption by Activity Type (from data)', fontsize=16, fontweight='bold', y=0.995)
    
    # Get top 15 activity types by count
    top_activities = df['activity_type'].value_counts().head(15).index.tolist()
    df_act = df[df['activity_type'].isin(top_activities)].copy()
    
    # Adoption distribution by activity
    ax = axes[0]
    act_adoption = pd.crosstab(df_act['activity_type'], df_act['adoption_level'], normalize='index') * 100
    cat_order = df_act.groupby('activity_type')['advanced_features_count'].mean().sort_values(ascending=True).index.tolist()
    act_adoption_sorted = act_adoption.reindex(cat_order)
    x = np.arange(len(cat_order))
    for i, adoption in enumerate(adoption_order):
        values = [act_adoption_sorted.loc[cat, adoption] if adoption in act_adoption_sorted.columns else 0 for cat in cat_order]
        values = [v if np.isfinite(v) else 0 for v in values]
        ax.barh(x + i * 0.18, values, 0.18, label=adoption, color=PALETTE_5[i], edgecolor='white', linewidth=1.5)
    ax.set_ylabel('Activity Type', fontweight='bold')
    ax.set_xlabel('Percentage (%)', fontweight='bold')
    ax.set_title('Adoption Within Each Activity Type (Top 15 by volume)', fontweight='bold', pad=10)
    ax.set_yticks(x + 0.36)
    ax.set_yticklabels(cat_order, fontsize=8)
    ax.legend(loc='lower right', fontsize=7, ncol=2)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.grid(axis='x', alpha=0.3, linestyle='--', linewidth=0.5)
    
    # Average feature count by activity
    ax = axes[1]
    avg_features_act = df_act.groupby('activity_type')['advanced_features_count'].mean().reindex(cat_order)
    avg_features_act = avg_features_act.fillna(0)
    bars = ax.barh(cat_order, avg_features_act, color='#FF6B35', edgecolor='white', linewidth=2)
    ax.set_ylabel('Activity Type', fontweight='bold')
    ax.set_xlabel('Average Features Used', fontweight='bold')
    ax.set_title('Average Feature Count by Activity Type', fontweight='bold', pad=10)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.grid(axis='x', alpha=0.3, linestyle='--', linewidth=0.5)
    
    plt.tight_layout()
    display(fig)
    plt.close()
else:
    print("activity_type field not found in data")

# COMMAND ----------

# MARKDOWN
"""
## Chart 9: Most Common Features by Activity Type (Top 8)

**What this analyzes:**
- Which specific features are most frequently used within each activity type
- Shows top 5 features for the top 8 activity types by volume
- Reveals category-specific feature preferences

**Business insights:**
- Do certain activity types naturally require specific features (e.g., Group Pricing for tours)?
- Are there features underutilized in specific categories where they could add value?
- Category-specific adoption patterns can inform product education tailored to activity type
"""

# COMMAND ----------

if 'activity_type' in df.columns:
    top_8_activities = df['activity_type'].value_counts().head(8).index.tolist()
    
    fig, axes = plt.subplots(4, 2, figsize=(16, 20))
    fig.suptitle('Most Common Features by Activity Type (Top 8)', fontsize=16, fontweight='bold', y=0.998)
    
    for idx, activity in enumerate(top_8_activities):
        row = idx // 2
        col = idx % 2
        ax = axes[row, col]
        
        act_df = df[df['activity_type'] == activity]
        
        feature_usage = {}
        for feat in feature_cols:
            if feat in act_df.columns:
                adoption_rate = (act_df[feat].sum() / len(act_df)) * 100
                feature_usage[feature_display_names.get(feat, feat)] = adoption_rate
        
        # Sort and get top 5
        sorted_features = sorted(feature_usage.items(), key=lambda x: x[1], reverse=True)[:5]
        feature_names = [f[0] for f in sorted_features]
        feature_vals = [f[1] for f in sorted_features]
        
        colors_grad = ['#FF6B35', '#FF7A47', '#FF8C5A', '#FFAD7F', '#CCCCCC']
        bars = ax.barh(feature_names, feature_vals, color=colors_grad[:len(feature_names)], 
                       edgecolor='white', linewidth=2)
        ax.set_xlabel('Adoption Rate (%)', fontweight='bold')
        ax.set_title(f'{activity} (n={len(act_df):,})', fontweight='bold', pad=10, fontsize=10)
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        ax.set_xlim(0, 100)
        
        for i, val in enumerate(feature_vals):
            if not np.isnan(val) and np.isfinite(val):
                ax.text(val + 2, i, f'{val:.1f}%', va='center', fontweight='bold', color=COLORS['dark'], fontsize=8)
    
    plt.tight_layout()
    display(fig)
    plt.close()
else:
    print("activity_type field not found in data")

# COMMAND ----------

# MARKDOWN
"""
## Chart 10: Cross-Segment Analysis - GMV Tier × Activity Type (Top 10)

**What this analyzes:**
- Average feature count across combinations of GMV tier and activity type
- Shows the top 10 activity types by volume
- Reveals how feature adoption varies when considering both GMV and category

**Why cross-segment matters:**
Are high-GMV tours in certain activity types adopting features faster than others? This can identify high-potential segments for targeted enablement.

**Heatmap interpretation:**
- Hot spots (dark orange/red) = segments with high feature adoption
- Cold spots (light colors) = opportunity segments
"""

# COMMAND ----------

if 'activity_type' in df.columns:
    top_10_activities = df['activity_type'].value_counts().head(10).index.tolist()
    df_cross = df[df['activity_type'].isin(top_10_activities)].copy()
    
    fig, ax = plt.subplots(1, 1, figsize=(12, 8))
    fig.suptitle('Cross-Segment: GMV Tier × Activity Type (Top 10)', fontsize=16, fontweight='bold', y=0.96)
    
    # Average features heatmap
    avg_feat_cross = pd.crosstab(df_cross['gmv_tier'], df_cross['activity_type'], 
                                 values=df_cross['advanced_features_count'], aggfunc='mean')
    avg_feat_cross = avg_feat_cross.reindex(index=gmv_tier_order, columns=top_10_activities).fillna(0)
    
    im = ax.imshow(avg_feat_cross.values, cmap='Oranges', aspect='auto', vmin=0, vmax=max(avg_feat_cross.values.max(), 0.1))
    ax.set_xticks(range(len(top_10_activities)))
    ax.set_xticklabels(top_10_activities, rotation=45, ha='right', fontsize=8)
    ax.set_yticks(range(len(gmv_tier_order)))
    ax.set_yticklabels(gmv_tier_order, fontsize=9)
    ax.set_xlabel('Activity Type', fontweight='bold')
    ax.set_ylabel('GMV Tier', fontweight='bold')
    ax.set_title('Average Features Used', fontweight='bold', pad=10)
    
    for i in range(len(gmv_tier_order)):
        for j in range(len(top_10_activities)):
            val = avg_feat_cross.values[i, j]
            if np.isfinite(val) and val > 0:
                ax.text(j, i, f'{val:.1f}', ha='center', va='center',
                       color='white' if val > avg_feat_cross.values.max()/2 else 'black', fontweight='bold', fontsize=7)
    
    cbar = plt.colorbar(im, ax=ax)
    cbar.set_label('Avg Features', rotation=270, labelpad=20, fontweight='bold')
    
    plt.tight_layout()
    display(fig)
    plt.close()
else:
    print("activity_type field not found in data")

# COMMAND ----------

# MARKDOWN
"""
## Chart 11: Cross-Segment Analysis - Supplier Segment × Activity Type (Top 10)

**What this analyzes:**
- Average feature count across combinations of supplier segment and activity type
- Shows the top 10 activity types by volume
- Reveals whether certain supplier segments excel in specific activity categories

**Business insights:**
Do Enterprise suppliers in certain categories adopt features faster than SMB suppliers in the same categories? This can validate or challenge assumptions about segment-specific support needs.
"""

# COMMAND ----------

if 'supplier_segment' in df.columns and 'activity_type' in df.columns and len(segment_actual) > 0:
    top_10_activities = df['activity_type'].value_counts().head(10).index.tolist()
    df_cross = df[df['activity_type'].isin(top_10_activities)].copy()
    
    fig, ax = plt.subplots(1, 1, figsize=(12, 6))
    fig.suptitle('Cross-Segment: Supplier Segment × Activity Type (Top 10)', fontsize=16, fontweight='bold', y=0.98)
    
    # Average features heatmap
    avg_feat_cross = pd.crosstab(df_cross['supplier_segment'], df_cross['activity_type'], 
                                 values=df_cross['advanced_features_count'], aggfunc='mean')
    avg_feat_cross = avg_feat_cross.reindex(index=segment_actual, columns=top_10_activities).fillna(0)
    
    im = ax.imshow(avg_feat_cross.values, cmap='Oranges', aspect='auto', vmin=0, vmax=max(avg_feat_cross.values.max(), 0.1))
    ax.set_xticks(range(len(top_10_activities)))
    ax.set_xticklabels(top_10_activities, rotation=45, ha='right', fontsize=8)
    ax.set_yticks(range(len(segment_actual)))
    ax.set_yticklabels(segment_actual, fontsize=9)
    ax.set_xlabel('Activity Type', fontweight='bold')
    ax.set_ylabel('Supplier Segment', fontweight='bold')
    ax.set_title('Average Features Used', fontweight='bold', pad=10)
    
    for i in range(len(segment_actual)):
        for j in range(len(top_10_activities)):
            val = avg_feat_cross.values[i, j]
            if np.isfinite(val) and val > 0:
                ax.text(j, i, f'{val:.1f}', ha='center', va='center',
                       color='white' if val > avg_feat_cross.values.max()/2 else 'black', fontweight='bold', fontsize=7)
    
    cbar = plt.colorbar(im, ax=ax)
    cbar.set_label('Avg Features', rotation=270, labelpad=20, fontweight='bold')
    
    plt.tight_layout()
    display(fig)
    plt.close()
else:
    print("Required fields not found in data")

# COMMAND ----------

# MARKDOWN
"""
## Chart 12: Cross-Segment Analysis - Market Maturity × Activity Type (Top 10)

**What this analyzes:**
- Average feature count across combinations of market maturity and activity type
- Shows the top 10 activity types by volume
- Reveals whether new suppliers in certain categories adopt features faster than mature suppliers in other categories

**Business insights:**
Are new suppliers in high-complexity categories (e.g., multi-day tours) adopting features faster than mature suppliers in simpler categories? This can inform category-specific onboarding strategies.
"""

# COMMAND ----------

if 'activity_type' in df.columns:
    top_10_activities = df['activity_type'].value_counts().head(10).index.tolist()
    df_cross = df[df['activity_type'].isin(top_10_activities)].copy()
    
    fig, ax = plt.subplots(1, 1, figsize=(12, 5))
    fig.suptitle('Cross-Segment: Market Maturity × Activity Type (Top 10)', fontsize=16, fontweight='bold', y=0.98)
    
    # Average features heatmap
    avg_feat_cross = pd.crosstab(df_cross['market_maturity'], df_cross['activity_type'], 
                                 values=df_cross['advanced_features_count'], aggfunc='mean')
    avg_feat_cross = avg_feat_cross.reindex(index=maturity_order, columns=top_10_activities).fillna(0)
    
    im = ax.imshow(avg_feat_cross.values, cmap='Oranges', aspect='auto', vmin=0, vmax=max(avg_feat_cross.values.max(), 0.1))
    ax.set_xticks(range(len(top_10_activities)))
    ax.set_xticklabels(top_10_activities, rotation=45, ha='right', fontsize=8)
    ax.set_yticks(range(len(maturity_order)))
    ax.set_yticklabels(['New (0-6m)', 'Developing (6-18m)', 'Mature (18m+)'], fontsize=9)
    ax.set_xlabel('Activity Type', fontweight='bold')
    ax.set_ylabel('Market Maturity', fontweight='bold')
    ax.set_title('Average Features Used', fontweight='bold', pad=10)
    
    for i in range(len(maturity_order)):
        for j in range(len(top_10_activities)):
            val = avg_feat_cross.values[i, j]
            if np.isfinite(val) and val > 0:
                ax.text(j, i, f'{val:.1f}', ha='center', va='center',
                       color='white' if val > avg_feat_cross.values.max()/2 else 'black', fontweight='bold', fontsize=7)
    
    cbar = plt.colorbar(im, ax=ax)
    cbar.set_label('Avg Features', rotation=270, labelpad=20, fontweight='bold')
    
    plt.tight_layout()
    display(fig)
    plt.close()
else:
    print("activity_type field not found in data")

# COMMAND ----------

# MARKDOWN
"""
## Chart 13: Cross-Segment Analysis - GMV Tier × Market Maturity

**What this analyzes:**
- Power user adoption rates across combinations of GMV tier and maturity
- Average feature usage across combinations

**Why cross-segment matters:**
Single-dimension analysis can hide important patterns. For example:
- Are NEW high-GMV tours adopting faster than MATURE low-GMV tours?
- Do high-GMV tours start strong or grow into features over time?

**Heatmap interpretation:**
- Hot spots (orange/red) = high adoption segments to use as case studies
- Cold spots (light) = opportunity segments for targeted enablement
"""

# COMMAND ----------

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Cross-Segment: GMV Tier × Market Maturity', fontsize=16, fontweight='bold', y=1.02)

# Power user adoption heatmap
ax = axes[0]
power_cross = pd.crosstab(df['gmv_tier'], df['market_maturity'], 
                          values=(df['adoption_level'] == 'Power (8+)').astype(int), aggfunc='mean') * 100
power_cross = power_cross.reindex(index=gmv_tier_order, columns=maturity_order).fillna(0)
im = ax.imshow(power_cross.values, cmap='Oranges', aspect='auto', vmin=0, vmax=max(power_cross.values.max(), 0.1))
ax.set_xticks(range(len(maturity_order)))
ax.set_xticklabels(['New\n(0-6m)', 'Developing\n(6-18m)', 'Mature\n(18m+)'], rotation=0)
ax.set_yticks(range(len(gmv_tier_order)))
ax.set_yticklabels(gmv_tier_order, fontsize=8)
ax.set_xlabel('Market Maturity', fontweight='bold')
ax.set_ylabel('GMV Tier', fontweight='bold')
ax.set_title('Power User Adoption Rate (%)', fontweight='bold', pad=10)
for i in range(len(gmv_tier_order)):
    for j in range(len(maturity_order)):
        val = power_cross.values[i, j]
        if np.isfinite(val):
            ax.text(j, i, f'{val:.1f}%', ha='center', va='center', 
                   color='white' if val > power_cross.values.max()/2 else 'black', fontweight='bold', fontsize=8)
cbar = plt.colorbar(im, ax=ax)
cbar.set_label('Adoption Rate (%)', rotation=270, labelpad=20, fontweight='bold')

# Average features heatmap
ax = axes[1]
avg_feat_cross = pd.crosstab(df['gmv_tier'], df['market_maturity'], 
                             values=df['advanced_features_count'], aggfunc='mean')
avg_feat_cross = avg_feat_cross.reindex(index=gmv_tier_order, columns=maturity_order).fillna(0)
im = ax.imshow(avg_feat_cross.values, cmap='Oranges', aspect='auto', vmin=0, vmax=max(avg_feat_cross.values.max(), 0.1))
ax.set_xticks(range(len(maturity_order)))
ax.set_xticklabels(['New\n(0-6m)', 'Developing\n(6-18m)', 'Mature\n(18m+)'], rotation=0)
ax.set_yticks(range(len(gmv_tier_order)))
ax.set_yticklabels(gmv_tier_order, fontsize=8)
ax.set_xlabel('Market Maturity', fontweight='bold')
ax.set_ylabel('GMV Tier', fontweight='bold')
ax.set_title('Average Features Used', fontweight='bold', pad=10)
for i in range(len(gmv_tier_order)):
    for j in range(len(maturity_order)):
        val = avg_feat_cross.values[i, j]
        if np.isfinite(val):
            ax.text(j, i, f'{val:.2f}', ha='center', va='center',
                   color='white' if val > avg_feat_cross.values.max()/2 else 'black', fontweight='bold', fontsize=8)
cbar = plt.colorbar(im, ax=ax)
cbar.set_label('Avg Features', rotation=270, labelpad=20, fontweight='bold')

plt.tight_layout()
display(fig)
plt.close()

# COMMAND ----------

# MARKDOWN
"""
## Chart 14: Adoption by Connection and Management Status

**What this analyzes:**
- API-connected vs non-connected suppliers (API integration enables advanced features)
- Managed vs non-managed suppliers (managed suppliers receive direct account support)

**Hypothesis:**
- API-connected suppliers should have higher adoption (API unlocks scaling features)
- Managed suppliers should have higher adoption (direct support drives feature usage)

**Business value:**
- Quantifies the impact of API connectivity on feature adoption
- Validates the ROI of supplier management programs
"""

# COMMAND ----------

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Adoption by Connection and Management Status', fontsize=16, fontweight='bold', y=0.995)

# Adoption by connection status
ax = axes[0, 0]
conn_adoption = pd.crosstab(df['connection_status'], df['adoption_level'], normalize='index') * 100
conn_order = ['Not Connected', 'Connected']
x = np.arange(len(conn_order))
for i, adoption in enumerate(adoption_order):
    values = [conn_adoption.loc[conn, adoption] if conn in conn_adoption.index and adoption in conn_adoption.columns else 0 for conn in conn_order]
    values = [v if np.isfinite(v) else 0 for v in values]
    ax.bar(x + i * 0.18, values, 0.18, label=adoption, color=PALETTE_5[i], edgecolor='white', linewidth=1.5)
ax.set_xlabel('Connection Status', fontweight='bold')
ax.set_ylabel('Percentage (%)', fontweight='bold')
ax.set_title('Adoption by Connection', fontweight='bold', pad=10)
ax.set_xticks(x + 0.36)
ax.set_xticklabels(conn_order, rotation=0)
ax.legend(loc='upper left', fontsize=6)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.grid(axis='y', alpha=0.3, linestyle='--', linewidth=0.5)

# Average features by connection
ax = axes[0, 1]
avg_feat_conn = df.groupby('connection_status')['advanced_features_count'].mean().reindex(conn_order)
avg_feat_conn = avg_feat_conn.fillna(0)
bars = ax.bar(conn_order, avg_feat_conn, color=['#FF6B35', '#FFAD7F'], edgecolor='white', linewidth=2)
ax.set_xlabel('Connection Status', fontweight='bold')


In [0]:
# COMMAND ----------

# MARKDOWN
"""
## Chart 14: Adoption by Connection and Management Status

**What this analyzes:**
- API-connected vs non-connected suppliers (API integration enables advanced features)
- Managed vs non-managed suppliers (managed suppliers receive direct account support)

**Hypothesis:**
- API-connected suppliers should have higher adoption (API unlocks scaling features)
- Managed suppliers should have higher adoption (direct support drives feature usage)

**Business value:**
- Quantifies the impact of API connectivity on feature adoption
- Validates the ROI of supplier management programs
"""

# COMMAND ----------

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Adoption by Connection and Management Status', fontsize=16, fontweight='bold', y=0.995)

# Adoption by connection status
ax = axes[0, 0]
conn_adoption = pd.crosstab(df['connection_status'], df['adoption_level'], normalize='index') * 100
conn_order = ['Not Connected', 'Connected']
x = np.arange(len(conn_order))
for i, adoption in enumerate(adoption_order):
    values = [conn_adoption.loc[conn, adoption] if conn in conn_adoption.index and adoption in conn_adoption.columns else 0 for conn in conn_order]
    values = [v if np.isfinite(v) else 0 for v in values]
    ax.bar(x + i * 0.18, values, 0.18, label=adoption, color=PALETTE_5[i], edgecolor='white', linewidth=1.5)
ax.set_xlabel('Connection Status', fontweight='bold')
ax.set_ylabel('Percentage (%)', fontweight='bold')
ax.set_title('Adoption by Connection', fontweight='bold', pad=10)
ax.set_xticks(x + 0.36)
ax.set_xticklabels(conn_order, rotation=0)
ax.legend(loc='upper left', fontsize=6)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.grid(axis='y', alpha=0.3, linestyle='--', linewidth=0.5)

# Average features by connection
ax = axes[0, 1]
avg_feat_conn = df.groupby('connection_status')['advanced_features_count'].mean().reindex(conn_order)
avg_feat_conn = avg_feat_conn.fillna(0)
bars = ax.bar(conn_order, avg_feat_conn, color=['#FF6B35', '#FFAD7F'], edgecolor='white', linewidth=2)
ax.set_xlabel('Connection Status', fontweight='bold')
ax.set_ylabel('Average Features Used', fontweight='bold')
ax.set_title('Features by Connection', fontweight='bold', pad=10)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.grid(axis='y', alpha=0.3, linestyle='--', linewidth=0.5)
for i, val in enumerate(avg_feat_conn):
    if not np.isnan(val) and np.isfinite(val):
        ax.text(i, val + 0.1, f'{val:.2f}', ha='center', fontweight='bold', color=COLORS['dark'])

# Adoption by managed status
ax = axes[1, 0]
mgd_adoption = pd.crosstab(df['managed_status'], df['adoption_level'], normalize='index') * 100
mgd_order = ['Not Managed', 'Managed']
x = np.arange(len(mgd_order))
for i, adoption in enumerate(adoption_order):
    values = [mgd_adoption.loc[mgd, adoption] if mgd in mgd_adoption.index and adoption in mgd_adoption.columns else 0 for mgd in mgd_order]
    values = [v if np.isfinite(v) else 0 for v in values]
    ax.bar(x + i * 0.18, values, 0.18, label=adoption, color=PALETTE_5[i], edgecolor='white', linewidth=1.5)
ax.set_xlabel('Management Status', fontweight='bold')
ax.set_ylabel('Percentage (%)', fontweight='bold')
ax.set_title('Adoption by Managed Status', fontweight='bold', pad=10)
ax.set_xticks(x + 0.36)
ax.set_xticklabels(mgd_order, rotation=0)
ax.legend(loc='upper left', fontsize=6)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.grid(axis='y', alpha=0.3, linestyle='--', linewidth=0.5)

# Average features by managed status
ax = axes[1, 1]
avg_feat_mgd = df.groupby('managed_status')['advanced_features_count'].mean().reindex(mgd_order)
avg_feat_mgd = avg_feat_mgd.fillna(0)
bars = ax.bar(mgd_order, avg_feat_mgd, color=['#FF6B35', '#FFAD7F'], edgecolor='white', linewidth=2)
ax.set_xlabel('Management Status', fontweight='bold')
ax.set_ylabel('Average Features Used', fontweight='bold')
ax.set_title('Features by Managed Status', fontweight='bold', pad=10)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.grid(axis='y', alpha=0.3, linestyle='--', linewidth=0.5)
for i, val in enumerate(avg_feat_mgd):
    if not np.isnan(val) and np.isfinite(val):
        ax.text(i, val + 0.1, f'{val:.2f}', ha='center', fontweight='bold', color=COLORS['dark'])

plt.tight_layout()
display(fig)
plt.close()

# COMMAND ----------

# MARKDOWN
"""
## Summary Statistics and Key Takeaways

**What this provides:**
- Overall adoption rate (percentage using at least one feature)
- Power user rate (percentage using 8+ features)
- Average features per tour
- Distribution summary
- GMV tier ranges derived from data

**Use these metrics to:**
- Track adoption improvement over time (run this monthly)
- Set targets for product and enablement teams
- Communicate impact to leadership
"""

# COMMAND ----------

# Calculate summary metrics
adopters = (df['adoption_level'] != 'Non-adopters').sum()
adoption_rate = (adopters / len(df)) * 100
power_users = (df['adoption_level'] == 'Power (8+)').sum()
power_rate = (power_users / len(df)) * 100
avg_feat = df['advanced_features_count'].mean()
median_feat = df['advanced_features_count'].median()

print("="*60)
print("PRICING FEATURE ADOPTION SUMMARY")
print("="*60)
print(f"\nTotal tours analyzed: {len(df):,}")
print(f"Total suppliers: {df['supplier_id'].nunique():,}")
print(f"Features tracked: {len(feature_cols)}")
print(f"\nGMV TIERS (Percentile-Based):")
print(f"  Top 5%: GMV ≥ {gmv_percentiles[0.95]:,.0f}")
print(f"  Top 25%: GMV ≥ {gmv_percentiles[0.75]:,.0f}")
print(f"  Top 50%: GMV ≥ {gmv_percentiles[0.50]:,.0f}")
print(f"  Bottom 50%: GMV < {gmv_percentiles[0.50]:,.0f}")
print(f"\nOVERALL ADOPTION METRICS:")
print(f"  • Overall adoption rate: {adoption_rate:.1f}%")
print(f"  • Non-adopters: {100-adoption_rate:.1f}%")
print(f"  • Power users (8+ features): {power_rate:.1f}%")
print(f"  • Average features per tour: {avg_feat:.2f}")
print(f"  • Median features per tour: {median_feat:.0f}")
print(f"\nFEATURES ANALYZED:")
for i, feat in enumerate(feature_cols, 1):
    print(f"  {i}. {feat}")
print("="*60)

# Show adoption breakdown
print("\nADOPTION LEVEL BREAKDOWN:")
adoption_breakdown = df['adoption_level'].value_counts().reindex(adoption_order)
for level in adoption_order:
    count = adoption_breakdown.get(level, 0)
    pct = (count / len(df)) * 100
    print(f"  {level:20s}: {count:6,} tours ({pct:5.1f}%)")

print("\n" + "="*60)


In [0]:
# Databricks notebook source

# MARKDOWN
"""
# Pricing Feature Adoption Audit

## Objective
This notebook analyzes pricing feature adoption across GetYourGuide's supplier base to identify which segments are utilizing advanced pricing capabilities and which segments represent opportunities for feature education and adoption.

## Business Questions
1. What percentage of tours use advanced pricing features?
2. Which supplier segments (by volume, maturity, category) have highest/lowest adoption?
3. Which specific features are most used within each segment?
4. What opportunities exist to increase feature penetration?

## Features Analyzed (10 Total)
- Individual Pricing
- Group Pricing  
- Pricing without API (Native)
- Pricing with API
- Scales without API (Native Scales)
- Scales with API
- Live Pricing and Availability
- Non-live Price and Availability
- Add-ons
- Options
"""

# COMMAND ----------

# MARKDOWN
"""
## Section 1: Data Loading and Type Conversion

**What this does:** 
- Loads the pricing feature data from the Spark table
- Converts all columns to proper numeric types to avoid calculation errors
- Handles missing values by filling with 0
- Converts dates to datetime format

**Why it matters:**
Data type issues cause errors in aggregations and binning. This ensures clean numeric data for all calculations.
"""

# COMMAND ----------

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import rcParams
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Load data
print("Loading data from Spark...")
spark_df = spark.table("production.supply_analytics.pricing_feature_combined")
df = spark_df.toPandas()

print(f"Loaded {len(df):,} tours from {df['supplier_id'].nunique():,} suppliers")

# Fix data types - critical for avoiding errors
numeric_cols = ['gmv_l12mo', 'bookings_l12mo', 'has_group_pricing', 'has_individual_pricing',
                'has_addons', 'has_options', 'has_native_scales', 'has_api_scales', 
                'has_live_dynamic_pricing', 'has_non_live_pricing', 'uses_api_pricing',
                'has_scale_pricing', 'is_managed', 'num_individual_categories']

for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)

if 'date_of_first_online' in df.columns:
    df['date_of_first_online'] = pd.to_datetime(df['date_of_first_online'], errors='coerce')

# Ensure supplier_segment and activity_type are strings
if 'supplier_segment' in df.columns:
    df['supplier_segment'] = df['supplier_segment'].fillna('Unknown').astype(str)
if 'activity_type' in df.columns:
    df['activity_type'] = df['activity_type'].fillna('Unknown').astype(str)

print("✓ Data types converted")
print("\nChecking key fields:")
print(f"  - supplier_segment field exists: {'supplier_segment' in df.columns}")
print(f"  - activity_type field exists: {'activity_type' in df.columns}")

if 'supplier_segment' in df.columns:
    print(f"\nSupplier segments in data: {df['supplier_segment'].unique()}")
if 'activity_type' in df.columns:
    print(f"\nActivity types in data (first 10): {df['activity_type'].unique()[:10]}")

display(df.head())

# COMMAND ----------

# MARKDOWN
"""
## Section 2: Segmentation Creation

**What this does:**
- Creates GMV tiers based on percentile rankings: Top 5%, Top 25%, Top 50%, Bottom 50%
- Creates Market Maturity segments based on time on platform (New 0-6mo, Developing 6-18mo, Mature 18mo+)
- Uses the actual supplier_segment field from the data
- Uses the actual activity_type field from the data
- Identifies API connection status and managed supplier status from flags

**Why it matters:**
Percentile-based GMV tiers identify high-value suppliers who drive the most revenue. Different supplier segments have different adoption patterns.
"""

# COMMAND ----------

# GMV tier - percentile-based segmentation
gmv_percentiles = df['gmv_l12mo'].quantile([0, 0.50, 0.75, 0.95, 1.0])
print("GMV Distribution Percentiles:")
print(gmv_percentiles)

# Create percentile-based tiers
df['gmv_tier'] = 'Bottom 50%'
df.loc[df['gmv_l12mo'] >= gmv_percentiles[0.50], 'gmv_tier'] = 'Top 50%'
df.loc[df['gmv_l12mo'] >= gmv_percentiles[0.75], 'gmv_tier'] = 'Top 25%'
df.loc[df['gmv_l12mo'] >= gmv_percentiles[0.95], 'gmv_tier'] = 'Top 5%'

# Define tier order
gmv_tier_order = ['Top 5%', 'Top 25%', 'Top 50%', 'Bottom 50%']

print(f"\nGMV Tiers created:")
print(f"  Top 5%: GMV ≥ {gmv_percentiles[0.95]:,.0f}")
print(f"  Top 25%: GMV ≥ {gmv_percentiles[0.75]:,.0f}")
print(f"  Top 50%: GMV ≥ {gmv_percentiles[0.50]:,.0f}")
print(f"  Bottom 50%: GMV < {gmv_percentiles[0.50]:,.0f}")

for tier in gmv_tier_order:
    count = (df['gmv_tier'] == tier).sum()
    pct = count / len(df) * 100
    avg_gmv = df[df['gmv_tier'] == tier]['gmv_l12mo'].mean()
    print(f"    {tier}: {count:,} tours ({pct:.1f}%), Avg GMV: {avg_gmv:,.0f}")

# Market maturity - time-based segmentation
df['days_on_platform'] = (datetime.now() - df['date_of_first_online']).dt.days
df['market_maturity'] = 'Mature (18+ months)'
df.loc[df['days_on_platform'] <= 540, 'market_maturity'] = 'Developing (6-18 months)'
df.loc[df['days_on_platform'] <= 180, 'market_maturity'] = 'New (0-6 months)'

# Connection and management status from flags
df['connection_status'] = 'Not Connected'
df.loc[df['uses_api_pricing'] == 1, 'connection_status'] = 'Connected'

df['managed_status'] = 'Not Managed'
df.loc[df['is_managed'] == 1, 'managed_status'] = 'Managed'

# Verify segment distribution
print("\n✓ Segments created\n")
print("Market Maturity Distribution:")
display(df['market_maturity'].value_counts())
print("\nConnection Status Distribution:")
display(df['connection_status'].value_counts())
print("\nManaged Status Distribution:")
display(df['managed_status'].value_counts())
print("\nSupplier Segment Distribution (from data):")
if 'supplier_segment' in df.columns:
    display(df['supplier_segment'].value_counts())
print("\nActivity Type Distribution (from data - top 15):")
if 'activity_type' in df.columns:
    display(df['activity_type'].value_counts().head(15))

# COMMAND ----------

# MARKDOWN
"""
## Section 3: Feature Count Calculation and Adoption Level Assignment

**What this does:**
- Counts how many of the 10 pricing features each tour uses
- Creates 4 adoption levels based on realistic thresholds: Non-adopters (0-1), Low (2-3), Medium (4-5), High (6+)

**Why it matters:**
This is the core metric for measuring feature adoption maturity. Tours using more features likely generate more revenue per customer through upselling and dynamic optimization.
"""

# COMMAND ----------

# Build feature list dynamically based on available columns
feature_cols = []
if 'has_individual_pricing' in df.columns:
    feature_cols.append('has_individual_pricing')
if 'has_group_pricing' in df.columns:
    feature_cols.append('has_group_pricing')
if 'uses_api_pricing' in df.columns:
    feature_cols.append('uses_api_pricing')
if 'has_native_scales' in df.columns:
    feature_cols.append('has_native_scales')
if 'has_api_scales' in df.columns:
    feature_cols.append('has_api_scales')
if 'has_live_dynamic_pricing' in df.columns:
    feature_cols.append('has_live_dynamic_pricing')
if 'has_non_live_pricing' in df.columns:
    feature_cols.append('has_non_live_pricing')
if 'has_addons' in df.columns:
    feature_cols.append('has_addons')
if 'has_options' in df.columns:
    feature_cols.append('has_options')

# Calculate native pricing as inverse of API pricing
if 'uses_api_pricing' in df.columns:
    df['has_native_pricing'] = (1 - df['uses_api_pricing']).clip(0, 1)
    feature_cols.append('has_native_pricing')

# Sum features per tour
df['advanced_features_count'] = df[feature_cols].sum(axis=1)

print(f"Counting {len(feature_cols)} features: {feature_cols}")
print(f"Max features used by any tour: {df['advanced_features_count'].max()}")
print(f"Mean features per tour: {df['advanced_features_count'].mean():.2f}")
print(f"Median features per tour: {df['advanced_features_count'].median():.0f}")

# Create realistic adoption levels
df['adoption_level'] = 'High (6+)'
df.loc[df['advanced_features_count'] <= 5, 'adoption_level'] = 'Medium (4-5)'
df.loc[df['advanced_features_count'] <= 3, 'adoption_level'] = 'Low (2-3)'
df.loc[df['advanced_features_count'] <= 1, 'adoption_level'] = 'Non-adopters (0-1)'

df['uses_multiple_categories'] = (df['num_individual_categories'] >= 3).astype(int)

# Display distribution
print("\nAdoption Level Distribution:")
display(df['adoption_level'].value_counts())

# COMMAND ----------

# MARKDOWN
"""
## Section 4: Visualization Setup
"""

# COMMAND ----------

# Styling
rcParams['font.family'] = 'sans-serif'
rcParams['font.sans-serif'] = ['Arial', 'DejaVu Sans']
rcParams['font.size'] = 10

COLORS = {'primary': '#FF6B35', 'secondary': '#F77F51', 'tertiary': '#FFA07A', 
          'quaternary': '#C0C0C0', 'quinary': '#808080', 'dark': '#404040'}
PALETTE_4 = ['#FF6B35', '#FF8C5A', '#FFAD7F', '#CCCCCC']

# Define consistent orderings
maturity_order = ['New (0-6 months)', 'Developing (6-18 months)', 'Mature (18+ months)']
adoption_order = ['Non-adopters (0-1)', 'Low (2-3)', 'Medium (4-5)', 'High (6+)']

# Get actual segment order from data
if 'supplier_segment' in df.columns:
    segment_actual = df['supplier_segment'].value_counts().index.tolist()
    print(f"Supplier segments from data: {segment_actual}")
else:
    segment_actual = []

print("✓ Styling configured")

# COMMAND ----------

# MARKDOWN
"""
## Chart 1: Overall Feature Adoption Overview
"""

# COMMAND ----------

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Pricing Feature Adoption Overview', fontsize=16, fontweight='bold', y=0.995)

# Adoption level distribution
ax = axes[0, 0]
adoption_pcts = (df['adoption_level'].value_counts().reindex(adoption_order) / len(df) * 100)
adoption_pcts = adoption_pcts.fillna(0).values
bars = ax.barh(adoption_order, adoption_pcts, color=PALETTE_4, edgecolor='white', linewidth=2)
ax.set_xlabel('Percentage of Tours', fontweight='bold')
ax.set_title(f'Feature Adoption Levels ({len(feature_cols)} Features)', fontweight='bold', pad=10)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
for i, val in enumerate(adoption_pcts):
    if not np.isnan(val) and np.isfinite(val):
        ax.text(val + 1, i, f'{val:.1f}%', va='center', fontweight='bold', color=COLORS['dark'])

# Individual feature adoption
ax = axes[0, 1]
features = {}
if 'has_group_pricing' in df.columns:
    features['Group Pricing'] = df['has_group_pricing'].sum()
if 'has_individual_pricing' in df.columns:
    features['Individual Pricing'] = df['has_individual_pricing'].sum()
if 'has_addons' in df.columns:
    features['Add-ons'] = df['has_addons'].sum()
if 'has_options' in df.columns:
    features['Options'] = df['has_options'].sum()
if 'uses_api_pricing' in df.columns:
    features['API Pricing'] = df['uses_api_pricing'].sum()
if 'has_native_scales' in df.columns:
    features['Native Scales'] = df['has_native_scales'].sum()
if 'has_api_scales' in df.columns:
    features['API Scales'] = df['has_api_scales'].sum()
if 'has_live_dynamic_pricing' in df.columns:
    features['Live Pricing'] = df['has_live_dynamic_pricing'].sum()

feature_pcts = [(v / len(df)) * 100 for v in features.values()]
colors_bar = ['#FF6B35', '#FF7A47', '#FF8C5A', '#FF9E6C', '#FFAD7F', '#FFBC91', '#FFCBA4', '#D9D9D9']
bars = ax.barh(list(features.keys()), feature_pcts, color=colors_bar[:len(features)], 
               edgecolor='white', linewidth=2)
ax.set_xlabel('Adoption Rate (%)', fontweight='bold')
ax.set_title('Individual Feature Adoption', fontweight='bold', pad=10)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
for i, val in enumerate(feature_pcts):
    if not np.isnan(val) and np.isfinite(val):
        ax.text(val + 1, i, f'{val:.1f}%', va='center', fontweight='bold', color=COLORS['dark'])

# Category complexity
ax = axes[1, 0]
cat_dist = {'Adult Only': (df['num_individual_categories'] == 1).sum(),
            'Adult + Child': (df['num_individual_categories'] == 2).sum(),
            '3 Categories': (df['num_individual_categories'] == 3).sum(),
            '4+ Categories': (df['num_individual_categories'] >= 4).sum()}
cat_pcts = [(v / len(df)) * 100 for v in cat_dist.values()]
bars = ax.bar(range(len(cat_pcts)), cat_pcts, color=PALETTE_4[:4], edgecolor='white', linewidth=2)
ax.set_xticks(range(len(cat_dist)))
ax.set_xticklabels(cat_dist.keys(), rotation=0)
ax.set_ylabel('Percentage of Tours', fontweight='bold')
ax.set_title('Ticket Category Complexity', fontweight='bold', pad=10)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
for i, val in enumerate(cat_pcts):
    if not np.isnan(val) and np.isfinite(val):
        ax.text(i, val + 1, f'{val:.1f}%', ha='center', fontweight='bold', color=COLORS['dark'])

# Feature count distribution
ax = axes[1, 1]
feature_dist = df['advanced_features_count'].value_counts().sort_index()
ax.bar(feature_dist.index, feature_dist.values, color='#FF6B35', edgecolor='white', linewidth=2)
ax.set_xlabel('Number of Features Used', fontweight='bold')
ax.set_ylabel('Number of Tours', fontweight='bold')
ax.set_title('Distribution of Feature Count', fontweight='bold', pad=10)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
display(fig)
plt.close()

# COMMAND ----------

# MARKDOWN
"""
## Chart 2: Adoption by GMV Tier (Top 5%, Top 25%, Top 50%, Bottom 50%)
"""

# COMMAND ----------

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Feature Adoption by GMV Tier (Top 5%, Top 25%, Top 50%, Bottom 50%)', fontsize=16, fontweight='bold', y=1.02)

# Adoption distribution by GMV tier
ax = axes[0]
gmv_adoption = pd.crosstab(df['gmv_tier'], df['adoption_level'], normalize='index') * 100
x = np.arange(len(gmv_tier_order))
width = 0.22
for i, adoption in enumerate(adoption_order):
    values = [gmv_adoption.loc[tier, adoption] if tier in gmv_adoption.index and adoption in gmv_adoption.columns else 0 for tier in gmv_tier_order]
    values = [v if np.isfinite(v) else 0 for v in values]
    ax.bar(x + i * width, values, width, label=adoption, color=PALETTE_4[i], edgecolor='white', linewidth=1.5)
ax.set_xlabel('GMV Tier', fontweight='bold')
ax.set_ylabel('Percentage (%)', fontweight='bold')
ax.set_title('Adoption Distribution Within GMV Tier', fontweight='bold', pad=10)
ax.set_xticks(x + width * 1.5)
ax.set_xticklabels(gmv_tier_order, rotation=0, fontsize=9)
ax.legend(loc='upper right', fontsize=7)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.grid(axis='y', alpha=0.3, linestyle='--', linewidth=0.5)

# Average feature count by GMV tier
ax = axes[1]
avg_features_gmv = df.groupby('gmv_tier')['advanced_features_count'].mean().reindex(gmv_tier_order)
avg_features_gmv = avg_features_gmv.fillna(0)
bars = ax.bar(range(len(gmv_tier_order)), avg_features_gmv, color=['#D32F2F', '#FF6B35', '#FF8C5A', '#CCCCCC'], 
              edgecolor='white', linewidth=2)
ax.set_xlabel('GMV Tier', fontweight='bold')
ax.set_ylabel('Average Features Used', fontweight='bold')
ax.set_title('Average Feature Count by GMV Tier', fontweight='bold', pad=10)
ax.set_xticks(range(len(gmv_tier_order)))
ax.set_xticklabels(gmv_tier_order, rotation=0, fontsize=9)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.grid(axis='y', alpha=0.3, linestyle='--', linewidth=0.5)
for i, val in enumerate(avg_features_gmv):
    if not np.isnan(val) and np.isfinite(val):
        ax.text(i, val + 0.1, f'{val:.2f}', ha='center', fontweight='bold', color=COLORS['dark'])

plt.tight_layout()
display(fig)
plt.close()

# COMMAND ----------

# MARKDOWN
"""
## Chart 3: Most Common Features by GMV Tier
"""

# COMMAND ----------

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Most Common Features by GMV Tier', fontsize=16, fontweight='bold', y=0.995)

feature_display_names = {
    'has_individual_pricing': 'Individual Pricing',
    'has_group_pricing': 'Group Pricing',
    'uses_api_pricing': 'API Pricing',
    'has_native_pricing': 'Native Pricing',
    'has_native_scales': 'Native Scales',
    'has_api_scales': 'API Scales',
    'has_live_dynamic_pricing': 'Live Pricing',
    'has_non_live_pricing': 'Non-live Pricing',
    'has_addons': 'Add-ons',
    'has_options': 'Options'
}

for idx, tier in enumerate(gmv_tier_order):
    ax = axes[idx // 2, idx % 2]
    tier_df = df[df['gmv_tier'] == tier]
    
    feature_usage = {}
    for feat in feature_cols:
        if feat in tier_df.columns:
            adoption_rate = (tier_df[feat].sum() / len(tier_df)) * 100
            feature_usage[feature_display_names.get(feat, feat)] = adoption_rate
    
    # Sort and get top 5
    sorted_features = sorted(feature_usage.items(), key=lambda x: x[1], reverse=True)[:5]
    feature_names = [f[0] for f in sorted_features]
    feature_vals = [f[1] for f in sorted_features]
    
    colors_grad = ['#D32F2F', '#FF6B35', '#FF8C5A', '#FFAD7F', '#CCCCCC']
    bars = ax.barh(feature_names, feature_vals, color=colors_grad[:len(feature_names)], 
                   edgecolor='white', linewidth=2)
    ax.set_xlabel('Adoption Rate (%)', fontweight='bold')
    ax.set_title(f'{tier} (n={len(tier_df):,})', fontweight='bold', pad=10)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.set_xlim(0, 100)
    
    for i, val in enumerate(feature_vals):
        if not np.isnan(val) and np.isfinite(val):
            ax.text(val + 2, i, f'{val:.1f}%', va='center', fontweight='bold', color=COLORS['dark'])

plt.tight_layout()
display(fig)
plt.close()

# COMMAND ----------

# MARKDOWN
"""
## Chart 4: Adoption by Market Maturity
"""

# COMMAND ----------

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Feature Adoption by Market Maturity', fontsize=16, fontweight='bold', y=1.02)

# Adoption distribution by maturity
ax = axes[0]
mat_adoption = pd.crosstab(df['market_maturity'], df['adoption_level'], normalize='index') * 100
x = np.arange(len(maturity_order))
for i, adoption in enumerate(adoption_order):
    values = [mat_adoption.loc[mat, adoption] if mat in mat_adoption.index and adoption in mat_adoption.columns else 0 for mat in maturity_order]
    values = [v if np.isfinite(v) else 0 for v in values]
    ax.bar(x + i * 0.22, values, 0.22, label=adoption, color=PALETTE_4[i], edgecolor='white', linewidth=1.5)
ax.set_xlabel('Market Maturity', fontweight='bold')
ax.set_ylabel('Percentage (%)', fontweight='bold')
ax.set_title('Adoption Distribution by Maturity', fontweight='bold', pad=10)
ax.set_xticks(x + 0.33)
ax.set_xticklabels(['New\n(0-6m)', 'Developing\n(6-18m)', 'Mature\n(18m+)'], rotation=0)
ax.legend(loc='upper left', fontsize=7)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.grid(axis='y', alpha=0.3, linestyle='--', linewidth=0.5)

# Average feature count by maturity
ax = axes[1]
avg_features_mat = df.groupby('market_maturity')['advanced_features_count'].mean().reindex(maturity_order)
avg_features_mat = avg_features_mat.fillna(0)
bars = ax.bar(range(len(maturity_order)), avg_features_mat, color=['#FF6B35', '#FF8C5A', '#FFAD7F'], 
              edgecolor='white', linewidth=2)
ax.set_xlabel('Market Maturity', fontweight='bold')
ax.set_ylabel('Average Features Used', fontweight='bold')
ax.set_title('Average Feature Count by Maturity', fontweight='bold', pad=10)
ax.set_xticks(range(len(maturity_order)))
ax.set_xticklabels(['New\n(0-6m)', 'Developing\n(6-18m)', 'Mature\n(18m+)'], rotation=0)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.grid(axis='y', alpha=0.3, linestyle='--', linewidth=0.5)
for i, val in enumerate(avg_features_mat):
    if not np.isnan(val) and np.isfinite(val):
        ax.text(i, val + 0.1, f'{val:.2f}', ha='center', fontweight='bold', color=COLORS['dark'])

plt.tight_layout()
display(fig)
plt.close()

# COMMAND ----------

# MARKDOWN
"""
## Chart 5: Most Common Features by Market Maturity
"""

# COMMAND ----------

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Most Common Features by Market Maturity', fontsize=16, fontweight='bold', y=1.02)

for idx, maturity in enumerate(maturity_order):
    ax = axes[idx]
    mat_df = df[df['market_maturity'] == maturity]
    
    feature_usage = {}
    for feat in feature_cols:
        if feat in mat_df.columns:
            adoption_rate = (mat_df[feat].sum() / len(mat_df)) * 100
            feature_usage[feature_display_names.get(feat, feat)] = adoption_rate
    
    # Sort and get top 5
    sorted_features = sorted(feature_usage.items(), key=lambda x: x[1], reverse=True)[:5]
    feature_names = [f[0] for f in sorted_features]
    feature_vals = [f[1] for f in sorted_features]
    
    colors_grad = ['#FF6B35', '#FF7A47', '#FF8C5A', '#FFAD7F', '#CCCCCC']
    bars = ax.barh(feature_names, feature_vals, color=colors_grad[:len(feature_names)], 
                   edgecolor='white', linewidth=2)
    ax.set_xlabel('Adoption Rate (%)', fontweight='bold')
    ax.set_title(f'{maturity}\n(n={len(mat_df):,})', fontweight='bold', pad=10)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.set_xlim(0, 100)
    
    for i, val in enumerate(feature_vals):
        if not np.isnan(val) and np.isfinite(val):
            ax.text(val + 2, i, f'{val:.1f}%', va='center', fontweight='bold', color=COLORS['dark'])

plt.tight_layout()
display(fig)
plt.close()

# COMMAND ----------

# MARKDOWN
"""
## Chart 6: Adoption by Supplier Segment
"""

# COMMAND ----------

if 'supplier_segment' in df.columns and len(segment_actual) > 0:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle('Feature Adoption by Supplier Segment (from data)', fontsize=16, fontweight='bold', y=1.02)
    
    # Adoption distribution by segment
    ax = axes[0]
    seg_adoption = pd.crosstab(df['supplier_segment'], df['adoption_level'], normalize='index') * 100
    x = np.arange(len(segment_actual))
    for i, adoption in enumerate(adoption_order):
        values = [seg_adoption.loc[seg, adoption] if seg in seg_adoption.index and adoption in seg_adoption.columns else 0 for seg in segment_actual]
        values = [v if np.isfinite(v) else 0 for v in values]
        ax.bar(x + i * 0.22, values, 0.22, label=adoption, color=PALETTE_4[i], edgecolor='white', linewidth=1.5)
    ax.set_xlabel('Supplier Segment', fontweight='bold')
    ax.set_ylabel('Percentage (%)', fontweight='bold')
    ax.set_title('Adoption by Segment', fontweight='bold', pad=10)
    ax.set_xticks(x + 0.33)
    ax.set_xticklabels(segment_actual, rotation=45, ha='right')
    ax.legend(loc='upper right', fontsize=7)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.grid(axis='y', alpha=0.3, linestyle='--', linewidth=0.5)
    
    # Average feature count by segment
    ax = axes[1]
    avg_features_seg = df.groupby('supplier_segment')['advanced_features_count'].mean().reindex(segment_actual)
    avg_features_seg = avg_features_seg.fillna(0)
    colors_seg = ['#FF6B35', '#FF8C5A', '#FFAD7F', '#D9D9D9', '#B0B0B0', '#909090']
    bars = ax.bar(segment_actual, avg_features_seg, color=colors_seg[:len(segment_actual)], 
                  edgecolor='white', linewidth=2)
    ax.set_xlabel('Supplier Segment', fontweight='bold')
    ax.set_ylabel('Average Features Used', fontweight='bold')
    ax.set_title('Average Feature Count by Segment', fontweight='bold', pad=10)
    ax.set_xticklabels(segment_actual, rotation=45, ha='right')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.grid(axis='y', alpha=0.3, linestyle='--', linewidth=0.5)
    for i, val in enumerate(avg_features_seg):
        if not np.isnan(val) and np.isfinite(val):
            ax.text(i, val + 0.1, f'{val:.2f}', ha='center', fontweight='bold', color=COLORS['dark'])
    
    plt.tight_layout()
    display(fig)
    plt.close()
else:
    print("supplier_segment field not found in data or no segments available")

# COMMAND ----------

# MARKDOWN
"""
## Chart 7: Most Common Features by Supplier Segment
"""

# COMMAND ----------

if 'supplier_segment' in df.columns and len(segment_actual) > 0:
    n_segments = len(segment_actual)
    n_rows = (n_segments + 1) // 2
    
    fig, axes = plt.subplots(n_rows, 2, figsize=(14, n_rows * 4))
    fig.suptitle('Most Common Features by Supplier Segment', fontsize=16, fontweight='bold', y=0.998)
    
    if n_rows == 1:
        axes = axes.reshape(1, -1)
    
    for idx, segment in enumerate(segment_actual):
        row = idx // 2
        col = idx % 2
        ax = axes[row, col]
        
        seg_df = df[df['supplier_segment'] == segment]
        
        feature_usage = {}
        for feat in feature_cols:
            if feat in seg_df.columns:
                adoption_rate = (seg_df[feat].sum() / len(seg_df)) * 100
                feature_usage[feature_display_names.get(feat, feat)] = adoption_rate
        
        # Sort and get top 5
        sorted_features = sorted(feature_usage.items(), key=lambda x: x[1], reverse=True)[:5]
        feature_names = [f[0] for f in sorted_features]
        feature_vals = [f[1] for f in sorted_features]
        
        colors_grad = ['#FF6B35', '#FF7A47', '#FF8C5A', '#FFAD7F', '#CCCCCC']
        bars = ax.barh(feature_names, feature_vals, color=colors_grad[:len(feature_names)], 
                       edgecolor='white', linewidth=2)
        ax.set_xlabel('Adoption Rate (%)', fontweight='bold')
        ax.set_title(f'{segment} (n={len(seg_df):,})', fontweight='bold', pad=10)
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        ax.set_xlim(0, 100)
        
        for i, val in enumerate(feature_vals):
            if not np.isnan(val) and np.isfinite(val):
                ax.text(val + 2, i, f'{val:.1f}%', va='center', fontweight='bold', color=COLORS['dark'])
    
    # Hide unused subplots
    for idx in range(n_segments, n_rows * 2):
        row = idx // 2
        col = idx % 2
        axes[row, col].axis('off')
    
    plt.tight_layout()
    display(fig)
    plt.close()
else:
    print("supplier_segment field not found in data or no segments available")

# COMMAND ----------

# MARKDOWN
"""
## Chart 8: Adoption by Activity Type (Top 15)
"""

# COMMAND ----------

if 'activity_type' in df.columns:
    fig, axes = plt.subplots(2, 1, figsize=(14, 10))
    fig.suptitle('Feature Adoption by Activity Type (Top 15)', fontsize=16, fontweight='bold', y=0.995)
    
    # Get top 15 activity types by count
    top_activities = df['activity_type'].value_counts().head(15).index.tolist()
    df_act = df[df['activity_type'].isin(top_activities)].copy()
    
    # Adoption distribution by activity
    ax = axes[0]
    act_adoption = pd.crosstab(df_act['activity_type'], df_act['adoption_level'], normalize='index') * 100
    cat_order = df_act.groupby('activity_type')['advanced_features_count'].mean().sort_values(ascending=True).index.tolist()
    act_adoption_sorted = act_adoption.reindex(cat_order)
    x = np.arange(len(cat_order))
    for i, adoption in enumerate(adoption_order):
        values = [act_adoption_sorted.loc[cat, adoption] if adoption in act_adoption_sorted.columns else 0 for cat in cat_order]
        values = [v if np.isfinite(v) else 0 for v in values]
        ax.barh(x + i * 0.22, values, 0.22, label=adoption, color=PALETTE_4[i], edgecolor='white', linewidth=1.5)
    ax.set_ylabel('Activity Type', fontweight='bold')
    ax.set_xlabel('Percentage (%)', fontweight='bold')
    ax.set_title('Adoption Within Each Activity Type', fontweight='bold', pad=10)
    ax.set_yticks(x + 0.33)
    ax.set_yticklabels(cat_order, fontsize=8)
    ax.legend(loc='lower right', fontsize=7, ncol=2)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.grid(axis='x', alpha=0.3, linestyle='--', linewidth=0.5)
    
    # Average feature count by activity
    ax = axes[1]
    avg_features_act = df_act.groupby('activity_type')['advanced_features_count'].mean().reindex(cat_order)
    avg_features_act = avg_features_act.fillna(0)
    bars = ax.barh(cat_order, avg_features_act, color='#FF6B35', edgecolor='white', linewidth=2)
    ax.set_ylabel('Activity Type', fontweight='bold')
    ax.set_xlabel('Average Features Used', fontweight='bold')
    ax.set_title('Average Feature Count by Activity Type', fontweight='bold', pad=10)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.grid(axis='x', alpha=0.3, linestyle='--', linewidth=0.5)
    
    plt.tight_layout()
    display(fig)
    plt.close()
else:
    print("activity_type field not found in data")

# COMMAND ----------

# MARKDOWN
"""
## Chart 9: Most Common Features by Activity Type (Top 8)
"""

# COMMAND ----------

if 'activity_type' in df.columns:
    top_8_activities = df['activity_type'].value_counts().head(8).index.tolist()
    
    fig, axes = plt.subplots(4, 2, figsize=(16, 20))
    fig.suptitle('Most Common Features by Activity Type (Top 8)', fontsize=16, fontweight='bold', y=0.998)
    
    for idx, activity in enumerate(top_8_activities):
        row = idx // 2
        col = idx % 2
        ax = axes[row, col]
        
        act_df = df[df['activity_type'] == activity]
        
        feature_usage = {}
        for feat in feature_cols:
            if feat in act_df.columns:
                adoption_rate = (act_df[feat].sum() / len(act_df)) * 100
                feature_usage[feature_display_names.get(feat, feat)] = adoption_rate
        
        # Sort and get top 5
        sorted_features = sorted(feature_usage.items(), key=lambda x: x[1], reverse=True)[:5]
        feature_names = [f[0] for f in sorted_features]
        feature_vals = [f[1] for f in sorted_features]
        
        colors_grad = ['#FF6B35', '#FF7A47', '#FF8C5A', '#FFAD7F', '#CCCCCC']
        bars = ax.barh(feature_names, feature_vals, color=colors_grad[:len(feature_names)], 
                       edgecolor='white', linewidth=2)
        ax.set_xlabel('Adoption Rate (%)', fontweight='bold')
        ax.set_title(f'{activity} (n={len(act_df):,})', fontweight='bold', pad=10, fontsize=10)
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        ax.set_xlim(0, 100)
        
        for i, val in enumerate(feature_vals):
            if not np.isnan(val) and np.isfinite(val):
                ax.text(val + 2, i, f'{val:.1f}%', va='center', fontweight='bold', color=COLORS['dark'], fontsize=8)
    
    plt.tight_layout()
    display(fig)
    plt.close()
else:
    print("activity_type field not found in data")

# COMMAND ----------

# MARKDOWN
"""
## Chart 10: Cross-Segment Analysis - GMV Tier × Activity Type (Top 10)
"""

# COMMAND ----------

if 'activity_type' in df.columns:
    top_10_activities = df['activity_type'].value_counts().head(10).index.tolist()
    df_cross = df[df['activity_type'].isin(top_10_activities)].copy()
    
    fig, ax = plt.subplots(1, 1, figsize=(12, 8))
    fig.suptitle('Cross-Segment: GMV Tier × Activity Type (Top 10)', fontsize=16, fontweight='bold', y=0.96)
    
    # Average features heatmap
    avg_feat_cross = pd.crosstab(df_cross['gmv_tier'], df_cross['activity_type'], 
                                 values=df_cross['advanced_features_count'], aggfunc='mean')
    avg_feat_cross = avg_feat_cross.reindex(index=gmv_tier_order, columns=top_10_activities).fillna(0)
    
    im = ax.imshow(avg_feat_cross.values, cmap='Oranges', aspect='auto', vmin=0, vmax=max(avg_feat_cross.values.max(), 0.1))
    ax.set_xticks(range(len(top_10_activities)))
    ax.set_xticklabels(top_10_activities, rotation=45, ha='right', fontsize=8)
    ax.set_yticks(range(len(gmv_tier_order)))
    ax.set_yticklabels(gmv_tier_order, fontsize=9)
    ax.set_xlabel('Activity Type', fontweight='bold')
    ax.set_ylabel('GMV Tier', fontweight='bold')
    ax.set_title('Average Features Used', fontweight='bold', pad=10)
    
    for i in range(len(gmv_tier_order)):
        for j in range(len(top_10_activities)):
            val = avg_feat_cross.values[i, j]
            if np.isfinite(val) and val > 0:
                ax.text(j, i, f'{val:.1f}', ha='center', va='center',
                       color='white' if val > avg_feat_cross.values.max()/2 else 'black', fontweight='bold', fontsize=7)
    
    cbar = plt.colorbar(im, ax=ax)
    cbar.set_label('Avg Features', rotation=270, labelpad=20, fontweight='bold')
    
    plt.tight_layout()
    display(fig)
    plt.close()
else:
    print("activity_type field not found in data")

# COMMAND ----------

# MARKDOWN
"""
## Chart 11: Cross-Segment Analysis - Supplier Segment × Activity Type (Top 10)
"""

# COMMAND ----------

if 'supplier_segment' in df.columns and 'activity_type' in df.columns and len(segment_actual) > 0:
    top_10_activities = df['activity_type'].value_counts().head(10).index.tolist()
    df_cross = df[df['activity_type'].isin(top_10_activities)].copy()
    
    fig, ax = plt.subplots(1, 1, figsize=(12, 6))
    fig.suptitle('Cross-Segment: Supplier Segment × Activity Type (Top 10)', fontsize=16, fontweight='bold', y=0.98)
    
    # Average features heatmap
    avg_feat_cross = pd.crosstab(df_cross['supplier_segment'], df_cross['activity_type'], 
                                 values=df_cross['advanced_features_count'], aggfunc='mean')
    avg_feat_cross = avg_feat_cross.reindex(index=segment_actual, columns=top_10_activities).fillna(0)
    
    im = ax.imshow(avg_feat_cross.values, cmap='Oranges', aspect='auto', vmin=0, vmax=max(avg_feat_cross.values.max(), 0.1))
    ax.set_xticks(range(len(top_10_activities)))
    ax.set_xticklabels(top_10_activities, rotation=45, ha='right', fontsize=8)
    ax.set_yticks(range(len(segment_actual)))
    ax.set_yticklabels(segment_actual, fontsize=9)
    ax.set_xlabel('Activity Type', fontweight='bold')
    ax.set_ylabel('Supplier Segment', fontweight='bold')
    ax.set_title('Average Features Used', fontweight='bold', pad=10)
    
    for i in range(len(segment_actual)):
        for j in range(len(top_10_activities)):
            val = avg_feat_cross.values[i, j]
            if np.isfinite(val) and val > 0:
                ax.text(j, i, f'{val:.1f}', ha='center', va='center',
                       color='white' if val > avg_feat_cross.values.max()/2 else 'black', fontweight='bold', fontsize=7)
    
    cbar = plt.colorbar(im, ax=ax)
    cbar.set_label('Avg Features', rotation=270, labelpad=20, fontweight='bold')
    
    plt.tight_layout()
    display(fig)
    plt.close()
else:
    print("Required fields not found in data")

# COMMAND ----------

# MARKDOWN
"""
## Chart 12: Cross-Segment Analysis - Market Maturity × Activity Type (Top 10)
"""

# COMMAND ----------

if 'activity_type' in df.columns:
    top_10_activities = df['activity_type'].value_counts().head(10).index.tolist()
    df_cross = df[df['activity_type'].isin(top_10_activities)].copy()
    
    fig, ax = plt.subplots(1, 1, figsize=(12, 5))
    fig.suptitle('Cross-Segment: Market Maturity × Activity Type (Top 10)', fontsize=16, fontweight='bold', y=0.98)
    
    # Average features heatmap
    avg_feat_cross = pd.crosstab(df_cross['market_maturity'], df_cross['activity_type'], 
                                 values=df_cross['advanced_features_count'], aggfunc='mean')
    avg_feat_cross = avg_feat_cross.reindex(index=maturity_order, columns=top_10_activities).fillna(0)
    
    im = ax.imshow(avg_feat_cross.values, cmap='Oranges', aspect='auto', vmin=0, vmax=max(avg_feat_cross.values.max(), 0.1))
    ax.set_xticks(range(len(top_10_activities)))
    ax.set_xticklabels(top_10_activities, rotation=45, ha='right', fontsize=8)
    ax.set_yticks(range(len(maturity_order)))
    ax.set_yticklabels(['New (0-6m)', 'Developing (6-18m)', 'Mature (18m+)'], fontsize=9)
    ax.set_xlabel('Activity Type', fontweight='bold')
    ax.set_ylabel('Market Maturity', fontweight='bold')
    ax.set_title('Average Features Used', fontweight='bold', pad=10)
    
    for i in range(len(maturity_order)):
        for j in range(len(top_10_activities)):
            val = avg_feat_cross.values[i, j]
            if np.isfinite(val) and val > 0:
                ax.text(j, i, f'{val:.1f}', ha='center', va='center',
                       color='white' if val > avg_feat_cross.values.max()/2 else 'black', fontweight='bold', fontsize=7)
    
    cbar = plt.colorbar(im, ax=ax)
    cbar.set_label('Avg Features', rotation=270, labelpad=20, fontweight='bold')
    
    plt.tight_layout()
    display(fig)
    plt.close()
else:
    print("activity_type field not found in data")

# COMMAND ----------

# MARKDOWN
"""
## Chart 13: Cross-Segment Analysis - GMV Tier × Market Maturity
"""

# COMMAND ----------

fig, ax = plt.subplots(1, 1, figsize=(10, 6))
fig.suptitle('Cross-Segment: GMV Tier × Market Maturity', fontsize=16, fontweight='bold', y=0.96)

# Average features heatmap
avg_feat_cross = pd.crosstab(df['gmv_tier'], df['market_maturity'], 
                             values=df['advanced_features_count'], aggfunc='mean')
avg_feat_cross = avg_feat_cross.reindex(index=gmv_tier_order, columns=maturity_order).fillna(0)
im = ax.imshow(avg_feat_cross.values, cmap='Oranges', aspect='auto', vmin=0, vmax=max(avg_feat_cross.values.max(), 0.1))
ax.set_xticks(range(len(maturity_order)))
ax.set_xticklabels(['New\n(0-6m)', 'Developing\n(6-18m)', 'Mature\n(18m+)'], rotation=0)
ax.set_yticks(range(len(gmv_tier_order)))
ax.set_yticklabels(gmv_tier_order, fontsize=9)
ax.set_xlabel('Market Maturity', fontweight='bold')
ax.set_ylabel('GMV Tier', fontweight='bold')
ax.set_title('Average Features Used', fontweight='bold', pad=10)
for i in range(len(gmv_tier_order)):
    for j in range(len(maturity_order)):
        val = avg_feat_cross.values[i, j]
        if np.isfinite(val):
            ax.text(j, i, f'{val:.2f}', ha='center', va='center',
                   color='white' if val > avg_feat_cross.values.max()/2 else 'black', fontweight='bold', fontsize=9)
cbar = plt.colorbar(im, ax=ax)
cbar.set_label('Avg Features', rotation=270, labelpad=20, fontweight='bold')

plt.tight_layout()
display(fig)
plt.close()

# COMMAND ----------

# MARKDOWN
"""
## Chart 14: Adoption by Connection and Management Status
"""

# COMMAND ----------

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Adoption by Connection and Management Status', fontsize=16, fontweight='bold', y=0.995)

# Adoption by connection status
ax = axes[0, 0]
conn_adoption = pd.crosstab(df['connection_status'], df['adoption_level'], normalize='index') * 100
conn_order = ['Not Connected', 'Connected']
x = np.arange(len(conn_order))
for i, adoption in enumerate(adoption_order):
    values = [conn_adoption.loc[conn, adoption] if conn in conn_adoption.index and adoption in conn_adoption.columns else 0 for conn in conn_order]
    values = [v if np.isfinite(v) else 0 for v in values]
    ax.bar(x + i * 0.22, values, 0.22, label=adoption, color=PALETTE_4[i], edgecolor='white', linewidth=1.5)
ax.set_xlabel('Connection Status', fontweight='bold')
ax.set_ylabel('Percentage (%)', fontweight='bold')
ax.set_title('Adoption by Connection', fontweight='bold', pad=10)
ax.set_xticks(x + 0.33)
ax.set_xticklabels(conn_order, rotation=0)
ax.legend(loc='upper left', fontsize=6)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.grid(axis='y', alpha=0.3, linestyle='--', linewidth=0.5)

# Average features by connection
ax = axes[0, 1]
avg_feat_conn = df.groupby('connection_status')['advanced_features_count'].mean().reindex(conn_order)
avg_feat_conn = avg_feat_conn.fillna(0)
bars = ax.bar(conn_order, avg_feat_conn, color=['#FF6B35', '#FFAD7F'], edgecolor='white', linewidth=2)
ax.set_xlabel('Connection Status', fontweight='bold')
ax.set_ylabel('Average Features Used', fontweight='bold')
ax.set_title('Features by Connection', fontweight='bold', pad=10)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.grid(axis='y', alpha=0.3, linestyle='--', linewidth=0.5)
for i, val in enumerate(avg_feat_conn):
    if not np.isnan(val) and np.isfinite(val):
        ax.text(i, val + 0.1, f'{val:.2f}', ha='center', fontweight='bold', color=COLORS['dark'])

# Adoption by managed status
ax = axes[1, 0]
mgd_adoption = pd.crosstab(df['managed_status'], df['adoption_level'], normalize='index') * 100
mgd_order = ['Not Managed', 'Managed']
x = np.arange(len(mgd_order))
for i, adoption in enumerate(adoption_order):
    values = [mgd_adoption.loc[mgd, adoption] if mgd in mgd_adoption.index and adoption in mgd_adoption.columns else 0 for mgd in mgd_order]
    values = [v if np.isfinite(v) else 0 for v in values]
    ax.bar(x + i * 0.22, values, 0.22, label=adoption, color=PALETTE_4[i], edgecolor='white', linewidth=1.5)
ax.set_xlabel('Management Status', fontweight='bold')
ax.set_ylabel('Percentage (%)', fontweight='bold')
ax.set_title('Adoption by Managed Status', fontweight='bold', pad=10)
ax.set_xticks(x + 0.33)
ax.set_xticklabels(mgd_order, rotation=0)
ax.legend(loc='upper left', fontsize=6)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.grid(axis='y', alpha=0.3, linestyle='--', linewidth=0.5)

# Average features by managed status
ax = axes[1, 1]
avg_feat_mgd = df.groupby('managed_status')['advanced_features_count'].mean().reindex(mgd_order)
avg_feat_mgd = avg_feat_mgd.fillna(0)
bars = ax.bar(mgd_order, avg_feat_mgd, color=['#FF6B35', '#FFAD7F'], edgecolor='white', linewidth=2)
ax.set_xlabel('Management Status', fontweight='bold')
ax.set_ylabel('Average Features Used', fontweight='bold')
ax.set_title('Features by Managed Status', fontweight='bold', pad=10)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.grid(axis='y', alpha=0.3, linestyle='--', linewidth=0.5)
for i, val in enumerate(avg_feat_mgd):
    if not np.isnan(val) and np.isfinite(val):
        ax.text(i, val + 0.1, f'{val:.2f}', ha='center', fontweight='bold', color=COLORS['dark'])

plt.tight_layout()
display(fig)
plt.close()

# COMMAND ----------

# MARKDOWN
"""
## Summary Statistics
"""

# COMMAND ----------

# Calculate summary metrics
adopters = (df['adoption_level'] != 'Non-adopters (0-1)').sum()
adoption_rate = (adopters / len(df)) * 100
avg_feat = df['advanced_features_count'].mean()
median_feat = df['advanced_features_count'].median()

print("="*60)
print("PRICING FEATURE ADOPTION SUMMARY")
print("="*60)
print(f"\nTotal tours analyzed: {len(df):,}")
print(f"Total suppliers: {df['supplier_id'].nunique():,}")
print(f"Features tracked: {len(feature_cols)}")
print(f"\nGMV TIERS (Percentile-Based):")
print(f"  Top 5%: GMV ≥ {gmv_percentiles[0.95]:,.0f}")
print(f"  Top 25%: GMV ≥ {gmv_percentiles[0.75]:,.0f}")
print(f"  Top 50%: GMV ≥ {gmv_percentiles[0.50]:,.0f}")
print(f"  Bottom 50%: GMV < {gmv_percentiles[0.50]:,.0f}")
print(f"\nOVERALL ADOPTION METRICS:")
print(f"  • Overall adoption rate (2+ features): {adoption_rate:.1f}%")
print(f"  • Average features per tour: {avg_feat:.2f}")
print(f"  • Median features per tour: {median_feat:.0f}")
print(f"\nFEATURES ANALYZED:")
for i, feat in enumerate(feature_cols, 1):
    print(f"  {i}. {feat}")
print("="*60)

# Show adoption breakdown
print("\nADOPTION LEVEL BREAKDOWN:")
adoption_breakdown = df['adoption_level'].value_counts().reindex(adoption_order)
for level in adoption_order:
    count = adoption_breakdown.get(level, 0)
    pct = (count / len(df)) * 100
    print(f"  {level:25s}: {count:6,} tours ({pct:5.1f}%)")

# Connection and Managed breakdown
print("\nCONNECTION STATUS:")
conn_counts = df['connection_status'].value_counts()
for status in ['Not Connected', 'Connected']:
    count = conn_counts.get(status, 0)
    pct = (count / len(df)) * 100
    avg_f = df[df['connection_status'] == status]['advanced_features_count'].mean()
    print(f"  {status:20s}: {count:6,} tours ({pct:5.1f}%), Avg Features: {avg_f:.2f}")

print("\nMANAGED STATUS:")
mgd_counts = df['managed_status'].value_counts()
for status in ['Not Managed', 'Managed']:
    count = mgd_counts.get(status, 0)
    pct = (count / len(df)) * 100
    avg_f = df[df['managed_status'] == status]['advanced_features_count'].mean()
    print(f"  {status:20s}: {count:6,} tours ({pct:5.1f}%), Avg Features: {avg_f:.2f}")

print("\n" + "="*60)


In [0]:
# Databricks notebook source

# MARKDOWN
"""
# Pricing Feature Adoption Audit

## Objective
This notebook analyzes pricing feature adoption across GetYourGuide's supplier base to identify which segments are utilizing advanced pricing capabilities and which segments represent opportunities for feature education and adoption.

## Business Questions
1. What percentage of tours use advanced pricing features?
2. Which supplier segments (by volume, maturity, category) have highest/lowest adoption?
3. Which specific features are most used within each segment?
4. What opportunities exist to increase feature penetration?

## Features Analyzed (10 Total)
- Individual Pricing
- Group Pricing  
- Pricing without API (Native)
- Pricing with API
- Scales without API (Native Scales)
- Scales with API
- Live Pricing and Availability
- Non-live Price and Availability
- Add-ons
- Options
"""

# COMMAND ----------

# MARKDOWN
"""
## Section 1: Data Loading and Type Conversion

**What this does:** 
- Loads the pricing feature data from the Spark table
- Converts all columns to proper numeric types to avoid calculation errors
- Handles missing values by filling with 0
- Converts dates to datetime format

**Why it matters:**
Data type issues cause errors in aggregations and binning. This ensures clean numeric data for all calculations.
"""

# COMMAND ----------

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import rcParams
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Load data
print("Loading data from Spark...")
spark_df = spark.table("production.supply_analytics.pricing_feature_combined")
df = spark_df.toPandas()

print(f"Loaded {len(df):,} tours from {df['supplier_id'].nunique():,} suppliers")

# Fix data types - critical for avoiding errors
numeric_cols = ['gmv_l12mo', 'bookings_l12mo', 'has_group_pricing', 'has_individual_pricing',
                'has_addons', 'has_options', 'has_native_scales', 'has_api_scales', 
                'has_live_dynamic_pricing', 'has_non_live_pricing', 'uses_api_pricing',
                'has_scale_pricing', 'is_managed', 'num_individual_categories']

for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)

if 'date_of_first_online' in df.columns:
    df['date_of_first_online'] = pd.to_datetime(df['date_of_first_online'], errors='coerce')

# Ensure supplier_segment and activity_type are strings
if 'supplier_segment' in df.columns:
    df['supplier_segment'] = df['supplier_segment'].fillna('Unknown').astype(str)
if 'activity_type' in df.columns:
    df['activity_type'] = df['activity_type'].fillna('Unknown').astype(str)

print("✓ Data types converted")
print("\nChecking key fields:")
print(f"  - supplier_segment field exists: {'supplier_segment' in df.columns}")
print(f"  - activity_type field exists: {'activity_type' in df.columns}")

if 'supplier_segment' in df.columns:
    print(f"\nSupplier segments in data: {df['supplier_segment'].unique()}")
if 'activity_type' in df.columns:
    print(f"\nActivity types in data (first 10): {df['activity_type'].unique()[:10]}")

display(df.head())

# COMMAND ----------

# MARKDOWN
"""
## Section 2: Segmentation Creation

**What this does:**
- Creates GMV tiers based on percentile rankings: Top 5%, Top 25%, Top 50%, Bottom 50%
- Creates Market Maturity segments based on time on platform (New 0-6mo, Developing 6-18mo, Mature 18mo+)
- Uses the actual supplier_segment field from the data
- Uses the actual activity_type field from the data
- Identifies API connection status and managed supplier status from flags

**Why it matters:**
Percentile-based GMV tiers identify high-value suppliers who drive the most revenue. Different supplier segments have different adoption patterns.
"""

# COMMAND ----------

# GMV tier - percentile-based segmentation
gmv_percentiles = df['gmv_l12mo'].quantile([0, 0.50, 0.75, 0.95, 1.0])
print("GMV Distribution Percentiles:")
print(gmv_percentiles)

# Create percentile-based tiers
df['gmv_tier'] = 'Bottom 50%'
df.loc[df['gmv_l12mo'] >= gmv_percentiles[0.50], 'gmv_tier'] = 'Top 50%'
df.loc[df['gmv_l12mo'] >= gmv_percentiles[0.75], 'gmv_tier'] = 'Top 25%'
df.loc[df['gmv_l12mo'] >= gmv_percentiles[0.95], 'gmv_tier'] = 'Top 5%'

# Define tier order
gmv_tier_order = ['Top 5%', 'Top 25%', 'Top 50%', 'Bottom 50%']

print(f"\nGMV Tiers created:")
print(f"  Top 5%: GMV ≥ {gmv_percentiles[0.95]:,.0f}")
print(f"  Top 25%: GMV ≥ {gmv_percentiles[0.75]:,.0f}")
print(f"  Top 50%: GMV ≥ {gmv_percentiles[0.50]:,.0f}")
print(f"  Bottom 50%: GMV < {gmv_percentiles[0.50]:,.0f}")

for tier in gmv_tier_order:
    count = (df['gmv_tier'] == tier).sum()
    pct = count / len(df) * 100
    avg_gmv = df[df['gmv_tier'] == tier]['gmv_l12mo'].mean()
    print(f"    {tier}: {count:,} tours ({pct:.1f}%), Avg GMV: {avg_gmv:,.0f}")

# Market maturity - time-based segmentation
df['days_on_platform'] = (datetime.now() - df['date_of_first_online']).dt.days
df['market_maturity'] = 'Mature (18+ months)'
df.loc[df['days_on_platform'] <= 540, 'market_maturity'] = 'Developing (6-18 months)'
df.loc[df['days_on_platform'] <= 180, 'market_maturity'] = 'New (0-6 months)'

# Connection and management status from flags
df['connection_status'] = 'Not Connected'
df.loc[df['uses_api_pricing'] == 1, 'connection_status'] = 'Connected'

df['managed_status'] = 'Not Managed'
df.loc[df['is_managed'] == 1, 'managed_status'] = 'Managed'

# Verify segment distribution
print("\n✓ Segments created\n")
print("Market Maturity Distribution:")
display(df['market_maturity'].value_counts())
print("\nConnection Status Distribution:")
display(df['connection_status'].value_counts())
print("\nManaged Status Distribution:")
display(df['managed_status'].value_counts())
print("\nSupplier Segment Distribution (from data):")
if 'supplier_segment' in df.columns:
    display(df['supplier_segment'].value_counts())
print("\nActivity Type Distribution (from data - top 15):")
if 'activity_type' in df.columns:
    display(df['activity_type'].value_counts().head(15))

# COMMAND ----------

# MARKDOWN
"""
## Section 3: Feature Count Calculation and Adoption Level Assignment

**What this does:**
- Counts how many of the 10 pricing features each tour uses
- Creates 4 adoption levels based on realistic thresholds: Non-adopters (0-1), Low (2-3), Medium (4-5), High (6+)

**Why it matters:**
This is the core metric for measuring feature adoption maturity. Tours using more features likely generate more revenue per customer through upselling and dynamic optimization.
"""

# COMMAND ----------

# Build feature list dynamically based on available columns
feature_cols = []
if 'has_individual_pricing' in df.columns:
    feature_cols.append('has_individual_pricing')
if 'has_group_pricing' in df.columns:
    feature_cols.append('has_group_pricing')
if 'uses_api_pricing' in df.columns:
    feature_cols.append('uses_api_pricing')
if 'has_native_scales' in df.columns:
    feature_cols.append('has_native_scales')
if 'has_api_scales' in df.columns:
    feature_cols.append('has_api_scales')
if 'has_live_dynamic_pricing' in df.columns:
    feature_cols.append('has_live_dynamic_pricing')
if 'has_non_live_pricing' in df.columns:
    feature_cols.append('has_non_live_pricing')
if 'has_addons' in df.columns:
    feature_cols.append('has_addons')
if 'has_options' in df.columns:
    feature_cols.append('has_options')

# Calculate native pricing as inverse of API pricing
if 'uses_api_pricing' in df.columns:
    df['has_native_pricing'] = (1 - df['uses_api_pricing']).clip(0, 1)
    feature_cols.append('has_native_pricing')

# Sum features per tour
df['advanced_features_count'] = df[feature_cols].sum(axis=1)

print(f"Counting {len(feature_cols)} features: {feature_cols}")
print(f"Max features used by any tour: {df['advanced_features_count'].max()}")
print(f"Mean features per tour: {df['advanced_features_count'].mean():.2f}")
print(f"Median features per tour: {df['advanced_features_count'].median():.0f}")

# Create realistic adoption levels
df['adoption_level'] = 'High (6+)'
df.loc[df['advanced_features_count'] <= 5, 'adoption_level'] = 'Medium (4-5)'
df.loc[df['advanced_features_count'] <= 3, 'adoption_level'] = 'Low (2-3)'
df.loc[df['advanced_features_count'] <= 1, 'adoption_level'] = 'Non-adopters (0-1)'

df['uses_multiple_categories'] = (df['num_individual_categories'] >= 3).astype(int)

# Display distribution
print("\nAdoption Level Distribution:")
display(df['adoption_level'].value_counts())

# COMMAND ----------

# MARKDOWN
"""
## Section 4: Visualization Setup
"""

# COMMAND ----------

# Styling
rcParams['font.family'] = 'sans-serif'
rcParams['font.sans-serif'] = ['Arial', 'DejaVu Sans']
rcParams['font.size'] = 10

COLORS = {'primary': '#FF6B35', 'secondary': '#F77F51', 'tertiary': '#FFA07A', 
          'quaternary': '#C0C0C0', 'quinary': '#808080', 'dark': '#404040'}
PALETTE_4 = ['#FF6B35', '#FF8C5A', '#FFAD7F', '#CCCCCC']

# Define consistent orderings
maturity_order = ['New (0-6 months)', 'Developing (6-18 months)', 'Mature (18+ months)']
adoption_order = ['Non-adopters (0-1)', 'Low (2-3)', 'Medium (4-5)', 'High (6+)']

# Get actual segment order from data
if 'supplier_segment' in df.columns:
    segment_actual = df['supplier_segment'].value_counts().index.tolist()
    print(f"Supplier segments from data: {segment_actual}")
else:
    segment_actual = []

print("✓ Styling configured")

# COMMAND ----------

# MARKDOWN
"""
## Chart 1: Overall Feature Adoption Overview
"""

# COMMAND ----------

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Pricing Feature Adoption Overview', fontsize=16, fontweight='bold', y=0.995)

# Adoption level distribution
ax = axes[0, 0]
adoption_pcts = (df['adoption_level'].value_counts().reindex(adoption_order) / len(df) * 100)
adoption_pcts = adoption_pcts.fillna(0).values
bars = ax.barh(adoption_order, adoption_pcts, color=PALETTE_4, edgecolor='white', linewidth=2)
ax.set_xlabel('Percentage of Tours', fontweight='bold')
ax.set_title(f'Feature Adoption Levels ({len(feature_cols)} Features)', fontweight='bold', pad=10)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
for i, val in enumerate(adoption_pcts):
    if not np.isnan(val) and np.isfinite(val):
        ax.text(val + 1, i, f'{val:.1f}%', va='center', fontweight='bold', color=COLORS['dark'])

# Individual feature adoption
ax = axes[0, 1]
features = {}
if 'has_group_pricing' in df.columns:
    features['Group Pricing'] = df['has_group_pricing'].sum()
if 'has_individual_pricing' in df.columns:
    features['Individual Pricing'] = df['has_individual_pricing'].sum()
if 'has_addons' in df.columns:
    features['Add-ons'] = df['has_addons'].sum()
if 'has_options' in df.columns:
    features['Options'] = df['has_options'].sum()
if 'uses_api_pricing' in df.columns:
    features['API Pricing'] = df['uses_api_pricing'].sum()
if 'has_native_scales' in df.columns:
    features['Native Scales'] = df['has_native_scales'].sum()
if 'has_api_scales' in df.columns:
    features['API Scales'] = df['has_api_scales'].sum()
if 'has_live_dynamic_pricing' in df.columns:
    features['Live Pricing'] = df['has_live_dynamic_pricing'].sum()

feature_pcts = [(v / len(df)) * 100 for v in features.values()]
colors_bar = ['#FF6B35', '#FF7A47', '#FF8C5A', '#FF9E6C', '#FFAD7F', '#FFBC91', '#FFCBA4', '#D9D9D9']
bars = ax.barh(list(features.keys()), feature_pcts, color=colors_bar[:len(features)], 
               edgecolor='white', linewidth=2)
ax.set_xlabel('Adoption Rate (%)', fontweight='bold')
ax.set_title('Individual Feature Adoption', fontweight='bold', pad=10)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
for i, val in enumerate(feature_pcts):
    if not np.isnan(val) and np.isfinite(val):
        ax.text(val + 1, i, f'{val:.1f}%', va='center', fontweight='bold', color=COLORS['dark'])

# Category complexity
ax = axes[1, 0]
cat_dist = {'Adult Only': (df['num_individual_categories'] == 1).sum(),
            'Adult + Child': (df['num_individual_categories'] == 2).sum(),
            '3 Categories': (df['num_individual_categories'] == 3).sum(),
            '4+ Categories': (df['num_individual_categories'] >= 4).sum()}
cat_pcts = [(v / len(df)) * 100 for v in cat_dist.values()]
bars = ax.bar(range(len(cat_pcts)), cat_pcts, color=PALETTE_4[:4], edgecolor='white', linewidth=2)
ax.set_xticks(range(len(cat_dist)))
ax.set_xticklabels(cat_dist.keys(), rotation=0)
ax.set_ylabel('Percentage of Tours', fontweight='bold')
ax.set_title('Ticket Category Complexity', fontweight='bold', pad=10)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
for i, val in enumerate(cat_pcts):
    if not np.isnan(val) and np.isfinite(val):
        ax.text(i, val + 1, f'{val:.1f}%', ha='center', fontweight='bold', color=COLORS['dark'])

# Feature count distribution
ax = axes[1, 1]
feature_dist = df['advanced_features_count'].value_counts().sort_index()
ax.bar(feature_dist.index, feature_dist.values, color='#FF6B35', edgecolor='white', linewidth=2)
ax.set_xlabel('Number of Features Used', fontweight='bold')
ax.set_ylabel('Number of Tours', fontweight='bold')
ax.set_title('Distribution of Feature Count', fontweight='bold', pad=10)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
display(fig)
plt.close()

# COMMAND ----------

# MARKDOWN
"""
## Chart 2: Adoption by GMV Tier (Top 5%, Top 25%, Top 50%, Bottom 50%)
"""

# COMMAND ----------

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Feature Adoption by GMV Tier (Top 5%, Top 25%, Top 50%, Bottom 50%)', fontsize=16, fontweight='bold', y=1.02)

# Adoption distribution by GMV tier
ax = axes[0]
gmv_adoption = pd.crosstab(df['gmv_tier'], df['adoption_level'], normalize='index') * 100
x = np.arange(len(gmv_tier_order))
width = 0.22
for i, adoption in enumerate(adoption_order):
    values = [gmv_adoption.loc[tier, adoption] if tier in gmv_adoption.index and adoption in gmv_adoption.columns else 0 for tier in gmv_tier_order]
    values = [v if np.isfinite(v) else 0 for v in values]
    ax.bar(x + i * width, values, width, label=adoption, color=PALETTE_4[i], edgecolor='white', linewidth=1.5)
ax.set_xlabel('GMV Tier', fontweight='bold')
ax.set_ylabel('Percentage (%)', fontweight='bold')
ax.set_title('Adoption Distribution Within GMV Tier', fontweight='bold', pad=10)
ax.set_xticks(x + width * 1.5)
ax.set_xticklabels(gmv_tier_order, rotation=0, fontsize=9)
ax.legend(loc='upper right', fontsize=7)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.grid(axis='y', alpha=0.3, linestyle='--', linewidth=0.5)

# Average feature count by GMV tier
ax = axes[1]
avg_features_gmv = df.groupby('gmv_tier')['advanced_features_count'].mean().reindex(gmv_tier_order)
avg_features_gmv = avg_features_gmv.fillna(0)
bars = ax.bar(range(len(gmv_tier_order)), avg_features_gmv, color=['#D32F2F', '#FF6B35', '#FF8C5A', '#CCCCCC'], 
              edgecolor='white', linewidth=2)
ax.set_xlabel('GMV Tier', fontweight='bold')
ax.set_ylabel('Average Features Used', fontweight='bold')
ax.set_title('Average Feature Count by GMV Tier', fontweight='bold', pad=10)
ax.set_xticks(range(len(gmv_tier_order)))
ax.set_xticklabels(gmv_tier_order, rotation=0, fontsize=9)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.grid(axis='y', alpha=0.3, linestyle='--', linewidth=0.5)
for i, val in enumerate(avg_features_gmv):
    if not np.isnan(val) and np.isfinite(val):
        ax.text(i, val + 0.1, f'{val:.2f}', ha='center', fontweight='bold', color=COLORS['dark'])

plt.tight_layout()
display(fig)
plt.close()

# COMMAND ----------

# MARKDOWN
"""
## Chart 3: Most Common Features by GMV Tier
"""

# COMMAND ----------

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Most Common Features by GMV Tier', fontsize=16, fontweight='bold', y=0.995)

feature_display_names = {
    'has_individual_pricing': 'Individual Pricing',
    'has_group_pricing': 'Group Pricing',
    'uses_api_pricing': 'API Pricing',
    'has_native_pricing': 'Native Pricing',
    'has_native_scales': 'Native Scales',
    'has_api_scales': 'API Scales',
    'has_live_dynamic_pricing': 'Live Pricing',
    'has_non_live_pricing': 'Non-live Pricing',
    'has_addons': 'Add-ons',
    'has_options': 'Options'
}

for idx, tier in enumerate(gmv_tier_order):
    ax = axes[idx // 2, idx % 2]
    tier_df = df[df['gmv_tier'] == tier]
    
    feature_usage = {}
    for feat in feature_cols:
        if feat in tier_df.columns:
            adoption_rate = (tier_df[feat].sum() / len(tier_df)) * 100
            feature_usage[feature_display_names.get(feat, feat)] = adoption_rate
    
    # Sort and get top 5
    sorted_features = sorted(feature_usage.items(), key=lambda x: x[1], reverse=True)[:5]
    feature_names = [f[0] for f in sorted_features]
    feature_vals = [f[1] for f in sorted_features]
    
    colors_grad = ['#D32F2F', '#FF6B35', '#FF8C5A', '#FFAD7F', '#CCCCCC']
    bars = ax.barh(feature_names, feature_vals, color=colors_grad[:len(feature_names)], 
                   edgecolor='white', linewidth=2)
    ax.set_xlabel('Adoption Rate (%)', fontweight='bold')
    ax.set_title(f'{tier} (n={len(tier_df):,})', fontweight='bold', pad=10)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.set_xlim(0, 100)
    
    for i, val in enumerate(feature_vals):
        if not np.isnan(val) and np.isfinite(val):
            ax.text(val + 2, i, f'{val:.1f}%', va='center', fontweight='bold', color=COLORS['dark'])

plt.tight_layout()
display(fig)
plt.close()

# COMMAND ----------

# MARKDOWN
"""
## Chart 4: Adoption by Market Maturity
"""

# COMMAND ----------

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Feature Adoption by Market Maturity', fontsize=16, fontweight='bold', y=1.02)

# Adoption distribution by maturity
ax = axes[0]
mat_adoption = pd.crosstab(df['market_maturity'], df['adoption_level'], normalize='index') * 100
x = np.arange(len(maturity_order))
for i, adoption in enumerate(adoption_order):
    values = [mat_adoption.loc[mat, adoption] if mat in mat_adoption.index and adoption in mat_adoption.columns else 0 for mat in maturity_order]
    values = [v if np.isfinite(v) else 0 for v in values]
    ax.bar(x + i * 0.22, values, 0.22, label=adoption, color=PALETTE_4[i], edgecolor='white', linewidth=1.5)
ax.set_xlabel('Market Maturity', fontweight='bold')
ax.set_ylabel('Percentage (%)', fontweight='bold')
ax.set_title('Adoption Distribution by Maturity', fontweight='bold', pad=10)
ax.set_xticks(x + 0.33)
ax.set_xticklabels(['New\n(0-6m)', 'Developing\n(6-18m)', 'Mature\n(18m+)'], rotation=0)
ax.legend(loc='upper left', fontsize=7)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.grid(axis='y', alpha=0.3, linestyle='--', linewidth=0.5)

# Average feature count by maturity
ax = axes[1]
avg_features_mat = df.groupby('market_maturity')['advanced_features_count'].mean().reindex(maturity_order)
avg_features_mat = avg_features_mat.fillna(0)
bars = ax.bar(range(len(maturity_order)), avg_features_mat, color=['#FF6B35', '#FF8C5A', '#FFAD7F'], 
              edgecolor='white', linewidth=2)
ax.set_xlabel('Market Maturity', fontweight='bold')
ax.set_ylabel('Average Features Used', fontweight='bold')
ax.set_title('Average Feature Count by Maturity', fontweight='bold', pad=10)
ax.set_xticks(range(len(maturity_order)))
ax.set_xticklabels(['New\n(0-6m)', 'Developing\n(6-18m)', 'Mature\n(18m+)'], rotation=0)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.grid(axis='y', alpha=0.3, linestyle='--', linewidth=0.5)
for i, val in enumerate(avg_features_mat):
    if not np.isnan(val) and np.isfinite(val):
        ax.text(i, val + 0.1, f'{val:.2f}', ha='center', fontweight='bold', color=COLORS['dark'])

plt.tight_layout()
display(fig)
plt.close()

# COMMAND ----------

# MARKDOWN
"""
## Chart 5: Most Common Features by Market Maturity
"""

# COMMAND ----------

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Most Common Features by Market Maturity', fontsize=16, fontweight='bold', y=1.02)

for idx, maturity in enumerate(maturity_order):
    ax = axes[idx]
    mat_df = df[df['market_maturity'] == maturity]
    
    feature_usage = {}
    for feat in feature_cols:
        if feat in mat_df.columns:
            adoption_rate = (mat_df[feat].sum() / len(mat_df)) * 100
            feature_usage[feature_display_names.get(feat, feat)] = adoption_rate
    
    # Sort and get top 5
    sorted_features = sorted(feature_usage.items(), key=lambda x: x[1], reverse=True)[:5]
    feature_names = [f[0] for f in sorted_features]
    feature_vals = [f[1] for f in sorted_features]
    
    colors_grad = ['#FF6B35', '#FF7A47', '#FF8C5A', '#FFAD7F', '#CCCCCC']
    bars = ax.barh(feature_names, feature_vals, color=colors_grad[:len(feature_names)], 
                   edgecolor='white', linewidth=2)
    ax.set_xlabel('Adoption Rate (%)', fontweight='bold')
    ax.set_title(f'{maturity}\n(n={len(mat_df):,})', fontweight='bold', pad=10)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.set_xlim(0, 100)
    
    for i, val in enumerate(feature_vals):
        if not np.isnan(val) and np.isfinite(val):
            ax.text(val + 2, i, f'{val:.1f}%', va='center', fontweight='bold', color=COLORS['dark'])

plt.tight_layout()
display(fig)
plt.close()

# COMMAND ----------

# MARKDOWN
"""
## Chart 6: Adoption by Supplier Segment
"""

# COMMAND ----------

if 'supplier_segment' in df.columns and len(segment_actual) > 0:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle('Feature Adoption by Supplier Segment (from data)', fontsize=16, fontweight='bold', y=1.02)
    
    # Adoption distribution by segment
    ax = axes[0]
    seg_adoption = pd.crosstab(df['supplier_segment'], df['adoption_level'], normalize='index') * 100
    x = np.arange(len(segment_actual))
    for i, adoption in enumerate(adoption_order):
        values = [seg_adoption.loc[seg, adoption] if seg in seg_adoption.index and adoption in seg_adoption.columns else 0 for seg in segment_actual]
        values = [v if np.isfinite(v) else 0 for v in values]
        ax.bar(x + i * 0.22, values, 0.22, label=adoption, color=PALETTE_4[i], edgecolor='white', linewidth=1.5)
    ax.set_xlabel('Supplier Segment', fontweight='bold')
    ax.set_ylabel('Percentage (%)', fontweight='bold')
    ax.set_title('Adoption by Segment', fontweight='bold', pad=10)
    ax.set_xticks(x + 0.33)
    ax.set_xticklabels(segment_actual, rotation=45, ha='right')
    ax.legend(loc='upper right', fontsize=7)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.grid(axis='y', alpha=0.3, linestyle='--', linewidth=0.5)
    
    # Average feature count by segment
    ax = axes[1]
    avg_features_seg = df.groupby('supplier_segment')['advanced_features_count'].mean().reindex(segment_actual)
    avg_features_seg = avg_features_seg.fillna(0)
    colors_seg = ['#FF6B35', '#FF8C5A', '#FFAD7F', '#D9D9D9', '#B0B0B0', '#909090']
    bars = ax.bar(segment_actual, avg_features_seg, color=colors_seg[:len(segment_actual)], 
                  edgecolor='white', linewidth=2)
    ax.set_xlabel('Supplier Segment', fontweight='bold')
    ax.set_ylabel('Average Features Used', fontweight='bold')
    ax.set_title('Average Feature Count by Segment', fontweight='bold', pad=10)
    ax.set_xticklabels(segment_actual, rotation=45, ha='right')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.grid(axis='y', alpha=0.3, linestyle='--', linewidth=0.5)
    for i, val in enumerate(avg_features_seg):
        if not np.isnan(val) and np.isfinite(val):
            ax.text(i, val + 0.1, f'{val:.2f}', ha='center', fontweight='bold', color=COLORS['dark'])
    
    plt.tight_layout()
    display(fig)
    plt.close()
else:
    print("supplier_segment field not found in data or no segments available")

# COMMAND ----------

# MARKDOWN
"""
## Chart 7: Most Common Features by Supplier Segment
"""

# COMMAND ----------

if 'supplier_segment' in df.columns and len(segment_actual) > 0:
    n_segments = len(segment_actual)
    n_rows = (n_segments + 1) // 2
    
    fig, axes = plt.subplots(n_rows, 2, figsize=(14, n_rows * 4))
    fig.suptitle('Most Common Features by Supplier Segment', fontsize=16, fontweight='bold', y=0.998)
    
    if n_rows == 1:
        axes = axes.reshape(1, -1)
    
    for idx, segment in enumerate(segment_actual):
        row = idx // 2
        col = idx % 2
        ax = axes[row, col]
        
        seg_df = df[df['supplier_segment'] == segment]
        
        feature_usage = {}
        for feat in feature_cols:
            if feat in seg_df.columns:
                adoption_rate = (seg_df[feat].sum() / len(seg_df)) * 100
                feature_usage[feature_display_names.get(feat, feat)] = adoption_rate
        
        # Sort and get top 5
        sorted_features = sorted(feature_usage.items(), key=lambda x: x[1], reverse=True)[:5]
        feature_names = [f[0] for f in sorted_features]
        feature_vals = [f[1] for f in sorted_features]
        
        colors_grad = ['#FF6B35', '#FF7A47', '#FF8C5A', '#FFAD7F', '#CCCCCC']
        bars = ax.barh(feature_names, feature_vals, color=colors_grad[:len(feature_names)], 
                       edgecolor='white', linewidth=2)
        ax.set_xlabel('Adoption Rate (%)', fontweight='bold')
        ax.set_title(f'{segment} (n={len(seg_df):,})', fontweight='bold', pad=10)
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        ax.set_xlim(0, 100)
        
        for i, val in enumerate(feature_vals):
            if not np.isnan(val) and np.isfinite(val):
                ax.text(val + 2, i, f'{val:.1f}%', va='center', fontweight='bold', color=COLORS['dark'])
    
    # Hide unused subplots
    for idx in range(n_segments, n_rows * 2):
        row = idx // 2
        col = idx % 2
        axes[row, col].axis('off')
    
    plt.tight_layout()
    display(fig)
    plt.close()
else:
    print("supplier_segment field not found in data or no segments available")

# COMMAND ----------

# MARKDOWN
"""
## Chart 8: Adoption by Activity Type (Top 15)
"""

# COMMAND ----------

if 'activity_type' in df.columns:
    fig, axes = plt.subplots(2, 1, figsize=(14, 10))
    fig.suptitle('Feature Adoption by Activity Type (Top 15)', fontsize=16, fontweight='bold', y=0.995)
    
    # Get top 15 activity types by count
    top_activities = df['activity_type'].value_counts().head(15).index.tolist()
    df_act = df[df['activity_type'].isin(top_activities)].copy()
    
    # Adoption distribution by activity
    ax = axes[0]
    act_adoption = pd.crosstab(df_act['activity_type'], df_act['adoption_level'], normalize='index') * 100
    cat_order = df_act.groupby('activity_type')['advanced_features_count'].mean().sort_values(ascending=True).index.tolist()
    act_adoption_sorted = act_adoption.reindex(cat_order)
    x = np.arange(len(cat_order))
    for i, adoption in enumerate(adoption_order):
        values = [act_adoption_sorted.loc[cat, adoption] if adoption in act_adoption_sorted.columns else 0 for cat in cat_order]
        values = [v if np.isfinite(v) else 0 for v in values]
        ax.barh(x + i * 0.22, values, 0.22, label=adoption, color=PALETTE_4[i], edgecolor='white', linewidth=1.5)
    ax.set_ylabel('Activity Type', fontweight='bold')
    ax.set_xlabel('Percentage (%)', fontweight='bold')
    ax.set_title('Adoption Within Each Activity Type', fontweight='bold', pad=10)
    ax.set_yticks(x + 0.33)
    ax.set_yticklabels(cat_order, fontsize=8)
    ax.legend(loc='lower right', fontsize=7, ncol=2)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.grid(axis='x', alpha=0.3, linestyle='--', linewidth=0.5)
    
    # Average feature count by activity
    ax = axes[1]
    avg_features_act = df_act.groupby('activity_type')['advanced_features_count'].mean().reindex(cat_order)
    avg_features_act = avg_features_act.fillna(0)
    bars = ax.barh(cat_order, avg_features_act, color='#FF6B35', edgecolor='white', linewidth=2)
    ax.set_ylabel('Activity Type', fontweight='bold')
    ax.set_xlabel('Average Features Used', fontweight='bold')
    ax.set_title('Average Feature Count by Activity Type', fontweight='bold', pad=10)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.grid(axis='x', alpha=0.3, linestyle='--', linewidth=0.5)
    
    plt.tight_layout()
    display(fig)
    plt.close()
else:
    print("activity_type field not found in data")

# COMMAND ----------

# MARKDOWN
"""
## Chart 9: Most Common Features by Activity Type (Top 8)
"""

# COMMAND ----------

if 'activity_type' in df.columns:
    top_8_activities = df['activity_type'].value_counts().head(8).index.tolist()
    
    fig, axes = plt.subplots(4, 2, figsize=(16, 20))
    fig.suptitle('Most Common Features by Activity Type (Top 8)', fontsize=16, fontweight='bold', y=0.998)
    
    for idx, activity in enumerate(top_8_activities):
        row = idx // 2
        col = idx % 2
        ax = axes[row, col]
        
        act_df = df[df['activity_type'] == activity]
        
        feature_usage = {}
        for feat in feature_cols:
            if feat in act_df.columns:
                adoption_rate = (act_df[feat].sum() / len(act_df)) * 100
                feature_usage[feature_display_names.get(feat, feat)] = adoption_rate
        
        # Sort and get top 5
        sorted_features = sorted(feature_usage.items(), key=lambda x: x[1], reverse=True)[:5]
        feature_names = [f[0] for f in sorted_features]
        feature_vals = [f[1] for f in sorted_features]
        
        colors_grad = ['#FF6B35', '#FF7A47', '#FF8C5A', '#FFAD7F', '#CCCCCC']
        bars = ax.barh(feature_names, feature_vals, color=colors_grad[:len(feature_names)], 
                       edgecolor='white', linewidth=2)
        ax.set_xlabel('Adoption Rate (%)', fontweight='bold')
        ax.set_title(f'{activity} (n={len(act_df):,})', fontweight='bold', pad=10, fontsize=10)
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        ax.set_xlim(0, 100)
        
        for i, val in enumerate(feature_vals):
            if not np.isnan(val) and np.isfinite(val):
                ax.text(val + 2, i, f'{val:.1f}%', va='center', fontweight='bold', color=COLORS['dark'], fontsize=8)
    
    plt.tight_layout()
    display(fig)
    plt.close()
else:
    print("activity_type field not found in data")

# COMMAND ----------

# MARKDOWN
"""
## Chart 10: Cross-Segment Analysis - GMV Tier × Activity Type (Top 10)
"""

# COMMAND ----------

if 'activity_type' in df.columns:
    top_10_activities = df['activity_type'].value_counts().head(10).index.tolist()
    df_cross = df[df['activity_type'].isin(top_10_activities)].copy()
    
    fig, ax = plt.subplots(1, 1, figsize=(12, 8))
    fig.suptitle('Cross-Segment: GMV Tier × Activity Type (Top 10)', fontsize=16, fontweight='bold', y=0.96)
    
    # Average features heatmap
    avg_feat_cross = pd.crosstab(df_cross['gmv_tier'], df_cross['activity_type'], 
                                 values=df_cross['advanced_features_count'], aggfunc='mean')
    avg_feat_cross = avg_feat_cross.reindex(index=gmv_tier_order, columns=top_10_activities).fillna(0)
    
    im = ax.imshow(avg_feat_cross.values, cmap='Oranges', aspect='auto', vmin=0, vmax=max(avg_feat_cross.values.max(), 0.1))
    ax.set_xticks(range(len(top_10_activities)))
    ax.set_xticklabels(top_10_activities, rotation=45, ha='right', fontsize=8)
    ax.set_yticks(range(len(gmv_tier_order)))
    ax.set_yticklabels(gmv_tier_order, fontsize=9)
    ax.set_xlabel('Activity Type', fontweight='bold')
    ax.set_ylabel('GMV Tier', fontweight='bold')
    ax.set_title('Average Features Used', fontweight='bold', pad=10)
    
    for i in range(len(gmv_tier_order)):
        for j in range(len(top_10_activities)):
            val = avg_feat_cross.values[i, j]
            if np.isfinite(val) and val > 0:
                ax.text(j, i, f'{val:.1f}', ha='center', va='center',
                       color='white' if val > avg_feat_cross.values.max()/2 else 'black', fontweight='bold', fontsize=7)
    
    cbar = plt.colorbar(im, ax=ax)
    cbar.set_label('Avg Features', rotation=270, labelpad=20, fontweight='bold')
    
    plt.tight_layout()
    display(fig)
    plt.close()
else:
    print("activity_type field not found in data")

# COMMAND ----------

# MARKDOWN
"""
## Chart 11: Cross-Segment Analysis - Supplier Segment × Activity Type (Top 10)
"""

# COMMAND ----------

if 'supplier_segment' in df.columns and 'activity_type' in df.columns and len(segment_actual) > 0:
    top_10_activities = df['activity_type'].value_counts().head(10).index.tolist()
    df_cross = df[df['activity_type'].isin(top_10_activities)].copy()
    
    fig, ax = plt.subplots(1, 1, figsize=(12, 6))
    fig.suptitle('Cross-Segment: Supplier Segment × Activity Type (Top 10)', fontsize=16, fontweight='bold', y=0.98)
    
    # Average features heatmap
    avg_feat_cross = pd.crosstab(df_cross['supplier_segment'], df_cross['activity_type'], 
                                 values=df_cross['advanced_features_count'], aggfunc='mean')
    avg_feat_cross = avg_feat_cross.reindex(index=segment_actual, columns=top_10_activities).fillna(0)
    
    im = ax.imshow(avg_feat_cross.values, cmap='Oranges', aspect='auto', vmin=0, vmax=max(avg_feat_cross.values.max(), 0.1))
    ax.set_xticks(range(len(top_10_activities)))
    ax.set_xticklabels(top_10_activities, rotation=45, ha='right', fontsize=8)
    ax.set_yticks(range(len(segment_actual)))
    ax.set_yticklabels(segment_actual, fontsize=9)
    ax.set_xlabel('Activity Type', fontweight='bold')
    ax.set_ylabel('Supplier Segment', fontweight='bold')
    ax.set_title('Average Features Used', fontweight='bold', pad=10)
    
    for i in range(len(segment_actual)):
        for j in range(len(top_10_activities)):
            val = avg_feat_cross.values[i, j]
            if np.isfinite(val) and val > 0:
                ax.text(j, i, f'{val:.1f}', ha='center', va='center',
                       color='white' if val > avg_feat_cross.values.max()/2 else 'black', fontweight='bold', fontsize=7)
    
    cbar = plt.colorbar(im, ax=ax)
    cbar.set_label('Avg Features', rotation=270, labelpad=20, fontweight='bold')
    
    plt.tight_layout()
    display(fig)
    plt.close()
else:
    print("Required fields not found in data")

# COMMAND ----------

# MARKDOWN
"""
## Chart 12: Cross-Segment Analysis - Market Maturity × Activity Type (Top 10)
"""

# COMMAND ----------

if 'activity_type' in df.columns:
    top_10_activities = df['activity_type'].value_counts().head(10).index.tolist()
    df_cross = df[df['activity_type'].isin(top_10_activities)].copy()
    
    fig, ax = plt.subplots(1, 1, figsize=(12, 5))
    fig.suptitle('Cross-Segment: Market Maturity × Activity Type (Top 10)', fontsize=16, fontweight='bold', y=0.98)
    
    # Average features heatmap
    avg_feat_cross = pd.crosstab(df_cross['market_maturity'], df_cross['activity_type'], 
                                 values=df_cross['advanced_features_count'], aggfunc='mean')
    avg_feat_cross = avg_feat_cross.reindex(index=maturity_order, columns=top_10_activities).fillna(0)
    
    im = ax.imshow(avg_feat_cross.values, cmap='Oranges', aspect='auto', vmin=0, vmax=max(avg_feat_cross.values.max(), 0.1))
    ax.set_xticks(range(len(top_10_activities)))
    ax.set_xticklabels(top_10_activities, rotation=45, ha='right', fontsize=8)
    ax.set_yticks(range(len(maturity_order)))
    ax.set_yticklabels(['New (0-6m)', 'Developing (6-18m)', 'Mature (18m+)'], fontsize=9)
    ax.set_xlabel('Activity Type', fontweight='bold')
    ax.set_ylabel('Market Maturity', fontweight='bold')
    ax.set_title('Average Features Used', fontweight='bold', pad=10)
    
    for i in range(len(maturity_order)):
        for j in range(len(top_10_activities)):
            val = avg_feat_cross.values[i, j]
            if np.isfinite(val) and val > 0:
                ax.text(j, i, f'{val:.1f}', ha='center', va='center',
                       color='white' if val > avg_feat_cross.values.max()/2 else 'black', fontweight='bold', fontsize=7)
    
    cbar = plt.colorbar(im, ax=ax)
    cbar.set_label('Avg Features', rotation=270, labelpad=20, fontweight='bold')
    
    plt.tight_layout()
    display(fig)
    plt.close()
else:
    print("activity_type field not found in data")

# COMMAND ----------

# MARKDOWN
"""
## Chart 13: Cross-Segment Analysis - GMV Tier × Market Maturity
"""

# COMMAND ----------

fig, ax = plt.subplots(1, 1, figsize=(10, 6))
fig.suptitle('Cross-Segment: GMV Tier × Market Maturity', fontsize=16, fontweight='bold', y=0.96)

# Average features heatmap
avg_feat_cross = pd.crosstab(df['gmv_tier'], df['market_maturity'], 
                             values=df['advanced_features_count'], aggfunc='mean')
avg_feat_cross = avg_feat_cross.reindex(index=gmv_tier_order, columns=maturity_order).fillna(0)
im = ax.imshow(avg_feat_cross.values, cmap='Oranges', aspect='auto', vmin=0, vmax=max(avg_feat_cross.values.max(), 0.1))
ax.set_xticks(range(len(maturity_order)))
ax.set_xticklabels(['New\n(0-6m)', 'Developing\n(6-18m)', 'Mature\n(18m+)'], rotation=0)
ax.set_yticks(range(len(gmv_tier_order)))
ax.set_yticklabels(gmv_tier_order, fontsize=9)
ax.set_xlabel('Market Maturity', fontweight='bold')
ax.set_ylabel('GMV Tier', fontweight='bold')
ax.set_title('Average Features Used', fontweight='bold', pad=10)
for i in range(len(gmv_tier_order)):
    for j in range(len(maturity_order)):
        val = avg_feat_cross.values[i, j]
        if np.isfinite(val):
            ax.text(j, i, f'{val:.2f}', ha='center', va='center',
                   color='white' if val > avg_feat_cross.values.max()/2 else 'black', fontweight='bold', fontsize=9)
cbar = plt.colorbar(im, ax=ax)
cbar.set_label('Avg Features', rotation=270, labelpad=20, fontweight='bold')

plt.tight_layout()
display(fig)
plt.close()

# COMMAND ----------

# MARKDOWN
"""
## Chart 14: Adoption by Connection and Management Status
"""

# COMMAND ----------

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Adoption by Connection and Management Status', fontsize=16, fontweight='bold', y=0.995)

# Adoption by connection status
ax = axes[0, 0]
conn_adoption = pd.crosstab(df['connection_status'], df['adoption_level'], normalize='index') * 100
conn_order = ['Not Connected', 'Connected']
x = np.arange(len(conn_order))
for i, adoption in enumerate(adoption_order):
    values = [conn_adoption.loc[conn, adoption] if conn in conn_adoption.index and adoption in conn_adoption.columns else 0 for conn in conn_order]
    values = [v if np.isfinite(v) else 0 for v in values]
    ax.bar(x + i * 0.22, values, 0.22, label=adoption, color=PALETTE_4[i], edgecolor='white', linewidth=1.5)
ax.set_xlabel('Connection Status', fontweight='bold')
ax.set_ylabel('Percentage (%)', fontweight='bold')
ax.set_title('Adoption by Connection', fontweight='bold', pad=10)
ax.set_xticks(x + 0.33)
ax.set_xticklabels(conn_order, rotation=0)
ax.legend(loc='upper left', fontsize=6)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.grid(axis='y', alpha=0.3, linestyle='--', linewidth=0.5)

# Average features by connection
ax = axes[0, 1]
avg_feat_conn = df.groupby('connection_status')['advanced_features_count'].mean().reindex(conn_order)
avg_feat_conn = avg_feat_conn.fillna(0)
bars = ax.bar(conn_order, avg_feat_conn, color=['#FF6B35', '#FFAD7F'], edgecolor='white', linewidth=2)
ax.set_xlabel('Connection Status', fontweight='bold')
ax.set_ylabel('Average Features Used', fontweight='bold')
ax.set_title('Features by Connection', fontweight='bold', pad=10)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.grid(axis='y', alpha=0.3, linestyle='--', linewidth=0.5)
for i, val in enumerate(avg_feat_conn):
    if not np.isnan(val) and np.isfinite(val):
        ax.text(i, val + 0.1, f'{val:.2f}', ha='center', fontweight='bold', color=COLORS['dark'])

# Adoption by managed status
ax = axes[1, 0]
mgd_adoption = pd.crosstab(df['managed_status'], df['adoption_level'], normalize='index') * 100
mgd_order = ['Not Managed', 'Managed']
x = np.arange(len(mgd_order))
for i, adoption in enumerate(adoption_order):
    values = [mgd_adoption.loc[mgd, adoption] if mgd in mgd_adoption.index and adoption in mgd_adoption.columns else 0 for mgd in mgd_order]
    values = [v if np.isfinite(v) else 0 for v in values]
    ax.bar(x + i * 0.22, values, 0.22, label=adoption, color=PALETTE_4[i], edgecolor='white', linewidth=1.5)
ax.set_xlabel('Management Status', fontweight='bold')
ax.set_ylabel('Percentage (%)', fontweight='bold')
ax.set_title('Adoption by Managed Status', fontweight='bold', pad=10)
ax.set_xticks(x + 0.33)
ax.set_xticklabels(mgd_order, rotation=0)
ax.legend(loc='upper left', fontsize=6)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.grid(axis='y', alpha=0.3, linestyle='--', linewidth=0.5)

# Average features by managed status
ax = axes[1, 1]
avg_feat_mgd = df.groupby('managed_status')['advanced_features_count'].mean().reindex(mgd_order)
avg_feat_mgd = avg_feat_mgd.fillna(0)
bars = ax.bar(mgd_order, avg_feat_mgd, color=['#FF6B35', '#FFAD7F'], edgecolor='white', linewidth=2)
ax.set_xlabel('Management Status', fontweight='bold')
ax.set_ylabel('Average Features Used', fontweight='bold')
ax.set_title('Features by Managed Status', fontweight='bold', pad=10)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.grid(axis='y', alpha=0.3, linestyle='--', linewidth=0.5)
for i, val in enumerate(avg_feat_mgd):
    if not np.isnan(val) and np.isfinite(val):
        ax.text(i, val + 0.1, f'{val:.2f}', ha='center', fontweight='bold', color=COLORS['dark'])

plt.tight_layout()
display(fig)
plt.close()

# COMMAND ----------

# MARKDOWN
"""
## Summary Statistics
"""

# COMMAND ----------

# Calculate summary metrics
adopters = (df['adoption_level'] != 'Non-adopters (0-1)').sum()
adoption_rate = (adopters / len(df)) * 100
avg_feat = df['advanced_features_count'].mean()
median_feat = df['advanced_features_count'].median()

print("="*60)
print("PRICING FEATURE ADOPTION SUMMARY")
print("="*60)
print(f"\nTotal tours analyzed: {len(df):,}")
print(f"Total suppliers: {df['supplier_id'].nunique():,}")
print(f"Features tracked: {len(feature_cols)}")
print(f"\nGMV TIERS (Percentile-Based):")
print(f"  Top 5%: GMV ≥ {gmv_percentiles[0.95]:,.0f}")
print(f"  Top 25%: GMV ≥ {gmv_percentiles[0.75]:,.0f}")
print(f"  Top 50%: GMV ≥ {gmv_percentiles[0.50]:,.0f}")
print(f"  Bottom 50%: GMV < {gmv_percentiles[0.50]:,.0f}")
print(f"\nOVERALL ADOPTION METRICS:")
print(f"  • Overall adoption rate (2+ features): {adoption_rate:.1f}%")
print(f"  • Average features per tour: {avg_feat:.2f}")
print(f"  • Median features per tour: {median_feat:.0f}")
print(f"\nFEATURES ANALYZED:")
for i, feat in enumerate(feature_cols, 1):
    print(f"  {i}. {feat}")
print("="*60)

# Show adoption breakdown
print("\nADOPTION LEVEL BREAKDOWN:")
adoption_breakdown = df['adoption_level'].value_counts().reindex(adoption_order)
for level in adoption_order:
    count = adoption_breakdown.get(level, 0)
    pct = (count / len(df)) * 100
    print(f"  {level:25s}: {count:6,} tours ({pct:5.1f}%)")

# Connection and Managed breakdown
print("\nCONNECTION STATUS:")
conn_counts = df['connection_status'].value_counts()
for status in ['Not Connected', 'Connected']:
    count = conn_counts.get(status, 0)
    pct = (count / len(df)) * 100
    avg_f = df[df['connection_status'] == status]['advanced_features_count'].mean()
    print(f"  {status:20s}: {count:6,} tours ({pct:5.1f}%), Avg Features: {avg_f:.2f}")

print("\nMANAGED STATUS:")
mgd_counts = df['managed_status'].value_counts()
for status in ['Not Managed', 'Managed']:
    count = mgd_counts.get(status, 0)
    pct = (count / len(df)) * 100
    avg_f = df[df['managed_status'] == status]['advanced_features_count'].mean()
    print(f"  {status:20s}: {count:6,} tours ({pct:5.1f}%), Avg Features: {avg_f:.2f}")

print("\n" + "="*60)


In [0]:
# COMMAND ----------

# MARKDOWN
"""
## Debug: Check Managed Field
"""

# COMMAND ----------

import pandas as pd

# Load data
print("Loading data from Spark...")
spark_df = spark.table("production.supply_analytics.pricing_feature_combined")
df = spark_df.toPandas()

print(f"\nTotal rows: {len(df):,}")
print(f"Total unique suppliers: {df['supplier_id'].nunique():,}")

# Check if is_managed field exists
if 'is_managed' in df.columns:
    print("\n✓ is_managed field exists")
    
    # Show unique values in the field
    print(f"\nUnique values in is_managed field: {df['is_managed'].unique()}")
    
    # Convert to numeric to handle any data type issues
    df['is_managed'] = pd.to_numeric(df['is_managed'], errors='coerce').fillna(0)
    
    print(f"\nAfter numeric conversion - Unique values: {df['is_managed'].unique()}")
    
    # Count tours by managed status
    print("\n" + "="*60)
    print("TOURS BY MANAGED STATUS:")
    print("="*60)
    tour_counts = df['is_managed'].value_counts().sort_index()
    for val, count in tour_counts.items():
        pct = (count / len(df)) * 100
        print(f"  is_managed = {val}: {count:,} tours ({pct:.1f}%)")
    
    # Count SUPPLIERS by managed status
    print("\n" + "="*60)
    print("SUPPLIERS BY MANAGED STATUS:")
    print("="*60)
    
    # Get one row per supplier with their managed status
    supplier_managed = df.groupby('supplier_id')['is_managed'].max().reset_index()
    
    supplier_counts = supplier_managed['is_managed'].value_counts().sort_index()
    for val, count in supplier_counts.items():
        pct = (count / len(supplier_managed)) * 100
        print(f"  is_managed = {val}: {count:,} suppliers ({pct:.1f}%)")
    
    # Show sample of managed suppliers
    print("\n" + "="*60)
    print("SAMPLE OF MANAGED SUPPLIERS (is_managed = 1):")
    print("="*60)
    managed_suppliers = df[df['is_managed'] == 1]['supplier_id'].unique()[:10]
    print(f"First 10 managed supplier IDs: {managed_suppliers}")
    
    print("\n" + "="*60)
    print("SAMPLE OF NON-MANAGED SUPPLIERS (is_managed = 0):")
    print("="*60)
    non_managed_suppliers = df[df['is_managed'] == 0]['supplier_id'].unique()[:10]
    print(f"First 10 non-managed supplier IDs: {non_managed_suppliers}")
    
else:
    print("\n✗ is_managed field NOT found in data")
    print(f"\nAvailable columns: {df.columns.tolist()}")


In [0]:
# COMMAND ----------

# MARKDOWN
"""
## Debug: Check Managed Field (Correctly)
"""

# COMMAND ----------

import pandas as pd

# Load data
print("Loading data from Spark...")
spark_df = spark.table("production.supply_analytics.pricing_feature_combined")
df = spark_df.toPandas()

print(f"\nTotal rows: {len(df):,}")
print(f"Total unique suppliers: {df['supplier_id'].nunique():,}")

# Check if is_managed field exists
if 'is_managed' in df.columns:
    print("\n✓ is_managed field exists")
    
    # Show unique values in the field
    print(f"\nUnique values in is_managed field: {df['is_managed'].unique()}")
    
    # Count tours by managed status
    print("\n" + "="*60)
    print("TOURS BY MANAGED STATUS:")
    print("="*60)
    tour_counts = df['is_managed'].value_counts()
    for val, count in tour_counts.items():
        pct = (count / len(df)) * 100
        print(f"  is_managed = '{val}': {count:,} tours ({pct:.1f}%)")
    
    # Count SUPPLIERS by managed status
    print("\n" + "="*60)
    print("SUPPLIERS BY MANAGED STATUS:")
    print("="*60)
    
    # Get one row per supplier with their managed status
    supplier_managed = df.groupby('supplier_id')['is_managed'].first().reset_index()
    
    supplier_counts = supplier_managed['is_managed'].value_counts()
    for val, count in supplier_counts.items():
        pct = (count / len(supplier_managed)) * 100
        print(f"  is_managed = '{val}': {count:,} suppliers ({pct:.1f}%)")
    
    print("\n" + "="*60)
    
else:
    print("\n✗ is_managed field NOT found in data")
    print(f"\nAvailable columns: {df.columns.tolist()}")
